# Alo Yoga -- Multimodal AI for Reducing Return-Journey Friction
### Group 1 -- Multimodal AI Strategy Group Project (v3 -- real verified data + image, video and embedding analytics)

**Problem:** Alo Yoga customers face friction across the return journey (eligibility confusion, final-sale traps, no in-house alterations, return-and-rebuy inventory loss, slow refunds). This notebook builds the end-to-end pipeline: dataset compilation, text/image/video analysis, multimodal fusion, RAG prototype.

**What's new in v2:** replaces the seed sample with two real, verified data sources:
1. Our own team-collected + web-sourced Trustpilot reviews (`Alo_Yoga_Data_Collection_Template_CLEANED.xlsx`)
2. A teammate's independently-researched, source-verified Evidence corpus (`Alo_Yoga_Multimodal_Evidence_Dataset.xlsx`) -- 121 core-analysis records with per-record Verification_Status, Evidence_Strength, and RAG-inclusion flags, plus a 17-row granular Reference_Corpus and a Media_Assets index of real customer photos.

**What's new in v3** (mapped to the course outline):
- **4c Image analytics (Sessions 6-8):** the notebook now analyses pixels, not just text about images. It extracts interpretable visual features (colour, brightness, clutter, symmetry), runs a pre-trained CLIP model zero-shot and evaluates it against known labels, and uses Gemini vision captioning in place of the old stub. Images with a visible face never leave the runtime.
- **5c Video analytics (Sessions 11-12):** engagement dynamics from the scraped metadata, timestamped transcript tagging, Gemini watching the videos directly (what is said vs. what is visible), and a latency/cost trade-off table. Videos the title tags left as `other` are re-tagged from their content.
- **7c Text embeddings (Session 5):** local sentence embeddings (no API, no quota) tested against TF-IDF, a hybrid and policy-routed retrieval on gold-labelled queries (Hit@3 / Recall@3 / MRR), the two documented retrieval failures re-tested, and customer segmentation by *what* customers say (k-means) and *who* is saying it (self-reported shopper profile).
- **6e Competitor benchmark (proposal commitment):** a cited policy scorecard for Alo vs Lululemon vs Vuori on sizing communication and return guidance; Lululemon customer complaints (Instagram) compared with Lululemon's stated refund timing and with Alo's friction mix; and the Lululemon-focused YouTube landscape. Competitor evidence is kept out of Alo's counts, and the RAG only uses it when a question names a competitor.
- **7d / 8e Multimodal RAG:** image and video evidence now enter the retrieval corpus, alongside cross-modal text-to-image search. The v3 system is set up for side-by-side human scoring against the v2 text-only baseline.
- **v4, gaps against the assignment brief and course outline:** sentiment analysis (3b: lexicon vs transformer, validated against human labels and star ratings, plus aspect-based sentiment), topic modeling (3c: LDA with coherence-chosen k, and BERTopic), a UI/UX screenshot audit of the return journey (4e), influencer and short-form dynamics (5d), visual brand alignment of creator thumbnails (5e), a dataset compilation and EDA inventory against the brief's four packages (6f), a working PII scrubber (7d-0), sentiment-aware retrieval, the help-page baseline promised in the proposal (8f), and further Session 16 governance risks (9f).
- **Corrections:** the 95 catalog images are 19 unique files; the fusion table's text column no longer double-counts video rows; several stale counts in the markdown have been fixed.

**A note on verification:** before merging the second dataset, we spot-checked several of its cited sources (Zappos reviews, an Alo product page) against the live pages. This caught one broken source URL (wrong word order in a product slug) and one case where our own fetch tool couldn't render JavaScript-loaded review widgets, which we initially mis-read as a data problem rather than a tool limitation -- corrected after re-checking with fuller page text and a manual browser check. This process is itself worth keeping in the governance section as a concrete example of the hallucination/verification discipline the assignment asks for.


In [ ]:
# --- Setup ---
import pandas as pd
import numpy as np
import re
import json
import matplotlib.pyplot as plt
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 120)
print("Environment ready.")

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (key icon in the left sidebar) if
    available, else falls back to a plain environment variable. Using this instead of
    hardcoding a key means the notebook file itself never contains your actual key --
    safe to save, share, or submit. Set it ONCE in Colab Secrets and it survives
    runtime restarts, unlike os.environ, which resets every time the runtime does."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)


In [ ]:
# --- v3 setup: libraries for image, video and embedding analysis (all free / local except Gemini) ---
# Colab already ships torch, transformers, sentence-transformers, scipy and statsmodels.
# youtube-transcript-api and google-genai are installed on first run if missing.
import importlib, subprocess, sys, time, hashlib

def ensure(pip_name, import_name):
    try:
        importlib.import_module(import_name)
        return True
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
            importlib.invalidate_caches()
            importlib.import_module(import_name)
            return True
        except Exception as e:
            print(f"  could not install {pip_name}: {type(e).__name__}")
            return False

HAVE = {
    "sentence_transformers": ensure("sentence-transformers", "sentence_transformers"),
    "transformers": ensure("transformers", "transformers"),
    "yt_transcripts": ensure("youtube-transcript-api", "youtube_transcript_api"),
    "google_genai": ensure("google-genai", "google.genai"),
    "vader": ensure("vaderSentiment", "vaderSentiment"),
}
print("Optional libraries available:", HAVE)

# Gemini via the current google-genai SDK (needed for image + YouTube-video input).
# Same Colab secret as before: GEMINI_API_KEY (and optionally GEMINI_MODEL_NAME).
GEMINI_MODEL_NAME = get_secret("GEMINI_MODEL_NAME") or "gemini-3.6-flash"
_genai_client = None

def get_genai_client():
    global _genai_client
    if _genai_client is not None:
        return _genai_client
    key = get_secret("GEMINI_API_KEY")
    if not key or not HAVE["google_genai"]:
        return None
    try:
        from google import genai
        _genai_client = genai.Client(api_key=key)
    except Exception as e:
        print("Gemini client unavailable:", type(e).__name__, str(e)[:150])
        _genai_client = None
    return _genai_client

def parse_json_loose(text):
    """Gemini sometimes wraps JSON in ```json fences or adds a sentence -- pull out the object."""
    if text is None:
        return None
    t = re.sub(r"^```(?:json)?|```$", "", str(text).strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
    return None

def gemini_json(parts, retries=3, pause=8):
    """One Gemini call that must return JSON. Returns (dict or None, seconds, total_tokens).
    Retries on 429/503 (free-tier rate limits); returns None instead of raising so a
    notebook run never dies half-way because of quota."""
    client = get_genai_client()
    if client is None:
        return None, 0.0, 0
    from google.genai import types
    for attempt in range(retries):
        t0 = time.time()
        try:
            resp = client.models.generate_content(
                model=GEMINI_MODEL_NAME, contents=parts,
                config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0),
            )
            usage = getattr(resp, "usage_metadata", None)
            tokens = getattr(usage, "total_token_count", 0) or 0
            out = parse_json_loose(resp.text)
            if isinstance(out, list):   # Gemini sometimes wraps the object in a list -- unwrap to the first object
                out = next((o for o in out if isinstance(o, dict)), None)
            return out, round(time.time() - t0, 2), tokens
        except Exception as e:
            msg = str(e)
            if any(s in msg for s in ["429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE", "overloaded"]):
                wait = pause * (attempt + 1)
                print(f"  rate-limited, waiting {wait}s ...")
                time.sleep(wait)
                continue
            print("  Gemini call failed:", msg[:200])
            return None, round(time.time() - t0, 2), 0
    return None, 0.0, 0

def load_cache(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return {}

def save_cache(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1, default=str)

print("Gemini model:", GEMINI_MODEL_NAME, "| key found:", bool(get_secret("GEMINI_API_KEY")))


## 1. Reference Package -- Policy & Fit-Claim Documents

Combines our own verified reference docs (ESG rating, FA19 sizing PDF) with a teammate's more granular, independently-researched Reference_Corpus (17 rows covering exact policy wording, workflow steps, and competitor benchmarks).


In [ ]:
# --- Load the granular Reference_Corpus (17 rows) ---
EVIDENCE_PATH = "Alo_Yoga_Multimodal_Evidence_Dataset.xlsx"

ref_corpus_df = pd.read_excel(EVIDENCE_PATH, sheet_name="Reference_Corpus", header=3)
ref_corpus_df = ref_corpus_df.dropna(subset=["Reference_ID"])
print(f"Loaded {len(ref_corpus_df)} granular reference rows")
ref_corpus_df[["Reference_ID", "Company", "Topic", "Policy_or_Claim"]].head(10)


In [ ]:
# --- Our own supplementary reference docs (unique info not in the teammate's corpus) ---
supplementary_docs = [
    {
        "doc_id": "SUPP-R001",
        "brand": "Alo Yoga",
        "doc_type": "Sustainability / ESG Rating (third-party, Good On You)",
        "source_url": "https://directory.goodonyou.eco/brand/alo",
        "text": (
            "Independent rating: overall 'Not Good Enough'. Planet 1/5 -- uses few lower-impact "
            "materials, no transparency on material composition, no evidence of initiatives on "
            "packaging waste, textile waste, microplastics, or hazardous chemicals. People 1/5 -- "
            "no disclosure of complete supplier information or final production locations, no "
            "evidence of living wages, no Code of Conduct. Animals 2/5 -- uses leather, wool, and "
            "cashmere, no animal welfare policy. This is a meaningful gap versus Alo's wellness-"
            "focused brand positioning -- a candidate for our sustainability-claims / greenwashing "
            "check in the governance section."
        ),
    },
    {
        "doc_id": "SUPP-R002",
        "brand": "Alo Yoga",
        "doc_type": "Sizing Guide (Women's, FA19 season -- verify against current site)",
        "source_url": "user-provided PDF: FA19SizeCharts.pdf",
        "text": (
            "Women's size chart (inches): XS bust 32-34, waist 25-26.5, hip 34.5-36, inseam 31.5; "
            "S bust 34-36, waist 27-28.5, hip 36.5-38, inseam 32; M bust 36-38, waist 29-30.5, "
            "hip 38.5-40, inseam 32.5; L bust 38-40, waist 31-33, hip 40.5-42, inseam 32.5; "
            "XL bust 40-42, waist 34-36, hip 42-44, inseam 32.5. Note: this chart is dated Fall 2019 "
            "(FA19) -- flagged as a data-provenance caveat since Alo may have since revised specs."
        ),
    },
]

# Combine into one unified reference_docs list for the RAG corpus
reference_docs = []
for _, row in ref_corpus_df.iterrows():
    reference_docs.append({
        "doc_id": row["Reference_ID"],
        "brand": row["Company"],
        "doc_type": f"{row['Authority_Type']} -- {row['Topic']}",
        "source_url": row["Source_URL"],
        "text": row["Policy_or_Claim"] + (
            f" [Benchmark note: {row['Conflict_or_Benchmark_Note']}]"
            if pd.notna(row.get("Conflict_or_Benchmark_Note")) else ""
        ),
    })
reference_docs.extend(supplementary_docs)

ref_df = pd.DataFrame(reference_docs)
print(f"Total unified reference documents: {len(ref_df)}")
ref_df.head(10)


## 2. Text / Evidence Package

Primary corpus: the teammate's verified Evidence dataset (121 core-analysis records, each with a real source URL, Verification_Status, and Evidence_Strength). Supplementary: our own cleaned Trustpilot compilation, filtered to `return_journey_relevant == True`.


In [ ]:
# --- Load the verified Evidence corpus ---
evidence_df = pd.read_excel(EVIDENCE_PATH, sheet_name="Evidence", header=3)
evidence_df = evidence_df.dropna(subset=["Record_ID"])
evidence_core = evidence_df[evidence_df["Include_in_Core_Analysis"] == "Yes"].copy()
print(f"Evidence corpus: {len(evidence_df)} total rows, {len(evidence_core)} flagged Include_in_Core_Analysis=Yes")
evidence_core[["Record_ID", "Platform", "Fit_Issue", "Friction_Theme", "Return_Journey_Stage", "Evidence_Summary"]].head(8)


In [ ]:
# --- Load our own cleaned supplementary text data ---
try:
    supp_text_df = pd.read_excel("Alo_Yoga_Data_Collection_Template_CLEANED.xlsx", sheet_name="Text_Reviews", header=3)
    supp_text_df = supp_text_df[supp_text_df["return_journey_relevant"] == True].copy()
    # Data-quality check: a couple of rows came through with a blank "id" cell in the
    # source spreadsheet. Rather than silently dropping them (losing real evidence) or
    # letting a NaN doc_id flow into the RAG corpus (which would show up as "nan" in a
    # citation), we assign a clearly-marked placeholder ID so the row stays usable and
    # traceable back to "needs an ID fixed in the source file."
    missing_id = supp_text_df["id"].isna()
    if missing_id.any():
        supp_text_df.loc[missing_id, "id"] = [f"T-NOID-{i}" for i in range(missing_id.sum())]
        print(f"Note: {missing_id.sum()} row(s) had a blank id in the source file -- assigned placeholder ID(s) so they stay traceable.")
    print(f"Supplementary text data: {len(supp_text_df)} return-journey-relevant rows")
except FileNotFoundError:
    supp_text_df = pd.DataFrame()
    print("Supplementary file not found -- continuing with the Evidence corpus only.")


## 3. Unified Friction Taxonomy

Maps the Evidence corpus's granular `Friction_Theme` / `Fit_Issue` / `Return_Journey_Stage` fields (40+ unique values, mostly n=1) and our own `friction_tag` labels into one consistent set of broad categories for the Return Friction Map.


In [ ]:
# --- Unified taxonomy mapping ---
UNIFIED_TAXONOMY = {
    "policy-transparency": "Return/refund rules not clearly disclosed or inconsistently applied.",
    "refund-delay": "Refund took an unexpectedly long time, was partial, or was never received.",
    "rebuy-loop": "No direct exchange -- customer must return, wait, then repurchase; risk of losing the size/sale price.",
    "product-quality": "Defects, pilling, durability issues that drove the return.",
    "support-quality": "Complaints about how a support interaction was handled.",
    "fit-sizing": "True-to-size / runs small / runs large / length mismatch signals -- the pre-purchase-facing sizing-guidance angle.",
    "final-sale-trap": "Item was discounted/on sale and turned out non-returnable.",
    "other": "Doesn't cleanly fit the above categories.",
}

def classify_blob(blob):
    """Keyword-rule classifier shared across text evidence AND video timestamp notes.
    NOTE: this is the documented fallback -- professor's guidance was to use LLM few-shot
    classification (Gemini) once an API key is available; see Section 6d."""
    blob = str(blob).lower()
    if any(k in blob for k in ["true to size", "too tight", "too small", "runs small", "runs large", "size down", "size up", "length", "petite", "sheer", "compression", "opacity", "sizing"]):
        return "fit-sizing"
    if any(k in blob for k in ["pilling", "durab", "defect", "damage", "quality", "snag", "hole"]):
        return "product-quality"
    if any(k in blob for k in ["final sale", "sale price", "price protection"]):
        return "final-sale-trap"
    if any(k in blob for k in ["exchange", "repurchase", "sequential", "adjacent size"]):
        return "rebuy-loop"
    if any(k in blob for k in ["refund", "reconciliation", "chargeback", "settlement", "gift card instead"]):
        return "refund-delay"
    if any(k in blob for k in ["customer support", "support team", "support ticket", "complaint", "unanswered", "escalation", "customer service"]):
        return "support-quality"  # NOTE: narrowed from bare "support" -- that was catching
        # "bra support/fit cues" (a fit term) as a false-positive customer-support complaint;
        # caught during self-verification of the video-tagging output below.
    if any(k in blob for k in ["policy", "transparency", "disclosure", "eligibility"]):
        return "policy-transparency"
    return "other"

def map_evidence_row(row):
    blob = " ".join(str(row.get(c, "")) for c in ["Friction_Theme", "Fit_Issue", "Return_Journey_Stage", "Evidence_Summary"])
    return classify_blob(blob)

evidence_core["unified_tag"] = evidence_core.apply(map_evidence_row, axis=1)
print(evidence_core["unified_tag"].value_counts())


## 3b. Sentiment Analysis: Lexicon vs Transformer, Validated, and Aspect-Based (Session 3)

The friction taxonomy (Section 3) says **what** a customer is talking about, not **how they feel about it**. That gap causes a documented failure: for "customer emailed support three times", retrieval surfaced a review *praising* fast support (Section 7). This section adds sentiment in the three forms Session 3 covers:

1. **Lexicon (VADER):** fast, transparent, no training. It is **validated against labels we already have**: the Evidence corpus's human-coded `Sentiment` column, and star ratings on the supplementary reviews.
2. **Transformer** (`cardiffnlp/twitter-roberta-base-sentiment-latest`, run in Colab): the same validation, so the two methods can be compared on accuracy, not assumed.
3. **Aspect-based sentiment (ABSA):** sentence-level sentiment attached to the journey aspect each sentence mentions (fit, length, fabric, price, refund, exchange, support, delivery, policy). This is the "aspect-based sentiment analysis for CRM and brand tracking" framing from Session 3.

The sentiment scores are reused later: 7d uses them so that a complaint-shaped question retrieves complaints.

In [ ]:
# --- 3b-1. Lexicon sentiment (VADER) on every text record, validated against human labels and star ratings ---
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vaderSentiment"])
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
_sia = SentimentIntensityAnalyzer()

def vader_compound(text):
    return _sia.polarity_scores(str(text))["compound"]

def to_label(c, band=0.05):
    return "Positive" if c >= band else ("Negative" if c <= -band else "Neutral")

# One table of all text evidence with whatever ground truth each row has
text_records = pd.concat([
    pd.DataFrame({"doc_id": evidence_core["Record_ID"], "source": evidence_core["Platform"].astype(str),
                  "text": evidence_core["Evidence_Summary"].astype(str),
                  "human_sentiment": evidence_core["Sentiment"] if "Sentiment" in evidence_core else np.nan, "stars": np.nan,
                  "unified_tag": evidence_core["unified_tag"]}),
    pd.DataFrame({"doc_id": supp_text_df["id"],
                  "source": supp_text_df["source"].astype(str) if "source" in supp_text_df else "supplementary",
                  "text": supp_text_df["text"].astype(str), "human_sentiment": np.nan,
                  "stars": pd.to_numeric(supp_text_df["rating"], errors="coerce") if "rating" in supp_text_df else np.nan,
                  "unified_tag": [{"no-hemming": "product-quality", "general-fit-chat": "fit-sizing"}.get(t, t)
                                  if t in list(UNIFIED_TAXONOMY) + ["no-hemming", "general-fit-chat"] else "other"
                                  for t in supp_text_df["friction_tag"]]}),
], ignore_index=True)
text_records["vader_compound"] = text_records["text"].apply(vader_compound)
text_records["vader_label"] = text_records["vader_compound"].apply(to_label)

# Validation 1: human-coded sentiment on the Evidence corpus (Mixed excluded -- no 3-class equivalent)
v1 = text_records[text_records["human_sentiment"].isin(["Positive", "Negative", "Neutral"])]
LABELS = ["Negative", "Neutral", "Positive"]
if len(v1):
    print(f"VADER vs human-coded Evidence sentiment (n={len(v1)}): accuracy {accuracy_score(v1['human_sentiment'], v1['vader_label']):.0%}, "
          f"macro-F1 {f1_score(v1['human_sentiment'], v1['vader_label'], labels=LABELS, average='macro'):.2f}")
    print(pd.DataFrame(confusion_matrix(v1["human_sentiment"], v1["vader_label"], labels=LABELS),
                       index=[f"human {l}" for l in LABELS], columns=[f"VADER {l}" for l in LABELS]).to_string())
else:
    print("No human-coded Sentiment column found -- validation 1 skipped.")

# Validation 2: star ratings on the supplementary reviews (1 = worst)
v2 = text_records.dropna(subset=["stars"])
v2 = v2[v2["stars"].between(1, 5)]
if len(v2) >= 10:
    from scipy import stats as _st
    rho, p = _st.spearmanr(v2["stars"], v2["vader_compound"])
    low = v2[v2["stars"] <= 2]
    print(f"\nVADER vs star ratings (n={len(v2)}): Spearman rho = {rho:.2f} (p = {p:.3f}); "
          f"{(low['vader_label'] == 'Negative').mean():.0%} of 1-2 star reviews scored Negative, "
          f"{(low['vader_label'] == 'Positive').mean():.0%} scored Positive (lexicon misses sarcasm / complaint-by-description).")

print("\nWhy VADER struggles on the Evidence corpus: those rows are third-person SUMMARIES ('Buyer reported pilling after ten"
      "\nminutes') -- the emotional words were removed when the teammate summarised them. On raw customer text (the"
      "\nsupplementary reviews) the lexicon has real words to work with. This is Session 2's 'domain-specific enterprise"
      "\ntext noise' point in practice: the same model behaves differently on paraphrased vs raw text.")


In [ ]:
# --- 3b-2. Transformer sentiment, same validation (Colab; ~500 MB model, CPU is fine for ~430 short texts) ---
TRANSFORMER_SENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
text_records["tf_label"] = None
try:
    from transformers import pipeline
    _tf_sent = pipeline("sentiment-analysis", model=TRANSFORMER_SENT_MODEL, truncation=True, max_length=256)
    out = _tf_sent(text_records["text"].str[:1000].tolist(), batch_size=32)
    text_records["tf_label"] = [o["label"].capitalize() for o in out]
    v1t = text_records[text_records["human_sentiment"].isin(LABELS)]
    if len(v1t) == 0:
        raise ValueError("no human-coded labels to validate against")
    cmp = pd.DataFrame([
        {"method": "VADER (lexicon)", "accuracy": accuracy_score(v1t["human_sentiment"], v1t["vader_label"]),
         "macro_F1": f1_score(v1t["human_sentiment"], v1t["vader_label"], labels=LABELS, average="macro")},
        {"method": "RoBERTa (transformer)", "accuracy": accuracy_score(v1t["human_sentiment"], v1t["tf_label"]),
         "macro_F1": f1_score(v1t["human_sentiment"], v1t["tf_label"], labels=LABELS, average="macro")},
    ]).round(3)
    print(f"Lexicon vs transformer on the human-coded Evidence sentiment (n={len(v1t)}):")
    print(cmp.to_string(index=False))
    low_t = text_records[text_records["stars"].between(1, 2)]
    if len(low_t):
        print(f"1-2 star reviews labelled Negative: VADER {(low_t['vader_label'] == 'Negative').mean():.0%} "
              f"vs transformer {(low_t['tf_label'] == 'Negative').mean():.0%}")
    SENT_COL = "tf_label"
except Exception as e:
    print(f"Transformer sentiment not run here ({type(e).__name__}) -- run in Colab. VADER results are used downstream.")
    SENT_COL = "vader_label"
text_records["sentiment"] = text_records[SENT_COL].fillna(text_records["vader_label"])
print("\nSentiment by friction category (share Negative):")
print(text_records.groupby("unified_tag")["sentiment"].apply(lambda s: (s == "Negative").mean()).sort_values(ascending=False)
      .round(2).to_string())


In [ ]:
# --- 3b-3. Aspect-based sentiment: which parts of the journey do customers feel worst about? ---
ASPECTS = {
    "fit / size":        r"\bfit|size|sizing|tight|loose|small|large|snug|true to size|runs",
    "length / inseam":   r"length|inseam|long|short|petite|tall|7/8",
    "fabric / quality":  r"fabric|quality|pill|see[- ]?through|sheer|material|thin|ripped|hole|durab|stitch|seam",
    "price / value":     r"price|expensive|worth|overpriced|cost|\$\d",
    "refund":            r"refund|money back|reimburs|credit card|paypal",
    "exchange":          r"exchange|swap|different size|re-?order|repurchase",
    "support":           r"support|customer service|agent|email|chat|phone|response|reply",
    "delivery / return shipping": r"ship|deliver|label|package|parcel|courier|ups|fedex|usps|arriv",
    "policy / eligibility": r"policy|final sale|eligib|30 days|window|tags|store credit|gift card",
}
_sent_split = re.compile(r"(?<=[.!?])\s+|\n+")
absa_rows = []
for r in text_records.itertuples():
    for sent in _sent_split.split(r.text):
        s_low = sent.lower()
        if len(s_low.split()) < 3:
            continue
        c = vader_compound(sent)
        for aspect, pat in ASPECTS.items():
            if re.search(pat, s_low):
                absa_rows.append({"doc_id": r.doc_id, "aspect": aspect, "compound": c, "sentence": sent.strip()})
absa_df = pd.DataFrame(absa_rows)
absa = (absa_df.groupby("aspect").agg(mentions=("compound", "size"), mean_sentiment=("compound", "mean"),
                                      share_negative=("compound", lambda x: (x <= -0.05).mean()))
        .sort_values("mean_sentiment"))
print(f"Aspect-based sentiment over {absa_df['doc_id'].nunique()} records ({len(absa_df)} aspect mentions):")
print(absa.round(2).to_string())

fig, ax = plt.subplots(figsize=(8.5, 4.2))
cols = ["#C2410C" if v < 0 else "#3B6FA0" for v in absa["mean_sentiment"]]
ax.barh(absa.index, absa["mean_sentiment"], color=cols, height=0.6, zorder=3)
for i, (m, n) in enumerate(zip(absa["mean_sentiment"], absa["mentions"])):
    ax.text(m + (0.01 if m >= 0 else -0.01), i, f"n={n}", va="center", ha="left" if m >= 0 else "right", fontsize=8, color="#374151")
ax.axvline(0, color="#6B7280", linewidth=1)
ax.set_xlabel("mean sentence sentiment (VADER compound, -1 to +1)")
ax.set_title("Aspect-based sentiment across the return journey", fontsize=12, fontweight="bold", loc="left")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.grid(axis="x", color="#E5E7EB", linewidth=0.8, zorder=0)
plt.tight_layout()
plt.savefig("aspect_based_sentiment.png", dpi=150)
plt.show()
worst = absa.index[0]
print(f"\nMost negative aspect: '{worst}'. Most negative sentences there:")
for sent in absa_df[absa_df["aspect"] == worst].nsmallest(3, "compound")["sentence"]:
    print("   -", sent[:150])


## 3c. Topic Modeling: LDA vs BERTopic (Session 4)

The keyword taxonomy is **top-down**: we decided the categories in advance. Topic modeling is **bottom-up**: it finds the themes customers actually talk about, including any the taxonomy missed. That is Session 4's "extracting latent business trends from consumer feedback". We run both approaches the session compares:

- **LDA** (bag-of-words, probabilistic). The number of topics is chosen by **UMass coherence**, not by eye.
- **BERTopic** (contextual embeddings + clustering + class-based TF-IDF). It runs in Colab. If it can't be installed, the k-means embedding segments in 7c-5 are the embedding-based comparison.

Each topic is then mapped back to the friction taxonomy, and we check where the keyword-`other` rows land.

In [ ]:
# --- 3c-1. LDA with k chosen by UMass coherence ---
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

TOPIC_STOP = list(TfidfVectorizer(stop_words="english").get_stop_words()) + [
    "alo", "yoga", "just", "like", "get", "got", "really", "don", "didn", "im", "ive", "one", "would", "also", "even",
    "buyer", "reviewer", "commenter", "customer", "reported", "said", "says", "source", "https", "www", "com"]
topic_texts = text_records[text_records["text"].str.split().str.len() >= 5].copy()
cv = CountVectorizer(stop_words=TOPIC_STOP, min_df=3, max_df=0.4, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
Xc = cv.fit_transform(topic_texts["text"])
vocab = np.array(cv.get_feature_names_out())
Xbin = (Xc > 0).astype(int).tocsc()
doc_freq = np.asarray(Xbin.sum(axis=0)).ravel()

def umass(top_idx):
    score, pairs = 0.0, 0
    for i in range(1, len(top_idx)):
        for j in range(i):
            wi, wj = top_idx[i], top_idx[j]
            co = Xbin[:, wi].multiply(Xbin[:, wj]).sum()
            score += np.log((co + 1) / doc_freq[wj])
            pairs += 1
    return score / pairs

coh = {}
lda_models = {}
for k in range(4, 11):
    lda = LatentDirichletAllocation(n_components=k, random_state=0, learning_method="batch", max_iter=50).fit(Xc)
    tops = [np.argsort(-comp)[:10] for comp in lda.components_]
    coh[k] = np.mean([umass(t) for t in tops])
    lda_models[k] = lda
LDA_K = max(coh, key=coh.get)
lda = lda_models[LDA_K]
print("UMass coherence by k (closer to 0 = more coherent):", {k: round(v, 2) for k, v in coh.items()}, "-> k =", LDA_K)

doc_topic = lda.transform(Xc)
topic_texts["lda_topic"] = doc_topic.argmax(1)
topic_texts["lda_conf"] = doc_topic.max(1)
topic_words = {t: ", ".join(vocab[np.argsort(-lda.components_[t])[:7]]) for t in range(LDA_K)}
tp = (topic_texts.groupby("lda_topic")
      .agg(docs=("doc_id", "size"),
           main_friction_tag=("unified_tag", lambda s: s.value_counts().index[0]),
           share_negative=("sentiment", lambda s: round((s == "Negative").mean(), 2)))
      .assign(top_words=lambda d: d.index.map(topic_words)).sort_values("docs", ascending=False))
print(f"\nLDA topics over {len(topic_texts)} text records:")
print(tp.to_string())
for t in tp.index:
    ex = topic_texts[topic_texts["lda_topic"] == t].nlargest(1, "lda_conf")["text"].str[:120].iloc[0]
    print(f"  topic {t} most typical: {ex}")

oth = topic_texts[topic_texts["unified_tag"] == "other"]
print(f"\nThe {len(oth)} keyword-'other' records, by LDA topic -- themes the keyword rules miss:")
print(oth["lda_topic"].map(lambda t: f"topic {t}: {topic_words[t]}").value_counts().to_string())


In [ ]:
# --- 3c-2. BERTopic (contextual embeddings) -- Colab; skipped cleanly if the package/model is unavailable ---
RUN_BERTOPIC = True
bertopic_info = None
if RUN_BERTOPIC:
    try:
        if not ensure("bertopic", "bertopic"):
            raise ImportError("bertopic not installable here")
        from bertopic import BERTopic
        from sklearn.feature_extraction.text import CountVectorizer as _CV
        _bt = BERTopic(embedding_model="sentence-transformers/all-MiniLM-L6-v2", min_topic_size=8,
                       vectorizer_model=_CV(stop_words=TOPIC_STOP), random_state=0) \
            if "random_state" in BERTopic.__init__.__code__.co_varnames else \
            BERTopic(embedding_model="sentence-transformers/all-MiniLM-L6-v2", min_topic_size=8,
                     vectorizer_model=_CV(stop_words=TOPIC_STOP))
        bt_topics, _ = _bt.fit_transform(topic_texts["text"].tolist())
        topic_texts["bertopic"] = bt_topics
        bertopic_info = _bt.get_topic_info()
        print(f"BERTopic found {int((bertopic_info['Topic'] >= 0).sum())} topics "
              f"({(topic_texts['bertopic'] == -1).mean():.0%} of records left as outliers, topic -1):")
        print(bertopic_info.head(12)[["Topic", "Count", "Name"]].to_string(index=False))
        from sklearn.metrics import adjusted_rand_score
        m = topic_texts["bertopic"] >= 0
        print(f"\nAgreement LDA vs BERTopic on the same records (adjusted Rand index): "
              f"{adjusted_rand_score(topic_texts.loc[m, 'lda_topic'], topic_texts.loc[m, 'bertopic']):.2f} "
              f"(0 = unrelated partitions, 1 = identical)")
    except Exception as e:
        print(f"BERTopic not run here ({type(e).__name__}: {str(e)[:100]}). Run in Colab; the k-means embedding "
              f"segments in 7c-5 are the embedding-based comparison if it cannot be installed.")
print("\nLDA vs BERTopic, per Session 4: LDA treats each review as a bag of words, so short reviews give it little signal;"
      "\nBERTopic embeds whole sentences, so 'waited three weeks for my money' and 'refund never arrived' land together"
      "\neven with no shared words. Where the two disagree, the embedding topics are usually the more meaningful.")


## 4. Image Package -- Media Assets

Real customer photo evidence (Nordstrom/Bazaarvoice-hosted, Depop UGC) with rights-use guidance already documented. URLs only -- not downloaded or redistributed, per our governance/data-minimization principle.


### 4a. Catalog Images -- Resolved to Real Direct URLs

Your team's Image_Data sheet listed 110 catalog rows pointing to Alo product **pages** (not direct image links) with a note to "save the actual image asset." We resolved these programmatically: Shopify (Alo's platform) embeds the real product photo in a server-rendered `og:image` meta tag, so each product page's true image URL can be extracted without needing to render JavaScript.

**Result: 95 of 110 rows resolved to real, direct CDN image URLs -- but only 19 unique image files** (each product page's `og:image` returns a single hero shot, so all 5 slots per product resolve to the same URL; see 4c). 15 rows (3 product colorways) returned 404 -- these appear to be discontinued or renamed since your team compiled the list, a data-freshness caveat worth noting in the governance section (same issue as the FA19 sizing chart).

Note: this sandbox's shell can only reach an allowlisted set of domains, so downloading these images to test end-to-end isn't possible from here -- the URLs themselves are verified real (confirmed via page fetch). The download, feature-extraction and captioning pipeline is in Section 4c; run it in Colab.


In [ ]:
# --- Resolved catalog image URLs (real, verified via og:image meta tag) ---
resolved_catalog_images = [{'id': 'I001', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_01_b1_s1_a1_1_m224_1024x1024.jpg?v=1762820993', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I002', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_01_b1_s1_a1_1_m224_1024x1024.jpg?v=1762820993', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I003', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_01_b1_s1_a1_1_m224_1024x1024.jpg?v=1762820993', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I004', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_01_b1_s1_a1_1_m224_1024x1024.jpg?v=1762820993', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I005', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_01_b1_s1_a1_1_m224_1024x1024.jpg?v=1762820993', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I006', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_04064_b1_s1_a1_1_m205_1024x1024.jpg?v=1773270024', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I007', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_04064_b1_s1_a1_1_m205_1024x1024.jpg?v=1773270024', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I008', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_04064_b1_s1_a1_1_m205_1024x1024.jpg?v=1773270024', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I009', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_04064_b1_s1_a1_1_m205_1024x1024.jpg?v=1773270024', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I010', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W5766R_04064_b1_s1_a1_1_m205_1024x1024.jpg?v=1773270024', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I011', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820543', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I012', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820543', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I013', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820543', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I014', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820543', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I015', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820543', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I016', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_05013_b1_s1_a4_1_m161_1024x1024.jpg?v=1767027830', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I017', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_05013_b1_s1_a4_1_m161_1024x1024.jpg?v=1767027830', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I018', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_05013_b1_s1_a4_1_m161_1024x1024.jpg?v=1767027830', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I019', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_05013_b1_s1_a4_1_m161_1024x1024.jpg?v=1767027830', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I020', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51428R_05013_b1_s1_a4_1_m161_1024x1024.jpg?v=1767027830', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I021', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51298R_0001_b1_s1_a1_dSP26_m246_1024x1024.jpg?v=1774546887', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I022', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51298R_0001_b1_s1_a1_dSP26_m246_1024x1024.jpg?v=1774546887', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I023', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51298R_0001_b1_s1_a1_dSP26_m246_1024x1024.jpg?v=1774546887', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I024', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51298R_0001_b1_s1_a1_dSP26_m246_1024x1024.jpg?v=1774546887', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I025', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51298R_0001_b1_s1_a1_dSP26_m246_1024x1024.jpg?v=1774546887', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I036', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6295R_01_b1_s1_a4_dFA26_m241_1024x1024.jpg?v=1789614232', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I037', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6295R_01_b1_s1_a4_dFA26_m241_1024x1024.jpg?v=1789614232', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I038', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6295R_01_b1_s1_a4_dFA26_m241_1024x1024.jpg?v=1789614232', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I039', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6295R_01_b1_s1_a4_dFA26_m241_1024x1024.jpg?v=1789614232', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I040', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6295R_01_b1_s1_a4_dFA26_m241_1024x1024.jpg?v=1789614232', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I041', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6364R_04064_b1_s1_a2_1_m18_4e21488a-f45c-4a8c-898d-37047c01c88d_1024x1024.jpg?v=1767400544', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I042', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6364R_04064_b1_s1_a2_1_m18_4e21488a-f45c-4a8c-898d-37047c01c88d_1024x1024.jpg?v=1767400544', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I043', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6364R_04064_b1_s1_a2_1_m18_4e21488a-f45c-4a8c-898d-37047c01c88d_1024x1024.jpg?v=1767400544', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I044', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6364R_04064_b1_s1_a2_1_m18_4e21488a-f45c-4a8c-898d-37047c01c88d_1024x1024.jpg?v=1767400544', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I045', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6364R_04064_b1_s1_a2_1_m18_4e21488a-f45c-4a8c-898d-37047c01c88d_1024x1024.jpg?v=1767400544', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I046', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6442R_01_b1_s1_a2_1_m233_1024x1024.jpg?v=1757972445', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I047', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6442R_01_b1_s1_a2_1_m233_1024x1024.jpg?v=1757972445', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I048', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6442R_01_b1_s1_a2_1_m233_1024x1024.jpg?v=1757972445', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I049', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6442R_01_b1_s1_a2_1_m233_1024x1024.jpg?v=1757972445', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I050', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W6442R_01_b1_s1_a2_1_m233_1024x1024.jpg?v=1757972445', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I051', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9533R_06659_b1_s1_a3_m189_1024x1024.jpg?v=1775150439', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I052', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9533R_06659_b1_s1_a3_m189_1024x1024.jpg?v=1775150439', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I053', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9533R_06659_b1_s1_a3_m189_1024x1024.jpg?v=1775150439', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I054', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9533R_06659_b1_s1_a3_m189_1024x1024.jpg?v=1775150439', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I055', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9533R_06659_b1_s1_a3_m189_1024x1024.jpg?v=1775150439', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I056', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9935R_01_b1_s1_a2_dSP26_m191_1024x1024.jpg?v=1772522654', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I057', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9935R_01_b1_s1_a2_dSP26_m191_1024x1024.jpg?v=1772522654', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I058', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9935R_01_b1_s1_a2_dSP26_m191_1024x1024.jpg?v=1772522654', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I059', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9935R_01_b1_s1_a2_dSP26_m191_1024x1024.jpg?v=1772522654', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I060', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9935R_01_b1_s1_a2_dSP26_m191_1024x1024.jpg?v=1772522654', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I061', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9780R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820381', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I062', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9780R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820381', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I063', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9780R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820381', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I064', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9780R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820381', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I065', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9780R_0100_b1_s1_a2_1_m224_1024x1024.jpg?v=1762820381', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I066', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9824R_06401_b1_s1_a3_m261_1024x1024.jpg?v=1769132666', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I067', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9824R_06401_b1_s1_a3_m261_1024x1024.jpg?v=1769132666', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I068', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9824R_06401_b1_s1_a3_m261_1024x1024.jpg?v=1769132666', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I069', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9824R_06401_b1_s1_a3_m261_1024x1024.jpg?v=1769132666', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I070', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9824R_06401_b1_s1_a3_m261_1024x1024.jpg?v=1769132666', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I071', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9835R_01_b1_s1_a4_m91_1024x1024.jpg?v=1749674384', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I072', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9835R_01_b1_s1_a4_m91_1024x1024.jpg?v=1749674384', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I073', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9835R_01_b1_s1_a4_m91_1024x1024.jpg?v=1749674384', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I074', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9835R_01_b1_s1_a4_m91_1024x1024.jpg?v=1749674384', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I075', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9835R_01_b1_s1_a4_m91_1024x1024.jpg?v=1749674384', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I076', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_06662_b1_s1_a2_dSP26_m125_1024x1024.jpg?v=1770137847', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I077', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_06662_b1_s1_a2_dSP26_m125_1024x1024.jpg?v=1770137847', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I078', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_06662_b1_s1_a2_dSP26_m125_1024x1024.jpg?v=1770137847', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I079', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_06662_b1_s1_a2_dSP26_m125_1024x1024.jpg?v=1770137847', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I080', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_06662_b1_s1_a2_dSP26_m125_1024x1024.jpg?v=1770137847', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I081', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_03842_b1_s1_a1_dSP26_m54_1024x1024.jpg?v=1775110213', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I082', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_03842_b1_s1_a1_dSP26_m54_1024x1024.jpg?v=1775110213', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I083', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_03842_b1_s1_a1_dSP26_m54_1024x1024.jpg?v=1775110213', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I084', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_03842_b1_s1_a1_dSP26_m54_1024x1024.jpg?v=1775110213', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I085', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W9884R_03842_b1_s1_a1_dSP26_m54_1024x1024.jpg?v=1775110213', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I086', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3032RG_03842_b1_s1_a2_dFA26_m161_1024x1024.jpg?v=1785950395', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I087', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3032RG_03842_b1_s1_a2_dFA26_m161_1024x1024.jpg?v=1785950395', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I088', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3032RG_03842_b1_s1_a2_dFA26_m161_1024x1024.jpg?v=1785950395', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I089', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3032RG_03842_b1_s1_a2_dFA26_m161_1024x1024.jpg?v=1785950395', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I090', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3032RG_03842_b1_s1_a2_dFA26_m161_1024x1024.jpg?v=1785950395', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I091', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3041RG_01_b1_s1_a3_dSP26_m125_1024x1024.jpg?v=1785179723', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I092', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3041RG_01_b1_s1_a3_dSP26_m125_1024x1024.jpg?v=1785179723', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I093', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3041RG_01_b1_s1_a3_dSP26_m125_1024x1024.jpg?v=1785179723', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I094', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3041RG_01_b1_s1_a3_dSP26_m125_1024x1024.jpg?v=1785179723', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I095', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3041RG_01_b1_s1_a3_dSP26_m125_1024x1024.jpg?v=1785179723', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I096', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3031RG_01_b1_s1_a2_dFA26_m191_12be2ea0-5c1e-48f8-b22a-07b2a39d7747_1024x1024.jpg?v=1789605393', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I097', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3031RG_01_b1_s1_a2_dFA26_m191_12be2ea0-5c1e-48f8-b22a-07b2a39d7747_1024x1024.jpg?v=1789605393', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I098', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3031RG_01_b1_s1_a2_dFA26_m191_12be2ea0-5c1e-48f8-b22a-07b2a39d7747_1024x1024.jpg?v=1789605393', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I099', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3031RG_01_b1_s1_a2_dFA26_m191_12be2ea0-5c1e-48f8-b22a-07b2a39d7747_1024x1024.jpg?v=1789605393', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I100', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/U3031RG_01_b1_s1_a2_dFA26_m191_12be2ea0-5c1e-48f8-b22a-07b2a39d7747_1024x1024.jpg?v=1789605393', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}, {'id': 'I101', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51432R_01_b1_s1_a3_dSU26_m191_1024x1024.jpg?v=1785175123', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-front; save the actual image asset from this product page for the shared repository.'}, {'id': 'I102', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51432R_01_b1_s1_a3_dSU26_m191_1024x1024.jpg?v=1785175123', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: model-back; save the actual image asset from this product page for the shared repository.'}, {'id': 'I103', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51432R_01_b1_s1_a3_dSU26_m191_1024x1024.jpg?v=1785175123', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: side/profile; save the actual image asset from this product page for the shared repository.'}, {'id': 'I104', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51432R_01_b1_s1_a3_dSU26_m191_1024x1024.jpg?v=1785175123', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: close-up/fabric; save the actual image asset from this product page for the shared repository.'}, {'id': 'I105', 'resolved_image_url': 'https://www.aloyoga.com/cdn/shop/files/W51432R_01_b1_s1_a3_dSU26_m191_1024x1024.jpg?v=1785175123', 'source': 'Alo Yoga website', 'notes': 'Catalog image slot: product/detail; save the actual image asset from this product page for the shared repository.'}]

resolved_images_df = pd.DataFrame(resolved_catalog_images)
print(f"{len(resolved_images_df)} resolved catalog images ready for the pipeline")
resolved_images_df.head(10)


### 4b. UGC & UI-Screenshot Images -- Manual Extraction (Batch 1, in progress)

The `Image_Data` sheet originally had 10 UGC rows pointing to **Reddit threads** (not direct image links) plus 1 UI-screenshot row referencing a file that hadn't been uploaded. The team has started pulling actual photos from those threads. First batch: 4 images checked against their source links.

**Result: 2 of 4 are genuine try-on photos we can use as UGC evidence. 2 of 4 turned out to be official Alo catalog/product-page screenshots, not the user's own photo from the thread -- flagged below rather than silently logged as UGC.**

| Row | Source thread | Image received | Verdict |
|---|---|---|---|
| I113 | `keep_the_size_small_leggings_or_get_a_medium` | Mirror selfie, blue bra + green leggings, face visible | ✅ Genuine UGC -- usable |
| I121 (new) | `should_i_size_up_need_help_with_alo_fit_check` (not in original 10 -- added because it carries a self-reported height/weight + exact SKU + size) | Mirror selfie, brown Alo bra + black flared leggings, face blurred | ✅ Genuine UGC -- usable, richest data point of the batch |
| I114 | `alosoft_sunray_sizing` | Alo's own catalog photo for "Alosoft Sunray Bra - Black/White" ("Model is 5'9.5" wearing S") | ⚠️ Mismatch -- this is the brand's official model shot, not the Reddit poster's own photo. Kept separately as a catalog sizing-reference point, **not** counted as UGC for I114. The real UGC photo for this thread is still outstanding. |
| I117 | `consisting_sizing` | No image supplied this round | Link confirmed against the original sheet's entry; still pending an actual photo |
| (no ID assigned) | Not provided | Alo's own catalog photo for "Navigator Jacket - Brownstone" ("Model is 5'7.5" wearing S") | ⚠️ Not usable as evidence -- no source link or fit-complaint text was provided with it, and it's an official catalog image rather than a user's photo. Kept only as a second catalog sizing-reference point. |

**Privacy handling (consistent with the Media_Assets rights-guidance approach already applied elsewhere):** I113 and I121 show real people. I121's face is blurred by the poster themselves; I113's face is visible. Both are used here only for internal classroom/research analysis (fit/sizing friction classification) -- neither should go into the public deck or be redistributed without confirming reuse rights, and I113 in particular should not appear in any external-facing artifact with the face visible.

**Update: I111, I112, I115, I116, I118, I119, I120 checked by the team directly and confirmed unavailable** (threads deleted/edited or no photo was ever attached to them) -- these are closed out as "source exhausted," not left as an open action item. `screenshot_return_step2.png` is a screenshot Alo's *return portal, step 2 of 4 (order-lookup form)* -- this needs someone (originally assigned to teammate Souradeep) to open Alo's returns flow and capture that screen; it can't be sourced from Reddit or a product page, and this sandbox has no browser tool to grab it. The real (non-catalog) I114 photo is also unavailable per the same check. **Net effect: the UGC image evidence base is now closed at 2 usable photos (I113, I121) plus the 19 unique catalog images (95 catalog rows) -- we're moving forward on this basis rather than continuing to chase threads that no longer have the content.**


In [ ]:
# --- UGC batch 1: manually extracted photos, checked against source links ---
ugc_batch1 = pd.DataFrame([
    {
        "id": "I113",
        "type": "UGC",
        "source_platform": "Reddit",
        "source_url": "https://www.reddit.com/r/aloyoga/comments/1968c4k/keep_the_size_small_leggings_or_get_a_medium/",
        "product_mentioned": "Airbrush leggings (size Small vs Medium)",
        "self_reported_stats": None,
        "image_file": "ugc_images/I113_ugc_fitcheck_bluebra_greenleggings.png",
        "has_visible_face": True,
        "friction_tag": "fit-sizing",
        "usable_as_ugc": True,
        "rights_note": "Research/classroom use only; do not publish externally with face visible.",
    },
    {
        "id": "I121",
        "type": "UGC",
        "source_platform": "Reddit",
        "source_url": "https://www.reddit.com/r/aloyoga/comments/1p03hjr/should_i_size_up_need_help_with_alo_fit_check/",
        "product_mentioned": "Seamless Ribbed Favorite Bra + Airbrush High-Waisted 7/8 Bootcut Legging, size Small",
        "self_reported_stats": "5'4\", 136 lbs; jeans size 6 tight / size 8 loose",
        "image_file": "ugc_images/I121_ugc_fitcheck_brownbra_blackflare.png",
        "has_visible_face": False,
        "friction_tag": "fit-sizing",
        "usable_as_ugc": True,
        "rights_note": "Face self-blurred by poster; research/classroom use only.",
    },
    {
        "id": "I114_CATALOG_MISMATCH",
        "type": "Catalog (mismatched against UGC request)",
        "source_platform": "Alo Yoga app/site",
        "source_url": "https://www.reddit.com/r/aloyoga/comments/1mr4gbp/alosoft_sunray_sizing/",
        "product_mentioned": "Alosoft Sunray Bra - Black/White",
        "self_reported_stats": "Model is 5'9.5\" wearing S (brand model, not the Reddit poster)",
        "image_file": "ugc_images/CATALOG_alosoft_sunray_bra_screenshot.png",
        "has_visible_face": True,
        "friction_tag": "reference-only",
        "usable_as_ugc": False,
        "rights_note": "Official brand imagery; not a real UGC data point for I114 -- the poster's own photo is still outstanding.",
    },
    {
        "id": "UNLABELED_CATALOG",
        "type": "Catalog (no source link provided)",
        "source_platform": "Alo Yoga app/site",
        "source_url": None,
        "product_mentioned": "Navigator Jacket - Brownstone",
        "self_reported_stats": "Model is 5'7.5\" wearing S (brand model)",
        "image_file": "ugc_images/CATALOG_navigator_jacket_brownstone_screenshot.png",
        "has_visible_face": True,
        "friction_tag": "reference-only",
        "usable_as_ugc": False,
        "rights_note": "No fit-complaint text or source link attached; kept only as a second catalog sizing-reference point.",
    },
])

ugc_batch1_usable = ugc_batch1[ugc_batch1["usable_as_ugc"]]
print(f"UGC batch 1: {len(ugc_batch1)} images checked, {len(ugc_batch1_usable)} usable as genuine UGC evidence.")
ugc_batch1[["id", "type", "product_mentioned", "usable_as_ugc", "has_visible_face"]]


In [ ]:
# --- Load Media_Assets ---
media_df = pd.read_excel(EVIDENCE_PATH, sheet_name="Media_Assets", header=3)
media_df = media_df.dropna(subset=["Asset_ID"])
print(f"Loaded {len(media_df)} media assets")
media_df[["Asset_ID", "Modality", "Platform", "Evidence_Note", "Fit_Return_Relevance", "Rights_Use_Guidance"]].head(8)


## 4c. Image Analytics: Analysing the Pixels (Sessions 6-8)

**Why this section exists:** up to v2, no image pixels were processed anywhere in this notebook. The "image" column in the fusion table (6b) came from `classify_blob` run on the *text written about* each image (`Evidence_Note`, `Fit_Return_Relevance`). This section analyses the images themselves, in three layers that follow the course sessions:

| Layer | Course session | What it does here |
|---|---|---|
| 1. Interpretable visual features | Session 6 (Zhang et al. 2022, Airbnb image features) | Brightness, saturation, colorfulness, contrast, dominant colours, clutter (edge density), left-right symmetry, sharpness |
| 2. Pre-trained vision model, zero-shot | Sessions 7-8 (CNN/ViT, transfer learning, small-sample constraints) | CLIP ViT-B/32 classifies each image's *type* and *visible friction signal* without fine-tuning, and is evaluated against labels we already know |
| 3. Vision-language model | Session 12 (VLMs) | Gemini vision replaces the old `caption_image_stub` with a structured, factual caption of visible fit/quality signals |

**Why zero-shot instead of fine-tuning:** we have roughly 30 images. Fine-tuning even the last layer of a CNN on that many images would overfit. Session 8 lists two ways around a small sample: synthetic augmentation, or a pre-trained model used as-is. CLIP was trained on about 400M image-text pairs, so we can describe the classes in plain English prompts and use no training labels at all. Fine-tuning becomes worth it once Phase 1 of the roadmap (9b) produces a few hundred agent-labelled return photos.

**Data correction (found while building this section):** the "95 resolved catalog images" in 4a are **19 unique image files**. Each product page's `og:image` tag returns one hero shot, and that same URL was written into all 5 slots (model-front, model-back, side, close-up, detail). We analyse the 19 unique files and keep the 95 row IDs linked to them.

**Governance rule applied in code:** a photo with a **visible face** (I113, and the brand-model screenshots) is analysed **only locally** (pixel features + CLIP running inside this runtime). It is never sent to an external API. Free-tier Gemini traffic may be used by Google to improve its products, so data minimisation means real customers' faces do not leave our environment.

In [ ]:
# --- 4c-1. One image inventory: 19 unique catalog files + Media_Assets customer photos + Reddit UGC ---
catalog_unique = (resolved_images_df.groupby("resolved_image_url", sort=False)["id"]
                  .apply(list).reset_index().rename(columns={"id": "catalog_row_ids"}))
catalog_unique["sku"] = catalog_unique["resolved_image_url"].str.extract(r"/files/([A-Z0-9]+)_")[0]
print(f"Catalog: {len(resolved_images_df)} sheet rows -> {len(catalog_unique)} unique image files "
      f"across {catalog_unique['sku'].nunique()} styles (one og:image hero shot per product page, reused for all 5 slots).")

inventory_rows = []
for i, r in catalog_unique.iterrows():
    inventory_rows.append({
        "image_id": f"CAT-{i+1:02d}", "source_group": "catalog", "src": r["resolved_image_url"],
        "product": r["sku"], "linked_records": ",".join(r["catalog_row_ids"]),
        "has_visible_face": None, "external_api_ok": True,   # official brand imagery
    })
for _, r in media_df[media_df["Modality"] == "Image"].iterrows():
    inventory_rows.append({
        "image_id": r["Asset_ID"], "source_group": "ugc_media_assets", "src": r.get("Direct_Media_URL"),
        "product": r.get("Product"), "linked_records": r["Asset_ID"],
        "has_visible_face": None,
        # Public listing/review photos, rights guidance = research/classroom use. Sent for captioning,
        # never republished. Set to False here if the team prefers local-only analysis for these too.
        "external_api_ok": True,
    })
for _, r in ugc_batch1.iterrows():
    face = bool(r["has_visible_face"])
    inventory_rows.append({
        "image_id": r["id"], "source_group": "ugc_reddit" if r["usable_as_ugc"] else "catalog_screenshot",
        "src": r["image_file"], "product": r["product_mentioned"], "linked_records": r["id"],
        "has_visible_face": face,
        "external_api_ok": not face,   # governance rule: visible face -> local analysis only
    })

image_inventory = pd.DataFrame(inventory_rows)
image_inventory["is_ugc"] = image_inventory["source_group"].str.startswith("ugc")
print(image_inventory["source_group"].value_counts().to_string())
print(f"\n{(~image_inventory['external_api_ok']).sum()} image(s) restricted to local-only analysis (visible face).")


In [ ]:
# --- 4c-2. Download once, cache locally, load as RGB ---
# Run in Colab: this needs normal internet access (Alo's Shopify CDN, Bazaarvoice photo hosts).
# The two Reddit UGC photos are read from the local ugc_images/ folder -- upload it to Colab next to the notebook.
from PIL import Image, ImageFilter

IMG_CACHE_DIR = "img_cache"
os.makedirs(IMG_CACHE_DIR, exist_ok=True)
HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (SPJIMR MAS research notebook)"}

def load_image(src, max_side=512):
    """Returns (PIL RGB image or None, status string)."""
    if not isinstance(src, str) or not src.strip():
        return None, "no_source"
    try:
        if src.startswith("http"):
            path = os.path.join(IMG_CACHE_DIR, hashlib.md5(src.encode()).hexdigest() + ".img")
            if not os.path.exists(path):
                import requests
                resp = requests.get(src, headers=HTTP_HEADERS, timeout=20)
                ctype = resp.headers.get("content-type", "")
                if resp.status_code != 200:
                    return None, f"http_{resp.status_code}"
                if not ctype.startswith("image"):
                    return None, f"not_a_direct_image ({ctype.split(';')[0] or 'unknown'})"
                with open(path, "wb") as f:
                    f.write(resp.content)
        else:
            path = src
            if not os.path.exists(path):
                return None, "local_file_missing"
        img = Image.open(path).convert("RGB")
        img.thumbnail((max_side, max_side))
        return img, "ok"
    except Exception as e:
        return None, f"error: {type(e).__name__}"

images = {}
load_status = []
for r in image_inventory.itertuples():
    img, status = load_image(r.src)
    load_status.append(status)
    if img is not None:
        images[r.image_id] = img
image_inventory["load_status"] = load_status

print(f"Loaded {len(images)} of {len(image_inventory)} images.\n")
print(image_inventory.groupby(["source_group", "load_status"]).size().to_string())
if len(images) == 0:
    print("\nNo images loaded -- this runtime has no internet access to the image hosts. "
          "Run in Colab; every cell below skips gracefully until then.")


In [ ]:
# --- 4c-3. Interpretable visual features (Session 6: colour, brightness, clutter, symmetry) ---
from sklearn.cluster import KMeans

def colorfulness(arr):
    """Hasler & Suesstrunk (2003) colourfulness metric -- the one used in image-engagement papers."""
    R, G, B = (arr[..., i].astype(float) for i in range(3))
    rg, yb = R - G, 0.5 * (R + G) - B
    return float(np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))

def dominant_colors(arr, k=3):
    px = arr.reshape(-1, 3).astype(float)
    if len(px) > 4000:
        px = px[np.random.RandomState(0).choice(len(px), 4000, replace=False)]
    km = KMeans(n_clusters=k, n_init=4, random_state=0).fit(px)
    shares = np.bincount(km.labels_, minlength=k) / len(km.labels_)
    order = np.argsort(-shares)
    return [("#%02x%02x%02x" % tuple(int(c) for c in km.cluster_centers_[i]), float(shares[i])) for i in order]

LAPLACIAN = ImageFilter.Kernel((3, 3), [0, 1, 0, 1, -4, 1, 0, 1, 0], scale=1, offset=128)

def visual_features(img):
    arr = np.asarray(img)
    gray_img = img.convert("L")
    gray = np.asarray(gray_img).astype(float) / 255
    hsv = np.asarray(img.convert("HSV")).astype(float) / 255
    edges = np.asarray(gray_img.filter(ImageFilter.FIND_EDGES)).astype(float) / 255
    lap = np.asarray(gray_img.filter(LAPLACIAN)).astype(float)
    half = gray.shape[1] // 2
    left, right_mirrored = gray[:, :half], np.fliplr(gray[:, gray.shape[1] - half:])
    palette = dominant_colors(arr)
    return {
        "brightness": hsv[..., 2].mean(),                  # 0-1, mean HSV value
        "saturation": hsv[..., 1].mean(),                  # 0-1
        "colorfulness": colorfulness(arr),                 # higher = more vivid colour variety
        "contrast": gray.std(),                            # luminance spread
        "clutter_edge_density": (edges > 0.15).mean(),     # share of pixels on an edge = visual clutter
        "symmetry": 1 - np.abs(left - right_mirrored).mean(),  # 1 = perfectly left-right symmetric
        "sharpness": lap.var(),                            # Laplacian variance, higher = sharper
        "dominant_color": palette[0][0], "dominant_share": round(palette[0][1], 2),
        "palette": palette,
    }

feat_rows = []
for image_id, img in images.items():
    f = visual_features(img)
    f["image_id"] = image_id
    feat_rows.append(f)
FEATURE_COLS = ["brightness", "saturation", "colorfulness", "contrast", "clutter_edge_density", "symmetry", "sharpness"]
image_features_df = (pd.DataFrame(feat_rows, columns=["image_id"] + FEATURE_COLS + ["dominant_color", "dominant_share", "palette"])
                     .merge(image_inventory[["image_id", "source_group", "is_ugc", "product"]], on="image_id", how="left"))

if len(image_features_df):
    image_features_df["group"] = np.where(image_features_df["is_ugc"], "Customer (UGC)", "Official (catalog)")
    summary = image_features_df.groupby("group")[FEATURE_COLS].median().T.round(3)
    print("Median visual features, official catalog vs customer photos:")
    print(summary.to_string())
else:
    print("No images loaded yet -- run in Colab.")


In [ ]:
# --- 4c-4. Chart: how customer photos differ from the catalog, feature by feature ---
if len(image_features_df) and image_features_df["group"].nunique() == 2:
    show = ["brightness", "saturation", "colorfulness", "clutter_edge_density", "symmetry", "sharpness"]
    colors = {"Official (catalog)": "#3B6FA0", "Customer (UGC)": "#C2410C"}
    fig, axes = plt.subplots(2, 3, figsize=(11, 5.6))
    rng = np.random.RandomState(1)
    for ax, col in zip(axes.flat, show):
        for x, (grp, sub) in enumerate(image_features_df.groupby("group", sort=False)):
            jitter = rng.uniform(-0.12, 0.12, len(sub))
            ax.scatter(np.full(len(sub), x) + jitter, sub[col], s=36, color=colors[grp], alpha=0.85,
                       edgecolor="white", linewidth=1, label=grp, zorder=3)
            ax.hlines(sub[col].median(), x - 0.25, x + 0.25, color="#111827", linewidth=2, zorder=4)
        ax.set_title(col.replace("_", " "), fontsize=10, loc="left")
        ax.set_xticks([0, 1])
        ax.set_xticklabels([g.split(" (")[0] for g in image_features_df.groupby("group", sort=False).groups], fontsize=9)
        ax.set_xlim(-0.6, 1.6)
        for s in ["top", "right"]:
            ax.spines[s].set_visible(False)
        ax.grid(axis="y", color="#E5E7EB", linewidth=0.8, zorder=0)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", frameon=False, fontsize=9)
    fig.suptitle("Catalog vs customer photos -- interpretable visual features (black bar = median)",
                 fontsize=12, fontweight="bold", x=0.01, ha="left")
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig("image_features_catalog_vs_ugc.png", dpi=150)
    plt.show()

    # Dominant-colour palettes of the 19 catalog hero shots (colour is one of Session 6's named features)
    cat = image_features_df[~image_features_df["is_ugc"] & (image_features_df["source_group"] == "catalog")]
    if len(cat):
        fig, ax = plt.subplots(figsize=(9, 0.32 * len(cat) + 0.8))
        for y, row in enumerate(cat.itertuples()):
            x0 = 0
            for hex_, share in row.palette:
                ax.barh(y, share, left=x0, color=hex_, edgecolor="white", linewidth=2, height=0.8)
                x0 += share
        ax.set_yticks(range(len(cat)))
        ax.set_yticklabels([f"{r.image_id} {r.product}" for r in cat.itertuples()], fontsize=8)
        ax.set_xlim(0, 1)
        ax.set_xlabel("share of pixels (top-3 k-means colours)")
        ax.invert_yaxis()
        for s in ["top", "right", "left"]:
            ax.spines[s].set_visible(False)
        ax.set_title("Catalog hero shots -- dominant colour palettes", fontsize=11, fontweight="bold", loc="left")
        plt.tight_layout()
        plt.savefig("catalog_dominant_palettes.png", dpi=150)
        plt.show()
else:
    print("Needs both catalog and customer images loaded -- run in Colab.")


In [ ]:
# --- Data-driven reading of the chart (prints only what the numbers above actually show) ---
def pct_diff(a, b):
    return (a - b) / b * 100 if b else float("nan")

if len(image_features_df) and image_features_df["group"].nunique() == 2:
    med = image_features_df.groupby("group")[FEATURE_COLS].median()
    off, ugc = med.loc["Official (catalog)"], med.loc["Customer (UGC)"]
    n_off = (image_features_df["group"] == "Official (catalog)").sum()
    n_ugc = (image_features_df["group"] == "Customer (UGC)").sum()
    print(f"Official n={n_off}, customer n={n_ugc} -- descriptive only, too few customer photos for a significance test.\n")
    for col in FEATURE_COLS:
        d = pct_diff(ugc[col], off[col])
        print(f"  {col:<22} customer photos are {abs(d):5.1f}% {'higher' if d > 0 else 'lower'} than catalog (median)")
    print("\nHow to read this for the business case: catalog shots are controlled (even light, clean background,"
          "\nsymmetric studio pose). Customer photos show the garment in real light, on real bodies, in real rooms."
          "\nThe larger the gap on brightness/clutter/symmetry, the more the catalog is an idealised view --"
          "\nthat is the expectation gap behind 'looked different in person' fit returns, and why real try-on"
          "\nimagery belongs in the evidence base the assistant retrieves from.")


In [ ]:
# --- 4c-5. Transfer learning: pre-trained CLIP, zero-shot (Sessions 7-8) ---
# No training labels. Each class is described by English prompts; CLIP scores image-vs-prompt similarity.
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"   # ~600 MB download on first run, CPU is fine for ~30 images

IMAGE_TYPE_PROMPTS = {
    "studio_catalog":    "a professional studio photo of a model wearing activewear on a plain background",
    "ugc_mirror_selfie": "an amateur mirror selfie of a person trying on leggings at home",
    "fabric_closeup":    "a close-up photo of fabric texture on a piece of clothing",
    "screenshot":        "a screenshot of a shopping website or mobile app",
}
FRICTION_PROMPTS = {   # several prompts per class; class probability = sum over its prompts
    "product-quality": ["a close-up photo of fabric with pilling, snags or a hole",
                        "damaged, worn-out or faded leggings fabric"],
    "fit-sizing":      ["leggings that are too tight and too small on the person",
                        "activewear that is too loose and baggy on the person",
                        "see-through sheer leggings",
                        "leggings with a waistband rolling down"],
    "none":            ["activewear that fits well and looks brand new",
                        "a neutral product photo of clothing"],
}
CLIP_CONFIDENCE_GATE = 0.5   # below this, the image is tagged 'uncertain' instead of forcing a label

_clip = None
def get_clip():
    global _clip
    if _clip is None:
        from transformers import CLIPModel, CLIPProcessor
        _clip = (CLIPModel.from_pretrained(CLIP_MODEL_NAME).eval(), CLIPProcessor.from_pretrained(CLIP_MODEL_NAME))
    return _clip

def clip_probs(img_list, prompts):
    """(n_images x n_prompts) softmax probabilities."""
    import torch
    model, proc = get_clip()
    with torch.no_grad():
        inputs = proc(text=prompts, images=img_list, return_tensors="pt", padding=True)
        return model(**inputs).logits_per_image.softmax(dim=-1).cpu().numpy()

def clip_embed_images(img_list):
    import torch
    model, proc = get_clip()
    with torch.no_grad():
        e = model.get_image_features(**proc(images=img_list, return_tensors="pt"))
    e = e.cpu().numpy()
    return e / np.linalg.norm(e, axis=1, keepdims=True)

def clip_embed_text(texts):
    import torch
    model, proc = get_clip()
    with torch.no_grad():
        e = model.get_text_features(**proc(text=list(texts), return_tensors="pt", padding=True))
    e = e.cpu().numpy()
    return e / np.linalg.norm(e, axis=1, keepdims=True)

clip_df = pd.DataFrame(columns=["image_id", "clip_type", "clip_type_p", "clip_friction", "clip_friction_p"])
clip_img_ids, clip_img_emb = [], None
CLIP_OK = False
if images:
    try:
        ids = list(images.keys())
        imgs = [images[i] for i in ids]
        type_p = clip_probs(imgs, list(IMAGE_TYPE_PROMPTS.values()))
        flat_prompts, owner = [], []
        for cls, plist in FRICTION_PROMPTS.items():
            flat_prompts += plist
            owner += [cls] * len(plist)
        fr_p_raw = clip_probs(imgs, flat_prompts)
        classes = list(FRICTION_PROMPTS)
        fr_p = np.stack([fr_p_raw[:, [j for j, o in enumerate(owner) if o == c]].sum(axis=1) for c in classes], axis=1)
        type_names = list(IMAGE_TYPE_PROMPTS)
        clip_df = pd.DataFrame({
            "image_id": ids,
            "clip_type": [type_names[k] for k in type_p.argmax(1)],
            "clip_type_p": type_p.max(1).round(3),
            "clip_friction": [classes[k] if p >= CLIP_CONFIDENCE_GATE else "uncertain"
                              for k, p in zip(fr_p.argmax(1), fr_p.max(1))],
            "clip_friction_p": fr_p.max(1).round(3),
        })
        clip_img_ids, clip_img_emb = ids, clip_embed_images(imgs)
        CLIP_OK = True
    except Exception as e:
        print(f"CLIP unavailable here ({type(e).__name__}: {str(e)[:120]}) -- run in Colab.")

if CLIP_OK:
    clip_view = clip_df.merge(image_inventory[["image_id", "source_group", "product"]], on="image_id")
    print(clip_view[["image_id", "source_group", "clip_type", "clip_type_p", "clip_friction", "clip_friction_p"]].to_string(index=False))


In [ ]:
# --- 4c-6. Evaluating the vision model against labels we already know (Session 7: evaluating vision models) ---
# We know for certain whether each image is official brand imagery or customer-generated -- that is our
# ground truth. We never told CLIP. If it can't separate these two, its friction tags aren't worth trusting.
from sklearn.metrics import confusion_matrix, accuracy_score

if CLIP_OK:
    ev = clip_df.merge(image_inventory[["image_id", "source_group", "is_ugc"]], on="image_id")
    ev["truth"] = np.where(ev["is_ugc"], "customer", "official")
    ev["pred"] = np.where(ev["clip_type"].isin(["studio_catalog", "screenshot"]), "official", "customer")
    acc = accuracy_score(ev["truth"], ev["pred"])
    cm = pd.DataFrame(confusion_matrix(ev["truth"], ev["pred"], labels=["official", "customer"]),
                      index=["true official", "true customer"], columns=["pred official", "pred customer"])
    print(f"CLIP zero-shot, official-vs-customer photo: accuracy {acc:.0%} on n={len(ev)} (no training labels used)\n")
    print(cm.to_string())
    miss = ev[ev["truth"] != ev["pred"]]
    if len(miss):
        print("\nMisclassified (inspect these by eye):", ", ".join(miss["image_id"]))

    # Agreement: CLIP's pixel-based friction call vs the tag derived from the human-written note (v2 method)
    ma = media_df[media_df["Modality"] == "Image"].copy()
    ma["note_tag"] = (ma["Evidence_Note"].astype(str) + " " + ma["Fit_Return_Relevance"].astype(str)).apply(classify_blob)
    agree = ma[["Asset_ID", "Product", "Evidence_Note", "note_tag"]].merge(
        clip_df, left_on="Asset_ID", right_on="image_id", how="inner")
    if len(agree):
        agree["clip_as_taxonomy"] = agree["clip_friction"].replace({"none": "other", "uncertain": "other"})
        rate = (agree["clip_as_taxonomy"] == agree["note_tag"]).mean()
        print(f"\nMedia_Assets photos: CLIP pixel tag agrees with the human-note tag on {rate:.0%} of {len(agree)} images")
        print(agree[["Asset_ID", "Evidence_Note", "note_tag", "clip_friction", "clip_friction_p"]].to_string(index=False))
else:
    print("Skipped -- CLIP did not run (see 4c-5).")


In [ ]:
# --- 4c-7. VLM captioning with Gemini vision (replaces the old caption_image_stub) ---
import io
# Structured output, factual-only prompt, cached to disk so re-runs don't spend free-tier quota twice.
IMAGE_VLM_PROMPT = f"""You are analysing one photo for an apparel return-friction study (brand: Alo Yoga).
Return ONLY a JSON object with these keys:
"image_type": one of ["studio_catalog","ugc_try_on","fabric_closeup","screenshot","other"],
"garment": short description of the garment(s),
"fit_observation": what the photo visibly shows about fit (tight, loose, length, sheerness, rolling waistband), or "none visible",
"quality_observation": visible defects (pilling, snag, hole, fading, seam issue), or "none visible",
"friction_tag": one of {list(UNIFIED_TAXONOMY.keys())},
"caption": one factual sentence describing only what is visible,
"confidence": number from 0 to 1.
Describe only what is visible. Do not guess. Do not describe the person's identity, face or body beyond garment fit."""

IMG_VLM_CACHE = "cache_image_vlm.json"
image_vlm_cache = load_cache(IMG_VLM_CACHE)

def caption_image_vlm(image_id, img):
    if image_id in image_vlm_cache:
        return image_vlm_cache[image_id]
    if get_genai_client() is None:
        return None
    from google.genai import types
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    out, secs, toks = gemini_json([types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg"), IMAGE_VLM_PROMPT])
    if out:
        out["_seconds"], out["_tokens"] = secs, toks
        image_vlm_cache[image_id] = out
        save_cache(IMG_VLM_CACHE, image_vlm_cache)
        time.sleep(4)   # free-tier friendly pacing
    return out

vlm_rows = []
allowed = image_inventory.set_index("image_id")["external_api_ok"].to_dict()
if images and get_genai_client() is not None:
    for image_id, img in images.items():
        if not allowed.get(image_id, False):
            continue   # governance rule: visible face -> never sent to an external API
        out = caption_image_vlm(image_id, img)
        if out:
            vlm_rows.append({"image_id": image_id, **{k: out.get(k) for k in
                             ["image_type", "garment", "fit_observation", "quality_observation", "friction_tag", "caption", "confidence"]}})
elif images:
    # still use anything cached from an earlier run
    vlm_rows = [{"image_id": k, **{kk: v.get(kk) for kk in ["image_type", "garment", "fit_observation", "quality_observation",
                 "friction_tag", "caption", "confidence"]}} for k, v in image_vlm_cache.items() if isinstance(v, dict)]
image_vlm_df = pd.DataFrame(vlm_rows, columns=["image_id", "image_type", "garment", "fit_observation",
                                               "quality_observation", "friction_tag", "caption", "confidence"])
if len(image_vlm_df):
    image_vlm_df["friction_tag"] = image_vlm_df["friction_tag"].where(image_vlm_df["friction_tag"].isin(UNIFIED_TAXONOMY), "other")
    print(f"Gemini captioned {len(image_vlm_df)} images "
          f"({(~image_inventory['external_api_ok']).sum()} withheld by the face rule).")
    print(image_vlm_df[["image_id", "image_type", "friction_tag", "caption"]].to_string(index=False))
else:
    print("No Gemini captions yet -- set GEMINI_API_KEY in Colab Secrets and run in Colab.")


In [ ]:
# --- 4c-8. Final pixel-derived image tag per customer photo (feeds the fusion table in 6b) ---
# Priority: Gemini VLM tag (richest) -> CLIP tag if it cleared the confidence gate -> 'other'.
# Catalog images are official imagery, not friction evidence -- they are analysed above but not counted as complaints.
image_tags = image_inventory[image_inventory["is_ugc"]][["image_id", "source_group", "product"]].copy()
image_tags = image_tags.merge(clip_df[["image_id", "clip_friction", "clip_friction_p"]], on="image_id", how="left")
image_tags = image_tags.merge(image_vlm_df[["image_id", "friction_tag", "caption"]].rename(columns={"friction_tag": "vlm_tag"}),
                              on="image_id", how="left")

def final_image_tag(r):
    if isinstance(r["vlm_tag"], str):
        return r["vlm_tag"], "Gemini VLM"
    if isinstance(r["clip_friction"], str) and r["clip_friction"] in UNIFIED_TAXONOMY:
        return r["clip_friction"], "CLIP zero-shot"
    if isinstance(r["clip_friction"], str):
        return "other", "CLIP (none/uncertain)"
    return None, "not analysed (image not loaded)"

_res = [final_image_tag(r) for _, r in image_tags.iterrows()]
image_tags["pixel_tag"] = [a for a, _ in _res]
image_tags["pixel_tag_source"] = [b for _, b in _res]
print(image_tags[["image_id", "source_group", "product", "pixel_tag", "pixel_tag_source"]].to_string(index=False))


## 4e. UI/UX Screenshot Audit: The Return Journey as Customers See It

The brief's Image Package includes **"mobile app UI/UX discovery screenshots"**. For a return-friction project, the most relevant screens are the ones customers go through when they return something. **The team captures these (about 5 minutes; phone screenshots are fine) and saves them in a `ui_screenshots/` folder next to the notebook:**

| File name (suggested) | Screen | Why |
|---|---|---|
| `alo_01_returns_portal.png` | returns.aloyoga.com (order lookup) | Step 1 of the return flow (R006) |
| `alo_02_returns_policy.png` | aloyoga.com/pages/returns-exchanges | What eligibility and timing information is shown |
| `alo_03_size_guide.png` | Size guide pop-up on an Airlift legging page | Sizing communication before purchase |
| `alo_04_help_center.png` | Alo help center / FAQ | Where customers look first (R017) |
| `lulu_01_returns.png`, `vuori_01_returns.png` | Competitor return pages | Benchmark (6e) |

Each screenshot gets **pixel features** (clutter, contrast) plus a **Gemini UX audit**: which journey facts are visible (return window, refund timing, exchange route, fit-specific return reasons, final-sale rule, how to contact support), how many actions the screen asks for, and the friction it creates. Screenshots of public web pages carry no customer data, so they are safe to send to the API.

In [ ]:
# --- 4e-1. Gemini UX audit of return-journey screenshots ---
import glob, io
UI_DIR = "ui_screenshots"
UI_PROMPT = """You are auditing ONE screenshot of an apparel brand's website/app for return-journey friction.
Return ONLY JSON:
{"brand": string, "page": short description of the screen,
 "journey_step": one of ["pre-purchase sizing","return eligibility info","return request","exchange","refund status","support contact","other"],
 "facts_visible": {"return_window": bool, "refund_timing": bool, "exchange_option": bool, "fit_specific_return_reasons": bool,
                   "final_sale_rule": bool, "support_contact": bool, "size_guidance_by_height_or_measurement": bool},
 "actions_required": integer (clicks/fields the customer must complete on this screen),
 "friction_issues": list of short strings (only issues visible on this screen),
 "clarity_1to5": integer}
Judge only what is visible. Do not assume information that is not on the screen."""
ui_files = sorted(glob.glob(os.path.join(UI_DIR, "*.png")) + glob.glob(os.path.join(UI_DIR, "*.jp*g"))) if os.path.isdir(UI_DIR) else []
UI_CACHE = "cache_ui_audit.json"
ui_cache = load_cache(UI_CACHE)
ui_rows = []
for path in ui_files:
    img, status = load_image(path, max_side=1400)
    if img is None:
        continue
    feats = visual_features(img.resize((min(512, img.width), int(img.height * min(512, img.width) / img.width))))
    key = os.path.basename(path)
    if key not in ui_cache and get_genai_client() is not None:
        from google.genai import types
        buf = io.BytesIO(); img.save(buf, format="PNG")
        out, secs, toks = gemini_json([types.Part.from_bytes(data=buf.getvalue(), mime_type="image/png"), UI_PROMPT])
        if out:
            ui_cache[key] = out
            save_cache(UI_CACHE, ui_cache)
        time.sleep(4)
    a = ui_cache.get(key, {})
    a = a if isinstance(a, dict) else {}
    fv = a.get("facts_visible", {}) or {}
    ui_rows.append({"file": key, "brand": a.get("brand"), "page": a.get("page"), "step": a.get("journey_step"),
                    **{f"shows_{k}": fv.get(k) for k in ["return_window", "refund_timing", "exchange_option",
                                                         "fit_specific_return_reasons", "final_sale_rule", "support_contact"]},
                    "actions": a.get("actions_required"), "clarity": a.get("clarity_1to5"),
                    "issues": "; ".join(a.get("friction_issues", []) or []),
                    "clutter": round(feats["clutter_edge_density"], 3)})
ui_audit_df = pd.DataFrame(ui_rows)
if len(ui_audit_df):
    pd.set_option("display.max_colwidth", 120)
    print(ui_audit_df.drop(columns=["issues"]).to_string(index=False))
    print("\nFriction issues found on screen:")
    for r in ui_audit_df.itertuples():
        if r.issues:
            print(f"  {r.file}: {r.issues}")
    if "shows_refund_timing" in ui_audit_df:
        _shows = [c for c in ui_audit_df if c.startswith("shows_")]
        by_brand = ui_audit_df.assign(**{c: ui_audit_df[c].fillna(False).astype(bool) for c in _shows}).groupby("brand")[_shows].any()
        print("\nWhich journey facts each brand shows on ANY captured screen (cross-check with the policy scorecard, 6e-1):")
        print(by_brand.to_string())
else:
    print(f"No screenshots found in '{UI_DIR}/' -- capture the screens listed above and re-run this cell.")


## 5. Video Package -- Real Links, Tagged from Collected Timestamp Notes

19 rows across 6 unique videos (1 placeholder `watch?v=example` row excluded). All 6 sources checked this session:

| Video ID | Status |
|---|---|
| hB5c_Oh5BJU (Janice Eadie) | Verified real, matches claimed content (checked earlier session) |
| zOKAIHW1h5o (Peyton Johnson) | Verified real (checked earlier session) |
| OKtsMf_TzlE (Alex Michael May) | Verified real (checked earlier session) |
| weO5gRwjpKI | Verified real -- "ALO YOGA REVIEW" on YouTube |
| n6CjDa5dr1I | **Inconclusive** -- YouTube fetch was rate-limited (HTTP 429) both times it was tried this session; not disproven, just not independently confirmed. Flag for a manual spot-check if time allows. |
| tryongirls.com/.../yoga-try-on-haul-alo-15623.html | Confirmed live, non-fake page -- it's a **mirror/aggregator site** ("Bri Laurennn," Nov 2023) that embeds a YouTube video rather than a primary source. Kept, but labeled as a mirror in `source_type` below, per the same transparency principle used for the two mirror sources flagged earlier in the team's dataset. |

**Important limitation, stated plainly:** this sandbox cannot reach YouTube at all -- the shell's network is allowlisted to npm/pip/GitHub only, and the WebFetch route to YouTube pages is rate-limited/blocked. So we can't pull real ASR transcripts here. What we *can* do, and have done: use the **timestamp + product + fit-claim notes your team already collected** while watching each video (e.g. "8:02 Airbrush cropped leggings; creator gives size reference and says M runs true to size") as the analyzable text layer, and run the same friction-taxonomy classifier used on the text evidence against those notes. **v3:** Section 5c now runs that upgrade in Colab. It pulls full transcripts, has Gemini watch the videos directly, and analyses engagement metadata.
**Update:** Section 5b below adds a second, larger batch (131 more real customer/influencer videos, collected via targeted YouTube search) with a distinct, lower verification tier -- collected and title-tagged, not individually re-watched like the 6 above. Kept separate rather than blended into one number.


In [ ]:
# --- Video pipeline: load, verify status, and tag from collected notes ---
def load_video_data(path="Alo_Yoga_Data_Collection_Template_CLEANED.xlsx"):
    try:
        df = pd.read_excel(path, sheet_name="Video_Data", header=3)
        df = df.dropna(subset=["id"])
        df = df[~df["link"].str.contains("watch?v=example", regex=False, na=False)]
        return df.reset_index(drop=True)
    except FileNotFoundError:
        print("Video data not collected yet.")
        return pd.DataFrame(columns=["id", "type", "link", "platform", "notes"])

video_df = load_video_data()

VIDEO_SOURCE_STATUS = {
    "https://www.youtube.com/watch?v=hB5c_Oh5BJU": "verified",
    "https://www.youtube.com/watch?v=zOKAIHW1h5o": "verified",
    "https://www.youtube.com/watch?v=OKtsMf_TzlE": "verified",
    "https://www.youtube.com/watch?v=weO5gRwjpKI": "verified",
    "https://www.youtube.com/watch?v=n6CjDa5dr1I": "inconclusive_rate_limited",
    "https://www.tryongirls.com/video/yoga-try-on-haul-alo-15623.html": "verified_mirror_source",
}
video_df["source_status"] = video_df["link"].map(VIDEO_SOURCE_STATUS).fillna("unverified")
video_df["friction_tag"] = video_df["notes"].apply(classify_blob)

print(f"Loaded {len(video_df)} video-evidence rows across {video_df['link'].nunique()} unique videos.")
print(video_df["source_status"].value_counts())
print()
print(video_df["friction_tag"].value_counts())
video_df[["id", "link", "source_status", "friction_tag", "notes"]].head(10)

### 5b. Video Batch 2 -- Targeted YouTube Search Results (Real, Collected -- Title-Tagged, Not Individually Re-Verified)

A second, larger video collection pass: 19 targeted YouTube searches (e.g. `alo yoga pilling`, `alo yoga sizing guide try on`, `alo yoga returns`, `alo yoga vs lululemon try on`, `#aloyogatryon`) run via Apify's YouTube Search Scraper, 10 results each. This is **real, live-scraped data** -- actual video titles, channel names, view counts, and URLs, not fabricated -- but it carries a different, lower verification tier than the 6 videos above: those were individually opened and confirmed live during this session; these 131 were filtered and classified by title/search-context alone (the same `classify_blob` used throughout, applied for the first time to video *titles* rather than collected timestamp notes), not individually watched. That distinction is preserved below rather than blurred into one "verified" number.

Two real filtering steps were needed before this data was usable:
- **36 of 190 results were Alo's own official channel uploads** (workout classes, brand campaigns) -- excluded, since official brand content isn't customer/influencer evidence.
- **21 more were false-positive search noise** -- Lululemon corporate/stock-market news videos that surfaced for queries like `alo yoga quality declined`, unrelated to any actual garment. Caught by requiring the title to actually name Alo/Aloyoga and excluding finance-report language (stock, billion, turnaround, etc.).

In [ ]:
# --- Load and filter the second video batch (real Apify YouTube Search Scraper output) ---
import re as _re2

video_batch2_raw = pd.read_excel("Alo_Yoga_YouTube_Search_Batch2.xlsx")

# Exclude Alo's own official channel -- brand content, not customer/influencer evidence
customer_rows = video_batch2_raw[video_batch2_raw["channelName"] != "ALO"].drop_duplicates(subset=["id"]).copy()

title_lower = customer_rows["title"].astype(str).str.lower()
mentions_alo = title_lower.str.contains(r"\balo\b|aloyoga|alo yoga", regex=True)
finance_noise = title_lower.str.contains(
    r"stock|billion|brand\b.*decline|destroyed|edge|slump|guide cut|turnaround|structural|ceo search|rip-?off|fy20|\$\d+b\b",
    regex=True,
)
genuine = customer_rows[mentions_alo & ~finance_noise].copy()

# De-dupe against the 6 videos already verified in Section 5 above (by YouTube video ID)
already_ids = set(video_df["link"].astype(str).apply(lambda u: u.split("v=")[-1].split("/")[-1]))
genuine["vid_id"] = genuine["url"].astype(str).apply(lambda u: u.split("v=")[-1].split("/")[-1])
genuine = genuine[~genuine["vid_id"].isin(already_ids)]

genuine["blob"] = genuine["title"].astype(str) + " " + genuine["input"].astype(str)
genuine["friction_tag"] = genuine["blob"].apply(classify_blob)

video_batch2_df = pd.DataFrame({
    "id": ["YT2-" + str(i) for i in range(len(genuine))],
    "link": genuine["url"].values,
    "platform": "YouTube",
    "notes": genuine["title"].values,
    "source_status": "collected_title_tagged_not_individually_verified",
    "friction_tag": genuine["friction_tag"].values,
})

print(f"Batch 2: {len(video_batch2_raw)} raw results -> {len(customer_rows)} non-official-channel -> "
      f"{len(genuine)} genuine, on-topic, de-duped customer/influencer videos across "
      f"{genuine['channelName'].nunique()} distinct channels")
print()
print(video_batch2_df["friction_tag"].value_counts())

**Why this matters for the fusion section above (6b):** the coverage table found zero video evidence for `product-quality`. Batch 2 changes that -- real customer videos like *"Alo Yoga is the LITERAL WORST"* (422K views) and *"How To Remove Pilling From Your Leggings // ALO Motto"* are genuine video-modality evidence for the durability complaints already documented in text. `video_df` is extended below so this flows through to every downstream count (the friction map, the fusion coverage table) without needing separate logic.

In [ ]:
# --- Fold Batch 2 into the same video_df used everywhere downstream (friction map, fusion table) ---
video_df = pd.concat([video_df, video_batch2_df], ignore_index=True)
print(f"video_df now: {len(video_df)} total rows -- "
      f"{(video_df['source_status']=='verified').sum() + (video_df['source_status']=='verified_mirror_source').sum() + (video_df['source_status']=='inconclusive_rate_limited').sum()} from the "
      f"6 individually-verified videos (Section 5), "
      f"{(video_df['source_status']=='collected_title_tagged_not_individually_verified').sum()} from Batch 2 (collected + title-tagged, not individually re-watched)")
video_df["friction_tag"].value_counts()

## 5c. Video Analytics: Engagement, Audio and Visual Content (Sessions 11-12)

Up to v2, video evidence was tagged only from **titles** (Batch 2) and **hand-written timestamp notes** (the 6 verified videos), and 130 of 149 video rows ended up as `other`. Session 11 describes a video as three streams: *frames, audio and metadata*. This section analyses all three:

| Stream | Method | Course link |
|---|---|---|
| Metadata / engagement | Views, likes, comments, duration and upload age from the Apify scrape; does complaint content get more reach? | Session 11 (Munaro et al. 2021, YouTube features and consumer engagement) |
| Audio | Full YouTube transcripts, chunked into 60-second windows, each tagged with the same taxonomy, so every friction mention gets a timestamp | Session 11 (deconstructing audio) |
| Frames | Gemini watches the video directly from its YouTube URL and returns timestamped segments: what the creator *says* vs. what is *visible* | Session 12 (VLMs, agentic video search) |

Session 12 also covers **accuracy vs latency/throughput**, so 5c-5 records the actual cost of each method per video.

In [ ]:
# --- 5c-1. Engagement dynamics from the Batch 2 metadata (Session 11, Munaro et al. 2021) ---
from scipy import stats

def col_or_nan(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series(np.nan, index=df.index)

def parse_duration_min(s):
    if pd.isna(s):
        return np.nan
    if isinstance(s, (int, float, np.integer, np.floating)):
        return float(s) / 60   # seconds
    parts = [p for p in str(s).strip().split(":") if p != ""]
    try:
        nums = [float(p) for p in parts]
    except ValueError:
        return np.nan
    secs = 0
    for n in nums:
        secs = secs * 60 + n
    return secs / 60

CONTENT_RULES = [   # first match wins; order matters
    ("care / how-to",        r"how to|remove|wash|fix|care|de-?pill"),
    ("complaint / negative", r"worst|regret|disappoint|not worth|overrated|scam|waste|return(ed|ing)? (it|them|everything)|pilling|pilled|ripped|quality (is|has)|declin"),
    ("comparison / dupe",    r"\bvs\.?\b|versus|dupe|compar"),
    ("try-on / haul",        r"try[- ]?on|haul|unbox"),
    ("review",               r"review"),
]
def content_type(title):
    t = str(title).lower()
    for name, pat in CONTENT_RULES:
        if re.search(pat, t):
            return name
    return "other"

vm = genuine.copy()
vm["views"] = pd.to_numeric(col_or_nan(vm, "viewCount", "views"), errors="coerce")
vm["likes"] = pd.to_numeric(col_or_nan(vm, "likes", "likeCount"), errors="coerce")
vm["comments"] = pd.to_numeric(col_or_nan(vm, "commentsCount", "commentCount"), errors="coerce")
vm["duration_min"] = col_or_nan(vm, "duration").apply(parse_duration_min)
vm["upload_date"] = pd.to_datetime(col_or_nan(vm, "date", "uploadDate", "publishedAt"), errors="coerce", utc=True)
vm["age_days"] = (pd.Timestamp.now(tz="UTC") - vm["upload_date"]).dt.days.clip(lower=1)
vm["views_per_day"] = vm["views"] / vm["age_days"]
vm["like_rate"] = vm["likes"] / vm["views"]
vm["subscribers"] = pd.to_numeric(col_or_nan(vm, "numberOfSubscribers", "subscribers"), errors="coerce")
vm["views_per_subscriber"] = vm["views"] / vm["subscribers"].replace(0, np.nan)
vm["content_type"] = vm["title"].apply(content_type)
vm["title_len"] = vm["title"].astype(str).str.len()
vm["title_has_question_or_exclaim"] = vm["title"].astype(str).str.contains(r"[?!]").astype(int)
vm["title_caps_words"] = vm["title"].astype(str).apply(lambda t: sum(w.isupper() and len(w) > 2 for w in t.split()))

HAVE_ENGAGEMENT = vm["views"].notna().sum() >= 10
if not HAVE_ENGAGEMENT:
    print("View counts not found in Alo_Yoga_YouTube_Search_Batch2.xlsx -- engagement analysis skipped. "
          "Columns available:", list(genuine.columns))
else:
    eng = (vm.groupby("content_type")
             .agg(videos=("title", "size"), median_views=("views", "median"), total_views=("views", "sum"),
                  median_views_per_day=("views_per_day", "median"), median_like_rate=("like_rate", "median"))
             .sort_values("median_views", ascending=False))
    print("Engagement by content type (Batch 2, n =", len(vm), "customer/influencer videos):")
    print(eng.round({"median_views": 0, "total_views": 0, "median_views_per_day": 1, "median_like_rate": 4}).to_string())

    reach = vm.groupby("friction_tag")["views"].agg(["size", "sum", "median"]).rename(
        columns={"size": "videos", "sum": "total_views", "median": "median_views"}).sort_values("total_views", ascending=False)
    print("\nReach by friction tag (title-based tag):")
    print(reach.round(0).to_string())

    # Do quality-complaint videos reach more people -- and is that just because they come from bigger channels?
    pq = vm["friction_tag"].eq("product-quality")
    if pq.sum() >= 3:
        for metric, label in [("views", "raw views"), ("views_per_subscriber", "views per channel subscriber")]:
            a_, b_ = vm.loc[pq, metric].dropna(), vm.loc[~pq, metric].dropna()
            if len(a_) >= 3 and len(b_) >= 3:
                p_ = stats.mannwhitneyu(a_, b_, alternative="two-sided").pvalue
                print(f"\nproduct-quality videos (n={len(a_)}) vs the rest (n={len(b_)}), {label}: "
                      f"median {a_.median():,.1f} vs {b_.median():,.1f}, Mann-Whitney p = {p_:.3f}")
        print("If the raw-views gap is significant but the per-subscriber gap is not, complaint videos travel far because"
              "\nthey come from bigger channels -- channel size, not the complaint itself, drives the reach (Munaro et al.'s"
              "\npoint that creator features must be controlled for).")

    # Data-quality check on the v2 Batch 2 tags: classify_blob was run on title + the SEARCH QUERY that found the video.
    title_only = vm["title"].astype(str).apply(classify_blob)
    from_query = (vm["friction_tag"] != "other") & (title_only == "other")
    print(f"\nTag provenance: {int((vm['friction_tag'] != 'other').sum())} Batch 2 videos carry a friction tag; "
          f"{int(from_query.sum())} of them get it only from the search query (e.g. 'alo yoga pilling'), not from the title.")
    if from_query.any():
        print("   " + "\n   ".join(f"[{t}] {ti[:70]}" for t, ti in vm.loc[from_query, ["friction_tag", "title"]].values[:8]))
        print("These are 'found by a quality search' rather than 'is about quality' -- 5c-4 sends them to Gemini to check the content.")

    comp = vm["content_type"].isin(["complaint / negative", "care / how-to"])
    a, b = vm.loc[comp, "views_per_day"].dropna(), vm.loc[~comp, "views_per_day"].dropna()
    if len(a) >= 3 and len(b) >= 3:
        u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        print(f"\nComplaint + care/how-to videos (n={len(a)}) vs all others (n={len(b)}): "
              f"median views/day {a.median():.1f} vs {b.median():.1f}; Mann-Whitney U p = {p:.3f}"
              f" -> {'a statistically significant' if p < 0.05 else 'NOT a statistically significant'} difference at 5%.")

    feats = ["subscribers", "duration_min", "age_days", "title_len", "title_has_question_or_exclaim", "title_caps_words", "like_rate"]
    rows = []
    for f in feats:
        d = vm[[f, "views_per_day"]].dropna()
        if len(d) >= 10 and d[f].nunique() > 1:
            rho, p = stats.spearmanr(d[f], np.log1p(d["views_per_day"]))
            rows.append({"feature": f, "spearman_rho_vs_log_views_per_day": round(rho, 3), "p_value": round(p, 3), "n": len(d)})
    if rows:
        print("\nWhich video features move with reach? (Spearman, rank-based, robust to viral outliers)")
        print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# --- 5c-2. Chart: reach by content type ---
if HAVE_ENGAGEMENT:
    order = vm.groupby("content_type")["views_per_day"].median().sort_values().index.tolist()
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    rng = np.random.RandomState(2)
    for y, ct in enumerate(order):
        vals = np.log10(vm.loc[vm["content_type"] == ct, "views_per_day"].dropna() + 1)
        ax.scatter(vals, np.full(len(vals), y) + rng.uniform(-0.15, 0.15, len(vals)), s=28,
                   color="#3B6FA0", alpha=0.7, edgecolor="white", linewidth=0.8, zorder=3)
        if len(vals):
            ax.vlines(vals.median(), y - 0.3, y + 0.3, color="#111827", linewidth=2, zorder=4)
            ax.text(vals.max() + 0.08, y, f"n={len(vals)}", va="center", fontsize=8, color="#374151")
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=9)
    ax.set_xlabel("views per day since upload (log10 scale; 1 = 10/day, 3 = 1,000/day)")
    ax.set_title("Which Alo Yoga videos reach people? (black bar = median)", fontsize=12, fontweight="bold", loc="left")
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.grid(axis="x", color="#E5E7EB", linewidth=0.8, zorder=0)
    plt.tight_layout()
    plt.savefig("video_engagement_by_content_type.png", dpi=150)
    plt.show()


In [ ]:
# --- 5c-3. Audio layer: full transcripts, 60-second windows, each tagged with the friction taxonomy ---
# Needs Colab (YouTube is unreachable from our build sandbox). YouTube sometimes blocks transcript requests from
# cloud IPs ("RequestBlocked"); those videos are logged as unavailable, not silently dropped.
def yt_id(url):
    m = re.search(r"(?:v=|youtu\.be/|shorts/)([\w-]{11})", str(url))
    return m.group(1) if m else None

def fetch_transcript(video_id):
    from youtube_transcript_api import YouTubeTranscriptApi
    try:
        fetched = YouTubeTranscriptApi().fetch(video_id, languages=["en", "en-US", "en-GB"])   # v1.x API
        return [(float(s.start), s.text) for s in fetched]
    except AttributeError:
        segs = YouTubeTranscriptApi.get_transcript(video_id, languages=["en", "en-US", "en-GB"])  # pre-1.0 API
        return [(float(s["start"]), s["text"]) for s in segs]

def chunk_transcript(segments, window=60):
    chunks = {}
    for start, text in segments:
        chunks.setdefault(int(start // window), []).append(text)
    return [(k * window, " ".join(v).replace("\n", " ")) for k, v in sorted(chunks.items())]

video_df["vid"] = video_df["link"].apply(yt_id)
vm["vid"] = vm["url"].apply(yt_id)
TRANSCRIPT_MAX_VIDEOS = 40
priority = pd.concat([
    video_df.loc[video_df["source_status"] != "collected_title_tagged_not_individually_verified", "vid"],   # the 6 verified
    vm.loc[vm["friction_tag"] != "other", "vid"],                                                            # title already signals friction
    vm.sort_values("views", ascending=False)["vid"] if HAVE_ENGAGEMENT else vm["vid"],                     # then highest reach
]).dropna().drop_duplicates().head(TRANSCRIPT_MAX_VIDEOS).tolist()

TRANSCRIPT_CACHE = "cache_transcripts.json"
transcript_cache = load_cache(TRANSCRIPT_CACHE)
tx_status = {vid: "ok (cached)" for vid in priority if vid in transcript_cache}
if not HAVE["yt_transcripts"]:
    print("youtube-transcript-api not available -- using cached transcripts only.")
else:
    for vid in priority:
        if vid in transcript_cache:
            continue
        try:
            transcript_cache[vid] = fetch_transcript(vid)
            tx_status[vid] = "ok"
            time.sleep(1)
        except Exception as e:
            tx_status[vid] = type(e).__name__
    save_cache(TRANSCRIPT_CACHE, transcript_cache)

chunk_rows = []
for vid in priority:
    for start, text in chunk_transcript(transcript_cache.get(vid, [])):
        chunk_rows.append({"vid": vid, "start_s": start, "timestamp": f"{start // 60:02d}:{start % 60:02d}",
                           "text": text, "tag": classify_blob(text)})
transcript_chunks_df = pd.DataFrame(chunk_rows, columns=["vid", "start_s", "timestamp", "text", "tag"])

print(f"Transcripts requested for {len(priority)} videos: "
      f"{sum(s.startswith('ok') for s in tx_status.values())} fetched, "
      f"{sum(not s.startswith('ok') for s in tx_status.values())} unavailable {dict(pd.Series([s for s in tx_status.values() if not s.startswith('ok')]).value_counts()) if tx_status else ''}")
if len(transcript_chunks_df):
    print(f"{len(transcript_chunks_df)} one-minute windows; friction mentions by tag:")
    print(transcript_chunks_df["tag"].value_counts().to_string())
    # A video's transcript tag = its most frequent non-'other' tag, counted only if it appears in >= 2 windows
    # (one stray 'length' or 'quality' in a 15-minute haul shouldn't re-label the whole video).
    tagged = transcript_chunks_df[transcript_chunks_df["tag"] != "other"]
    counts = tagged.groupby(["vid", "tag"]).size().reset_index(name="windows")
    counts = counts[counts["windows"] >= 2].sort_values("windows", ascending=False).drop_duplicates("vid")
    transcript_video_tag = dict(zip(counts["vid"], counts["tag"]))
    print(f"\n{len(transcript_video_tag)} videos get a transcript-based friction tag.")
    print("\nSample of timestamped friction mentions:")
    print(tagged.head(8)[["vid", "timestamp", "tag", "text"]].assign(text=lambda d: d["text"].str[:110]).to_string(index=False))
else:
    transcript_video_tag = {}
    print("No transcripts available in this runtime -- run in Colab.")


In [ ]:
# --- 5c-4. Frames: Gemini watches the video itself (Session 12, vision-language models) ---
# Gemini accepts a public YouTube URL directly -- no download, no frame extraction on our side.
# Capped at 10 videos: each call processes the whole video and is the slowest, most expensive step here.
VIDEO_VLM_PROMPT = f"""Watch this YouTube video about Alo Yoga products. Return ONLY a JSON object:
{{"video_summary": one sentence,
 "creator_sentiment_toward_alo": "positive" | "mixed" | "negative",
 "segments": up to 8 objects, each {{"timestamp": "mm:ss", "product": string, "size_worn": string or null,
     "spoken_claim": what the creator says about fit, quality, price or returns,
     "visual_evidence": what is visibly on screen that supports or contradicts the claim, or "none",
     "friction_tag": one of {list(UNIFIED_TAXONOMY.keys())} -- use "other" UNLESS the creator reports a problem or the visuals show one (fits worse than expected or than the brand claims, a defect, a return/refund issue). Describing, holding up or neutrally wearing an item is "other"; an intentionally oversized style is not a fit problem.}},
 "visual_contradicts_speech": true if anything visible contradicts what is said (e.g. says 'fits perfectly' while the waistband visibly rolls), else false,
 "contradiction_note": string or null}}
Report only what is actually said or visible. Do not guess."""

VLM_MAX_VIDEOS = 15
# Five complaint videos identified from the team's thumbnail dataset ("alo yoga return experience" search) --
# selected by what the video title says, not by the search query, so they are genuine quality-complaint candidates.
EXTRA_COMPLAINT_VIDEOS = ["T7oIjadKyfA", "Mvv4UXb5jeo", "_uXCgC-hy5M", "QyknVYdEfsk", "-4t8RnZCTjU"]
vlm_targets = pd.concat([
    pd.Series(EXTRA_COMPLAINT_VIDEOS),
    video_df.loc[video_df["source_status"] != "collected_title_tagged_not_individually_verified", "vid"],
    vm.loc[vm["friction_tag"].isin(["product-quality", "fit-sizing"]) |
           vm["content_type"].isin(["complaint / negative", "care / how-to"]), "vid"],
]).dropna().drop_duplicates().head(VLM_MAX_VIDEOS).tolist()

VIDEO_VLM_CACHE = "cache_video_vlm_v2.json"   # v2 = stricter friction definition; old cache (topic-level tags) is not reused
video_vlm_cache = load_cache(VIDEO_VLM_CACHE)
if get_genai_client() is not None:
    from google.genai import types
    for vid in vlm_targets:
        if vid in video_vlm_cache:
            continue
        out, secs, toks = gemini_json([types.Part(file_data=types.FileData(file_uri=f"https://www.youtube.com/watch?v={vid}")),
                                       VIDEO_VLM_PROMPT])
        if out:
            out["_seconds"], out["_tokens"] = secs, toks
            video_vlm_cache[vid] = out
            save_cache(VIDEO_VLM_CACHE, video_vlm_cache)
        time.sleep(6)

seg_rows, contradictions = [], []
for vid, out in video_vlm_cache.items():
    if not isinstance(out, dict):
        continue
    for s in (out.get("segments") or []):
        if not isinstance(s, dict):
            continue
        tag = s.get("friction_tag") if s.get("friction_tag") in UNIFIED_TAXONOMY else "other"
        seg_rows.append({"vid": vid, "timestamp": s.get("timestamp"), "product": s.get("product"),
                         "size_worn": s.get("size_worn"), "spoken_claim": s.get("spoken_claim"),
                         "visual_evidence": s.get("visual_evidence"), "tag": tag})
    if out.get("visual_contradicts_speech"):
        contradictions.append({"vid": vid, "note": out.get("contradiction_note"),
                               "sentiment": out.get("creator_sentiment_toward_alo")})
vlm_segments_df = pd.DataFrame(seg_rows, columns=["vid", "timestamp", "product", "size_worn", "spoken_claim", "visual_evidence", "tag"])

if len(vlm_segments_df):
    print(f"Gemini analysed {len(video_vlm_cache)} videos -> {len(vlm_segments_df)} timestamped segments.")
    print(vlm_segments_df["tag"].value_counts().to_string())
    print("\nCreator sentiment:", pd.Series([o.get("creator_sentiment_toward_alo") for o in video_vlm_cache.values() if isinstance(o, dict)]).value_counts().to_dict())
    vis = vlm_segments_df[~vlm_segments_df["visual_evidence"].astype(str).str.lower().isin(["none", "", "nan", "none visible"])]
    print(f"\n{len(vis)} segments have on-screen visual evidence (not just speech). Examples:")
    print(vis.head(6)[["vid", "timestamp", "product", "tag", "visual_evidence"]].to_string(index=False))
    if contradictions:
        print("\nWithin-video cross-modal CONFLICT (speech vs visuals):")
        print(pd.DataFrame(contradictions).to_string(index=False))
    seg_counts = vlm_segments_df[vlm_segments_df["tag"] != "other"].groupby(["vid", "tag"]).size().reset_index(name="n")
    vlm_video_tag = dict(seg_counts.sort_values("n", ascending=False).drop_duplicates("vid")[["vid", "tag"]].values)
else:
    vlm_video_tag = {}
    print("No VLM video results yet -- set GEMINI_API_KEY and run in Colab.")


In [ ]:
# --- 5c-5. Accuracy vs latency/cost across the three video methods (Session 12) ---
t0 = time.time()
_ = [classify_blob(n) for n in video_df["notes"].astype(str)]
notes_secs = (time.time() - t0) / max(len(video_df), 1)

vlm_secs = [o.get("_seconds") for o in video_vlm_cache.values() if isinstance(o, dict) and o.get("_seconds")]
vlm_toks = [o.get("_tokens") for o in video_vlm_cache.values() if isinstance(o, dict) and o.get("_tokens")]
tradeoff = pd.DataFrame([
    {"method": "Title / notes keyword tagging", "videos": video_df["vid"].nunique(),
     "sees": "title or hand-written notes only", "sec_per_video": round(notes_secs, 5), "tokens_per_video": 0,
     "videos_not_other": int(video_df.drop_duplicates("link")["friction_tag"].ne("other").sum())},
    {"method": "Transcript (audio) windows", "videos": transcript_chunks_df["vid"].nunique(),
     "sees": "everything said, timestamped", "sec_per_video": "~1-2 (network)", "tokens_per_video": 0,
     "videos_not_other": len(transcript_video_tag)},
    {"method": "Gemini VLM (frames + audio)", "videos": len(video_vlm_cache),
     "sees": "what is said AND what is visible", "sec_per_video": round(float(np.median(vlm_secs)), 1) if vlm_secs else None,
     "tokens_per_video": int(np.median(vlm_toks)) if vlm_toks else None, "videos_not_other": len(vlm_video_tag)},
])
print(tradeoff.to_string(index=False))
print("\nDesign implication: keyword tagging is effectively free but blind (most videos stay 'other'); the VLM sees the"
      "\nmost but costs orders of magnitude more per video. A production pipeline would cascade them: tag every"
      "\nvideo cheaply, send only high-reach or friction-flagged videos to transcripts, and only the most-viewed"
      "\ncomplaint videos to the VLM.")


In [ ]:
# --- 5c-6. Upgrade video tags with content evidence (flows into the friction map in Section 6) ---
# Only rows the title/notes left as 'other' are upgraded; a VLM tag outranks a transcript tag.
video_df["friction_tag_metadata"] = video_df["friction_tag"]
content_tag = {**transcript_video_tag, **vlm_video_tag}
upgrade = video_df["friction_tag"].eq("other") & video_df["vid"].isin(content_tag)
video_df.loc[upgrade, "friction_tag"] = video_df.loc[upgrade, "vid"].map(content_tag)
video_df["tag_source"] = np.where(video_df["vid"].isin(vlm_video_tag) & upgrade, "Gemini VLM",
                          np.where(upgrade, "transcript", "title/notes"))
print(f"{int(upgrade.sum())} video rows upgraded from 'other' using transcript/VLM content.")
print(pd.DataFrame({"before (title/notes)": video_df["friction_tag_metadata"].value_counts(),
                    "after (+ content)": video_df["friction_tag"].value_counts()}).fillna(0).astype(int).to_string())


## 5d. Influencer Marketing and Short-Form Video Dynamics

The brief names **"shifting influencer marketing dynamics"** and **"short-form digital video engagement"** as threats. It also lists **official brand ad campaigns** in the Video Package. Using the scraped metadata, this section tests three things:

1. **Brand vs creators:** does Alo's own channel reach people as effectively as independent creators, after normalising for audience size?
2. **Video length:** do shorter videos get more engagement per subscriber? Our Alo scrape has few true shorts, so **Lululemon's Instagram Reels** serve as the short-form benchmark.
3. **Paid content:** how much content is disclosed as sponsored? This feeds the disclosure rule in governance (9f).

In [ ]:
# --- 5d-1. Brand channel vs independent creators, and video length ---
b2 = video_batch2_raw.drop_duplicates("id").copy()
b2["views"] = pd.to_numeric(b2["viewCount"], errors="coerce")
b2["likes_n"] = pd.to_numeric(b2["likes"], errors="coerce")
b2["subscribers"] = pd.to_numeric(col_or_nan(b2, "numberOfSubscribers"), errors="coerce")
b2["duration_min"] = b2["duration"].apply(parse_duration_min)
b2["views_per_subscriber"] = b2["views"] / b2["subscribers"].replace(0, np.nan)
b2["like_rate"] = b2["likes_n"] / b2["views"]
b2["who"] = np.where(b2["channelName"] == "ALO", "Alo official channel", np.where(b2["id"].isin(genuine["id"]), "Independent creators", "excluded (noise)"))
b2 = b2[b2["who"] != "excluded (noise)"]

who = b2.groupby("who").agg(videos=("id", "size"), median_views=("views", "median"),
                            median_views_per_subscriber=("views_per_subscriber", "median"),
                            median_like_rate=("like_rate", "median"), median_minutes=("duration_min", "median"))
print("Brand vs creators (Batch 2 scrape):")
print(who.round(3).to_string())
from scipy import stats as _st
a_, c_ = b2.loc[b2["who"] == "Alo official channel", "views_per_subscriber"].dropna(), b2.loc[b2["who"] == "Independent creators", "views_per_subscriber"].dropna()
if len(a_) >= 5 and len(c_) >= 5:
    print(f"Views per subscriber, official vs creators: Mann-Whitney p = {_st.mannwhitneyu(a_, c_).pvalue:.3f}")
print("Note: Alo's channel posts long workout classes (Alo Moves), not product ads -- so 'official' here means brand-owned"
      "\ncontent. The brand's product story reaches people mostly through creators it does not control.")

cr = b2[b2["who"] == "Independent creators"].copy()
cr["length"] = pd.cut(cr["duration_min"], [0, 3, 10, 20, 1e9], labels=["<=3 min", "3-10 min", "10-20 min", ">20 min"])
lb = cr.groupby("length", observed=False).agg(videos=("id", "size"), median_views_per_subscriber=("views_per_subscriber", "median"),
                                              median_like_rate=("like_rate", "median"))
print("\nCreator videos by length:")
print(lb.round(3).to_string())
paid = col_or_nan(b2, "isPaidContent").astype(str).str.lower().eq("true")
print(f"\nDisclosed paid/sponsored content: {int(paid.sum())} of {len(b2)} videos ({paid.mean():.1%}).")


In [ ]:
# --- 5d-2. Short-form benchmark: Lululemon Instagram Reels vs images vs carousels ---
import glob
IG_PATH = next(iter(sorted(glob.glob("dataset_instagram-scraper_*.csv"))), None)
if IG_PATH is None:
    print("Instagram scrape not found -- short-form benchmark skipped.")
else:
    igp = pd.read_csv(IG_PATH, low_memory=False).copy()
    igp["format"] = igp["type"].map({"Video": "Reel / video", "Image": "Single image", "Sidecar": "Carousel"}).fillna(igp["type"])
    igp["views"] = pd.to_numeric(igp.get("videoViewCount"), errors="coerce").fillna(pd.to_numeric(igp.get("videoPlayCount"), errors="coerce"))
    igp["comments_per_1k_likes"] = pd.to_numeric(igp["commentsCount"], errors="coerce") / pd.to_numeric(igp["likesCount"], errors="coerce").replace(0, np.nan) * 1000
    fmt = igp.groupby("format").agg(posts=("id", "size"), median_likes=("likesCount", "median"),
                                    median_comments=("commentsCount", "median"), median_video_views=("views", "median"),
                                    comments_per_1k_likes=("comments_per_1k_likes", "median"))
    print(f"Lululemon Instagram ({', '.join(sorted(igp['ownerUsername'].unique()))}), engagement by format:")
    print(fmt.round(1).to_string())
    print(f"Disclosed paid partnerships: {int(igp['paidPartnership'].astype(str).str.lower().eq('true').sum())} of {len(igp)} posts.")
    print("\nReading: this is a competitor's brand-owned feed, so it benchmarks FORMAT, not Alo's audience. Comments under"
          "\nthese posts also double as an unofficial support channel -- 6e-2 found refund and support complaints posted"
          "\non product launches, which is itself a governance point: complaints surface where the brand markets.")


## 5e. Visual Brand Alignment of Creator Content (Sessions 6-7 + 11)

The brief asks the taskforce to **"evaluate visual brand alignment"**. We measure it where most customers meet the brand: creator video thumbnails. For each thumbnail:

- **Brand alignment score:** the cosine similarity between the thumbnail's CLIP embedding and the **centroid of Alo's official catalog imagery** (4c). A high score means the thumbnail looks like Alo's own visual language.
- **Interpretable features** from 4c: colourfulness, brightness, clutter. Session 7's reading (Yu & Egger 2021) links colour to engagement.

We then test whether on-brand or colourful thumbnails get more **views per subscriber**. This joins the image and video pipelines in one analysis. Thumbnails are public, fetched by URL, and contain no customer data.

In [ ]:
# --- 5e-1. Thumbnail brand alignment vs engagement (Colab: needs thumbnail downloads + CLIP from 4c) ---
b2["thumbnailUrl"] = col_or_nan(b2, "thumbnailUrl")
thumbs = b2[["id", "who", "thumbnailUrl", "views_per_subscriber", "views"]].dropna(subset=["thumbnailUrl"]).copy()
thumb_imgs = {}
for r in thumbs.itertuples():
    img, st_ = load_image(r.thumbnailUrl, max_side=384)
    if img is not None:
        thumb_imgs[r.id] = img
print(f"Thumbnails loaded: {len(thumb_imgs)} of {len(thumbs)}")
cat_ids = [i for i in clip_img_ids if i.startswith("CAT-")] if CLIP_OK else []
if len(thumb_imgs) >= 20 and CLIP_OK and len(cat_ids) >= 5:
    ids = list(thumb_imgs)
    temb = clip_embed_images([thumb_imgs[i] for i in ids])
    centroid = clip_img_emb[[clip_img_ids.index(i) for i in cat_ids]].mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    tf = pd.DataFrame({"id": ids, "brand_alignment": temb @ centroid})
    feats = pd.DataFrame([{"id": i, **{k: v for k, v in visual_features(thumb_imgs[i]).items() if k in FEATURE_COLS}} for i in ids])
    thumbs = thumbs.merge(tf, on="id").merge(feats, on="id")
    print("\nMedian brand alignment:", thumbs.groupby("who")["brand_alignment"].median().round(3).to_dict())
    rows = []
    for f in ["brand_alignment", "colorfulness", "brightness", "clutter_edge_density", "saturation"]:
        d = thumbs[thumbs["who"] == "Independent creators"][[f, "views_per_subscriber"]].dropna()
        rho, p = _st.spearmanr(d[f], np.log1p(d["views_per_subscriber"]))
        rows.append({"thumbnail_feature": f, "spearman_rho_vs_views_per_subscriber": round(rho, 3), "p_value": round(p, 3), "n": len(d)})
    print("\nDo thumbnail visuals move with creator-video reach?")
    print(pd.DataFrame(rows).to_string(index=False))
    fig, ax = plt.subplots(figsize=(7.5, 4))
    for grp, colr in [("Independent creators", "#3B6FA0"), ("Alo official channel", "#C2410C")]:
        d = thumbs[thumbs["who"] == grp]
        ax.scatter(d["brand_alignment"], np.log10(d["views_per_subscriber"].clip(lower=1e-3)), s=26, alpha=0.75,
                   color=colr, edgecolor="white", linewidth=0.6, label=grp, zorder=3)
    ax.set_xlabel("visual brand alignment (CLIP similarity to Alo catalog centroid)")
    ax.set_ylabel("log10 views per subscriber")
    ax.set_title("Is on-brand creator content rewarded with reach?", fontsize=12, fontweight="bold", loc="left")
    ax.legend(frameon=False, fontsize=8)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.grid(color="#E5E7EB", linewidth=0.8, zorder=0)
    plt.tight_layout()
    plt.savefig("thumbnail_brand_alignment.png", dpi=150)
    plt.show()
    lo = thumbs.nsmallest(3, "brand_alignment")["id"].tolist()
    print("Least on-brand creator thumbnails (open these to see what 'off-brand' looks like):",
          ", ".join(f"https://www.youtube.com/watch?v={i}" for i in lo))
else:
    print("Needs thumbnail downloads, CLIP (4c-5) and at least 5 catalog images loaded -- run in Colab.")


## 6. Multimodal Fusion -- The Real Return Friction Map

Combines the verified Evidence corpus (121 records) with our supplementary Trustpilot data into one ranked view of where the return journey breaks down.


In [ ]:
# --- Combine text + video sources for the fusion chart ---
combined_tags = list(evidence_core["unified_tag"])

if len(supp_text_df) > 0:
    tag_map = {
        "policy-transparency": "policy-transparency", "refund-delay": "refund-delay",
        "rebuy-loop": "rebuy-loop", "product-quality": "product-quality",
        "support-quality": "support-quality", "final-sale-trap": "final-sale-trap",
        "no-hemming": "product-quality", "general-fit-chat": "fit-sizing",
    }
    combined_tags += [tag_map.get(t, "other") for t in supp_text_df["friction_tag"]]

# Video evidence now folds into the same taxonomy -- this is the multimodal fusion step
# (text + video sharing one taxonomy, one count). Images are evidence/context rather than
# independently-tagged friction counts, so they support the map without adding to n.
if len(video_df) > 0:
    combined_tags += list(video_df["friction_tag"])

friction_counts = pd.Series(combined_tags).value_counts().reindex(UNIFIED_TAXONOMY.keys(), fill_value=0)
friction_counts = friction_counts.sort_values(ascending=True)
n_text = len(evidence_core) + len(supp_text_df)
n_video = len(video_df)
print(f"Total combined evidence rows feeding the friction map: {len(combined_tags)} (text: {n_text}, video: {n_video})")
friction_counts

In [ ]:
# --- Return Friction Map (real data) ---
fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.barh(friction_counts.index, friction_counts.values, color="#3B6FA0", height=0.6)
ax.set_xlabel(f"Number of mentions (n={friction_counts.sum()} combined verified + supplementary records)")
ax.set_title("Return Friction Map -- Alo Yoga", fontsize=12, fontweight="bold", loc="left")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.grid(axis="x", color="#E5E7EB", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for bar in bars:
    w = bar.get_width()
    if w > 0:
        ax.text(w + 0.5, bar.get_y() + bar.get_height()/2, f"{int(w)}", va="center", fontsize=9, color="#374151")
plt.tight_layout()
plt.savefig("return_friction_map_real.png", dpi=150)
plt.show()


In [ ]:
# --- Benchmark comparison table (from the granular Reference_Corpus) ---
benchmark_topics = ref_corpus_df[ref_corpus_df["Topic"].isin(["Return window", "Refund timing", "Exchange channel", "Exchange method"])]
benchmark = benchmark_topics[["Company", "Topic", "Policy_or_Claim"]].sort_values(["Topic", "Company"])
benchmark


### 6b. Cross-Modal Corroboration & Conflict -- Where Modalities Agree, Disagree, or Are Missing

Section 6's friction map already fuses text + video into one taxonomy. This goes one level deeper: it looks at *specific* friction cases where more than one modality speaks to the same underlying claim, and checks whether they agree, conflict, or whether a modality is simply absent. This is the assignment's separately-named **Multimodal Pipeline** (Task 1 lists Text / Image / Video / Multimodal as four distinct pipelines) -- not just another chart of the same three sources.

**Fusion design used here: late fusion.** Each modality is independently tagged first -- text via `classify_blob` (Section 3), video via the same function applied to collected notes (Section 5), images via **pixel analysis** (4c: Gemini vision caption, else CLIP zero-shot). The v2 method, `classify_blob` on each `Media_Assets` row's `Evidence_Note` + `Fit_Return_Relevance`, is kept as a comparison column -- then the independent tags are combined and reconciled below at the decision level. We are not attempting **early fusion** (jointly embedding pixels and text into one shared representation before classifying) -- that's a real, harder rebuild, and the same trade-off logic as TF-IDF vs. embeddings in Section 7b applies: late fusion is auditable with a small, hand-checkable dataset like ours; early fusion becomes worth the investment once there's enough data (and Phase 1 edit-log evidence, per the roadmap) to justify it.

In [ ]:
# --- Tag Media_Assets (images + videos) with the same unified taxonomy used for text ---
media_df["blob"] = media_df["Evidence_Note"].astype(str) + " " + media_df["Fit_Return_Relevance"].astype(str)
media_df["unified_tag"] = media_df["blob"].apply(classify_blob)

# De-dupe: some Media_Assets video rows point at YouTube videos already counted in video_df. They are matched
# on URL and excluded from the *new* video tally, so video evidence isn't double-counted.
already_counted_video_urls = set(video_df["link"])
media_video_new = media_df[(media_df["Modality"] == "Video") & (~media_df["Direct_Media_URL"].isin(already_counted_video_urls))]
media_video_overlap = media_df[(media_df["Modality"] == "Video") & (media_df["Direct_Media_URL"].isin(already_counted_video_urls))]
print(f"Media_Assets video rows: {len(media_df[media_df['Modality']=='Video'])} total -- "
      f"{len(media_video_overlap)} already counted in video_df (excluded here to avoid double-counting), "
      f"{len(media_video_new)} genuinely new.")

# Image modality: v3 counts what the PIXELS show (4c-8); the v2 text-note method is kept alongside for comparison.
image_counts_textnotes = media_df[media_df["Modality"] == "Image"]["unified_tag"].value_counts()
image_counts_pixels = image_tags["pixel_tag"].dropna().value_counts()
use_pixels = len(image_counts_pixels) > 0
video_asset_counts = pd.concat([video_df["friction_tag"], media_video_new["unified_tag"]]).value_counts()

coverage = pd.DataFrame({
    # v3 fix: combined_tags holds text rows first, then video rows -- v2 counted all of them as 'text'
    "text_evidence": pd.Series(combined_tags[:n_text]).value_counts(),
    "image_evidence_pixels": image_counts_pixels,
    "image_evidence_textnotes_v2": image_counts_textnotes,
    "video_evidence": video_asset_counts,
}).reindex(UNIFIED_TAXONOMY.keys()).fillna(0).astype(int)
img_col = "image_evidence_pixels" if use_pixels else "image_evidence_textnotes_v2"
coverage["modalities_present"] = (coverage[["text_evidence", img_col, "video_evidence"]] > 0).sum(axis=1)
coverage = coverage.sort_values(["modalities_present", "text_evidence"], ascending=False)
print("Image modality counted from:", "pixel analysis (4c)" if use_pixels else
      "text notes about images -- pixels not analysed in this run (run 4c in Colab)")
coverage


> **v3 note:** the bullets below were written from the v2 run. After re-running with 4c/5c, check them against the table above. Three things changed. The image column now comes from pixels. The video column includes rows re-tagged from transcripts and Gemini's viewing. The text column no longer includes video rows (v2 counted them twice). Gemini's `visual_contradicts_speech` flag in 5c-4 adds a *within-video* conflict case: a creator's words vs. what the camera shows.

**Reading the coverage table:**
- **`fit-sizing` and `product-quality` have full three-modality corroboration** (text + image + video) -- not just a text claim, but independent image and video evidence pointing at the same category. Batch 2 (Section 5b) is what closed this for `product-quality`: before it, video evidence there was zero; real customer videos like *"Alo Yoga is the LITERAL WORST"* and *"How To Remove Pilling From Your Leggings"* now provide that third modality.
- **`rebuy-loop` has partial corroboration** -- text evidence plus one real video (a customer-made "Alo Yoga Return and Refund Policy: What You Must Know!!!" explainer), but no image evidence.
- **`refund-delay`, `support-quality`, `policy-transparency`, `final-sale-trap` remain text-only** -- a genuine missing-modality gap, not an oversight we're hiding. These are inherently harder to photograph or film (a delayed refund or an unhelpful support reply doesn't produce a natural UGC image/video the way a fit or defect does), which is itself a useful finding: multimodal coverage will always be uneven across friction types, and text remains the necessary backbone for the categories that visual evidence structurally can't reach.

**Case study -- agreement (category-level, not a shared record):** the text evidence documents pilling/durability complaints as a real friction category (e.g. Record `E021`, `E041`, `E079` -- pilling within minutes to a few washes) without naming a specific garment. Independently, `Media_Assets` rows `M001`/`M002` are Nordstrom customer photos showing a surface snag/hole and fabric damage on the **Airlift 7/8 Legging**, and Batch 2 adds a real video (*"Alo Yoga is the LITERAL WORST,"* 422K views) making the same durability complaint independently. Three different pieces of evidence, three different sources, all corroborating the *same friction category* -- which is what fusion is supposed to add over any single modality alone.

**Case study -- conflict (same named product):** text evidence repeatedly and specifically names the **Airlift** line as *compressive/runs tight* (`E003`, `E052`, `E057`, `E059`, `E071` -- "recommend sizing down," "too tight," "designed for a close fit"). Independently, `Media_Assets` row `M005` is a Depop resale listing for the **Airlift Line Up Set** where the seller notes *"size small was slightly too large."* This is a genuine, same-product cross-modal disagreement -- not a data error to smooth over. It's the multimodal equivalent of the text-only `E054` vs. `T144` conflict already surfaced in the governance section (9e): fusion's job includes **surfacing** disagreement, not silently averaging it into a false consensus.


In [ ]:
# --- The two cases above, pulled directly from the source data (not hand-typed) ---
print("AGREEMENT CASE -- product-quality, category-level:")
print(evidence_df[evidence_df["Record_ID"].isin(["E021", "E041", "E079"])][["Record_ID", "Evidence_Summary"]].to_string(index=False))
print()
print(media_df[media_df["Asset_ID"].isin(["M001", "M002"])][["Asset_ID", "Product", "Title"]].to_string(index=False))

print()
print("CONFLICT CASE -- Airlift sizing, same named product:")
print(evidence_df[evidence_df["Record_ID"].isin(["E003", "E052", "E057", "E059", "E071"])][["Record_ID", "Evidence_Summary"]].to_string(index=False))
print()
print(media_df[media_df["Asset_ID"] == "M005"][["Asset_ID", "Product", "Title", "Fit_Return_Relevance"]].to_string(index=False))

### 6d. LLM Few-Shot Classification (Gemini) -- Run in Colab (Confirmed Live 2026-09-22)

Per the professor's guidance: for consumer/UGC text (emoji, sarcasm, ambiguous phrasing) a hand-flagged few-shot prompt to an LLM beats keyword rules like `classify_blob` above. **Confirmed this session that this sandbox cannot reach the Gemini API at all** -- same network allowlist restriction that blocks YouTube (only npm/pip/GitHub are reachable from here; a direct test to `generativelanguage.googleapis.com` failed with a proxy 403 before the API key is even checked). So this isn't an API-key problem, it's a sandbox-network problem -- **the cell below needs to run in Colab, VS Code, or any local/unrestricted environment**, not here.

**Set your key via Colab Secrets, not a plaintext line of code:** click the key icon on the left sidebar in Colab, add a secret named `GEMINI_API_KEY` (and optionally `GEMINI_MODEL_NAME`), toggle notebook access on. The `get_secret()` helper defined in the Setup cell reads it from there automatically -- this survives runtime restarts (no more retyping your key every session) and the key never appears in the notebook file itself, so it stays safe to save, share, or submit.

In [ ]:
# --- Run this in Colab/local Jupyter, NOT this sandbox (network-blocked here) ---
import os

FEW_SHOT_EXAMPLES = [
    ("Ordered a medium, came back a week later smelling like it had already been worn.", "product-quality"),
    ("Wanted to exchange for a different size but they only offer store credit for final sale items.", "final-sale-trap"),
    ("Support took 12 days to even respond to my return request.", "support-quality"),
    ("Runs a full size small compared to their other leggings -- wish that was on the size chart.", "fit-sizing"),
    ("Refund showed up as a gift card instead of back to my card, no one told me that upfront.", "refund-delay"),
]

def classify_with_gemini(text, api_key=None):
    """Few-shot classification via Gemini. Falls back to the keyword classifier if no
    key/network is available (e.g. if accidentally run inside this sandbox)."""
    api_key = api_key or get_secret("GEMINI_API_KEY")
    if not api_key:
        return classify_blob(text)  # graceful fallback, not a silent wrong answer
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    GEMINI_MODEL_NAME = get_secret("GEMINI_MODEL_NAME") or "gemini-3.6-flash"  # 2.5-flash was deprecated for new users mid-project; confirmed working end-to-end by the team on 2026-09-22  # override if this 404s -- model names drift; run genai.list_models() to see what your key actually has access to
    model = genai.GenerativeModel(GEMINI_MODEL_NAME)
    examples_block = "\n".join(f'Text: "{t}"\nLabel: {l}' for t, l in FEW_SHOT_EXAMPLES)
    prompt = (
        f"Classify the following customer text into exactly one label from: "
        f"{list(UNIFIED_TAXONOMY.keys())}.\n\n{examples_block}\n\n"
        f'Text: "{text}"\nLabel:'
    )
    response = model.generate_content(prompt)
    label = response.text.strip().lower()
    return label if label in UNIFIED_TAXONOMY else "other"

# Example (will silently fall back to classify_blob if run without a key/network):
print(classify_with_gemini("The compression is way stronger than their old leggings, sized up and it still felt tight."))

## 6e. Competitor Benchmark: Lululemon and Vuori (as Committed in the Proposal)

The approved proposal names **Lululemon and Vuori as benchmarks for sizing communication and return guidance**. This section delivers that benchmark with three kinds of evidence:

1. **Policy scorecard:** what each brand *officially tells* customers about the return journey, taken clause by clause from the Reference_Corpus. Every cell cites a doc_id.
2. **Policy vs. experience:** what customers *say happens* at Lululemon (comments on @lululemon / @lululemonme Instagram posts), compared with Lululemon's own stated refund timing.
3. **Video landscape:** Lululemon-focused YouTube content compared with Alo's Batch 2 (reach, and how often titles signal fit or quality problems).

**Kept separate from Alo's friction map on purpose.** Competitor evidence never enters Alo's counts. In the RAG (7d), it is only retrieved when a question names a competitor. Without that rule, a *Lululemon* complaint could be cited as evidence in an *Alo* answer, which a test on our own data showed does happen if the data is simply merged.

**Data asymmetry, stated up front:** Alo's evidence comes from reviews, Reddit and Trustpilot. Lululemon's comes from comments under the brand's own marketing posts. Vuori has policy documents only (no customer-experience data was collected). The comparisons below are therefore **directional**, not like-for-like rates.

In [ ]:
# --- 6e-1. Policy scorecard: sizing communication & return guidance, every cell cited ---
# Dimensions follow the proposal's journey (eligibility -> request -> exchange -> refund) plus sizing communication.
# Values are read from the Reference_Corpus; None = no such clause in our collected policy documents.
SCORECARD = {
    "Return window":                         {"Alo Yoga": "R001", "Lululemon": "R010", "Vuori": "R013"},
    "Refund timing stated to customer":      {"Alo Yoga": None,   "Lululemon": "R011", "Vuori": "R015"},
    "Size exchange without re-buying":       {"Alo Yoga": "R004", "Lululemon": "R012", "Vuori": "R016"},
    "Fit-specific return reasons captured":  {"Alo Yoga": "R005", "Lululemon": None,   "Vuori": "R014"},
    "Height-based length guidance":          {"Alo Yoga": "R008", "Lululemon": None,   "Vuori": None},
}
ref_text = ref_corpus_df.set_index("Reference_ID")["Policy_or_Claim"].to_dict()
score_rows = []
for dim, per_brand in SCORECARD.items():
    row = {"dimension": dim}
    for brand, rid in per_brand.items():
        row[brand] = f"{rid}: {ref_text.get(rid, '')[:70]}" if rid else "-- not stated --"
    score_rows.append(row)
scorecard_df = pd.DataFrame(score_rows).set_index("dimension")
pd.set_option("display.max_colwidth", 90)
print(scorecard_df.to_string())

gaps = [d for d, pb in SCORECARD.items() if pb["Alo Yoga"] is None and any(pb[b] for b in ["Lululemon", "Vuori"])]
print("\nWhere Alo is behind BOTH/EITHER benchmark on what it tells customers:", gaps or "none")
print("Reading: Alo is the only one of the three that never tells customers how long a refund takes -- and "
      "\n'refund-delay' is Alo's largest text friction category after fit and quality (Section 6). Vuori also captures"
      "\nfit-specific return reasons (too small / too large / not true to size); Alo's portal asks only for a generic reason (R005)."
      "\nAlo leads on two: height-based length guidance (R008) and a direct online size exchange (R004).")


In [ ]:
# --- 6e-2. Lululemon customer experience: complaints under the brand's own Instagram posts ---
import glob
IG_PATH = next(iter(sorted(glob.glob("dataset_instagram-scraper_*.csv"))), None)
lulu_complaints = pd.DataFrame(columns=["doc_id", "brand", "text", "user", "likes", "tag", "scope"])
if IG_PATH is None:
    print("Instagram scrape (dataset_instagram-scraper_*.csv) not found next to the notebook -- 6e-2/6e-3 skipped.")
else:
    ig_raw = pd.read_csv(IG_PATH, low_memory=False)
    accounts = sorted(ig_raw["ownerUsername"].dropna().unique())
    ccols = [c for c in ig_raw.columns if re.fullmatch(r"latestComments/\d+/text", c)]
    rows = []
    for _, r in ig_raw.iterrows():
        for c in ccols:
            t = r[c]
            if isinstance(t, str) and t.strip():
                rows.append({"text": t.strip(), "user": r[c.replace("/text", "/ownerUsername")],
                             "likes": r[c.replace("/text", "/likesCount")]})
    ig_comments = pd.DataFrame(rows)
    n_all = len(ig_comments)
    ig_comments = ig_comments[~ig_comments["user"].isin(accounts)]            # drop the brand's own replies
    ig_comments = ig_comments.drop_duplicates("text")                          # one user pasted the same complaint on many posts
    COMPLAINT_RE = (r"refund|return(?:ed|ing)? (?:my|it|them|the)|never (?:received|arrived)|customer service|no (?:reply|response)|"
                    r"phone lines?|pill|quality|ripped|see[- ]through|too (?:tight|small|big|loose)|exchange|money back|waited|worst|stolen")
    PRAISE_RE = r"never disappoint|did not disappoint|\blove\b|obsessed|best day"
    low = ig_comments["text"].str.lower()
    comp = ig_comments[low.str.contains(COMPLAINT_RE) & ~low.str.contains(PRAISE_RE)].copy()
    comp["tag"] = comp["text"].apply(classify_blob)
    # Lost/stolen parcels are real friction but happen before the return journey starts -> out of scope, reported separately
    comp["scope"] = np.where(comp["text"].str.lower().str.contains(r"stolen|never (?:received|arrived)|courier|ontrac|delivered"),
                             "delivery (out of return scope)", "return journey")
    comp.loc[(comp["scope"] == "return journey") & (comp["tag"] == "other") &
             comp["text"].str.lower().str.contains(r"phone|email|reply|response|escalat"), "tag"] = "support-quality"
    comp["brand"] = "Lululemon"
    comp["doc_id"] = [f"LULU-IG{i:02d}" for i in range(len(comp))]
    lulu_complaints = comp[["doc_id", "brand", "text", "user", "likes", "tag", "scope"]].reset_index(drop=True)
    print(f"Instagram: {len(ig_raw)} posts from {accounts}; {n_all} comments -> {len(ig_comments)} unique customer comments "
          f"-> {len(lulu_complaints)} complaints ({lulu_complaints['user'].nunique()} distinct users).")
    print(lulu_complaints["scope"].value_counts().to_string())
    print("\nReturn-journey complaints by friction tag:")
    print(lulu_complaints[lulu_complaints["scope"] == "return journey"]["tag"].value_counts().to_string())
    print("\nExamples:")
    for r in lulu_complaints[lulu_complaints["scope"] == "return journey"].head(8).itertuples():
        print(f"  {r.doc_id} [{r.tag}] {r.text[:130]}")


In [ ]:
# --- 6e-3. Policy vs experience, and Alo vs Lululemon friction mix ---
if len(lulu_complaints):
    rj = lulu_complaints[lulu_complaints["scope"] == "return journey"]
    # (a) Lululemon states refund timing (R011) -- do customers report it being met?
    timing = rj[rj["text"].str.lower().str.contains(r"\d+\s*(?:business\s*)?(?:days?|weeks?)|two weeks|waited|still waiting")]
    print(f"Lululemon's stated refund timing (R011): {ref_text.get('R011', '')}")
    print(f"Customer comments reporting waits: {len(timing)}")
    for r in timing.itertuples():
        m = re.findall(r"\d+\s*(?:to\s*\d+\s*)?(?:business\s*)?(?:days?|weeks?)|two weeks", r.text.lower())
        print(f"  {r.doc_id}: reported wait {', '.join(m) or '(unspecified)'} -- {r.text[:100]}")
    gift = rj[rj["text"].str.lower().str.contains("gift card")]
    if len(gift):
        print(f"\nRefund-as-gift-card complaints at Lululemon: {len(gift)} -- the same friction Alo customers report "
              f"(refund issued as store credit, query 6 in Section 8).")

    # (b) Friction mix: Alo text evidence vs Lululemon return-journey complaints (shares, directional only)
    alo_mix = pd.Series(combined_tags[:n_text]).value_counts(normalize=True)
    lulu_mix = rj["tag"].value_counts(normalize=True)
    mix = pd.DataFrame({"Alo Yoga (reviews/Reddit/Trustpilot)": alo_mix, "Lululemon (IG comments)": lulu_mix}) \
            .reindex([k for k in UNIFIED_TAXONOMY if k != "other"]).fillna(0)
    print(f"\nShare of friction mentions (Alo n={n_text}, Lululemon n={len(rj)}; 'other' excluded from view):")
    print((mix * 100).round(0).astype(int).astype(str).add("%").to_string())

    fig, ax = plt.subplots(figsize=(8.5, 4))
    y = np.arange(len(mix))
    ax.barh(y - 0.2, mix.iloc[:, 0] * 100, height=0.38, color="#3B6FA0", label=mix.columns[0], zorder=3)
    ax.barh(y + 0.2, mix.iloc[:, 1] * 100, height=0.38, color="#C2410C", label=mix.columns[1], zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(mix.index, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("share of that brand's friction mentions (%)")
    ax.set_title(f"Same frictions, different brand -- Alo (n={n_text}) vs Lululemon (n={len(rj)})",
                 fontsize=12, fontweight="bold", loc="left")
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.grid(axis="x", color="#E5E7EB", linewidth=0.8, zorder=0)
    plt.tight_layout()
    plt.savefig("benchmark_friction_mix_alo_vs_lululemon.png", dpi=150)
    plt.show()
    print("Caveat: Lululemon n is small and comes from a different channel (comments under brand posts skew to people"
          "\nescalating unresolved problems). Read the chart as 'which frictions recur across brands', not as rates.")


In [ ]:
# --- 6e-4. Video landscape: Lululemon-focused YouTube scrape vs Alo's Batch 2 ---
YT_PATH = next(iter(sorted(glob.glob("dataset_youtube-scraper_*.csv"))), None)
if YT_PATH is None:
    print("YouTube competitor scrape (dataset_youtube-scraper_*.csv) not found -- 6e-4 skipped.")
else:
    ytc = pd.read_csv(YT_PATH, low_memory=False).dropna(subset=["id"]).drop_duplicates("id")
    t = ytc["title"].astype(str).str.lower()
    ytc["focus"] = np.select([t.str.contains("lulu") & t.str.contains(r"\balo\b|aloyoga|alo yoga"),
                              t.str.contains("lulu"), t.str.contains("vuori")],
                             ["Alo vs Lululemon", "Lululemon", "Vuori"], "other")
    ytc = ytc[~t.str.contains(r"stock|earnings|shares|ceo|quit my job|investor")]   # business-news noise
    ytc["views"] = pd.to_numeric(ytc["viewCount"], errors="coerce")
    ytc["subscribers"] = pd.to_numeric(ytc["numberOfSubscribers"], errors="coerce")
    ytc["views_per_subscriber"] = ytc["views"] / ytc["subscribers"].replace(0, np.nan)
    ytc["title_tag"] = ytc["title"].astype(str).apply(classify_blob)          # title only -- NOT the search query
    overlap = set(ytc["id"]) & set(genuine["id"])
    print(f"Competitor YouTube scrape: {len(ytc)} videos after noise filter; overlap with Alo Batch 2: {len(overlap)}")
    print(ytc["focus"].value_counts().to_string())

    alo_side = vm.assign(title_tag=vm["title"].astype(str).apply(classify_blob))
    cmp_rows = []
    for name, d in [("Alo (Batch 2)", alo_side), ("Lululemon", ytc[ytc["focus"] == "Lululemon"]),
                    ("Alo vs Lululemon", ytc[ytc["focus"] == "Alo vs Lululemon"])]:
        if len(d):
            cmp_rows.append({"set": name, "videos": len(d), "median_views": d["views"].median(),
                             "median_views_per_subscriber": round(d["views_per_subscriber"].median(), 2),
                             "titles_signalling_fit_or_quality": f"{d['title_tag'].isin(['fit-sizing', 'product-quality']).mean():.0%}"})
    print("\n" + pd.DataFrame(cmp_rows).to_string(index=False))
    print("\nComparison videos (candidates for side-by-side sizing evidence), by views:")
    for r in ytc[ytc["focus"] == "Alo vs Lululemon"].sort_values("views", ascending=False).head(6).itertuples():
        print(f"  {int(r.views):>8,} views  {r.title[:90]}")
    print("\nReading: search queries differed between the two scrapes, so view counts are context, not a head-to-head"
          "\nmetric. The useful signal is the share of titles that surface fit/quality problems in each brand's video landscape.")

print("\nVuori: benchmarked on policy only (6e-1). No Vuori customer-experience data was collected -- a stated gap (9d).")


## 6f. Dataset Compilation & EDA: All Four Packages Against the Brief

The brief asks for notebooks that demonstrate **dataset compilation and EDA** across four packages (Text, Image, Video, Reference). This section is the single inventory: each component the brief lists, what we compiled for it, record counts, and an honest status. It is followed by the core EDA charts.

In [ ]:
# --- 6f-1. Inventory vs the brief's four packages ---
def n_of(cond):
    return int(cond.sum())
ui_n = len(glob.glob(os.path.join("ui_screenshots", "*"))) if os.path.isdir("ui_screenshots") else 0
lulu_n = len(lulu_complaints) if "lulu_complaints" in globals() else 0
titles_all = genuine["title"].astype(str).str.lower()
inventory = [
    # package, brief component, what we have, n, status
    ("Text", "Customer reviews", "Trustpilot, Zappos, Nordstrom, BBB, Bloomingdale's, Alo site reviews",
     n_of(~text_records["source"].str.contains("Reddit", case=False)), "covered"),
    ("Text", "Social media discussions (Reddit, X, Instagram comments)", "Reddit threads; Instagram comments (Lululemon benchmark); X not collected",
     n_of(text_records["source"].str.contains("Reddit", case=False)) + lulu_n, "partial -- no X"),
    ("Text", "Customer support ticket transcripts", "Not publicly available; support EXPERIENCES captured via reviews (support-quality tag)",
     n_of(text_records["unified_tag"] == "support-quality"), "gap -- proxied"),
    ("Text", "Post-purchase survey feedback", "Not publicly available", 0, "gap"),
    ("Image", "Official product catalog imagery", "Unique og:image hero shots (95 slot rows)", len(catalog_unique), "covered"),
    ("Image", "Fabric texture close-ups", "Customer damage/pilling close-ups (Media_Assets)",
     n_of(media_df["Evidence_Note"].astype(str).str.contains("pill|snag|damage|detail|fabric", case=False) & (media_df["Modality"] == "Image")), "partial"),
    ("Image", "UGC fit/wear photos", "Reddit fit checks + Nordstrom/Depop/Lemon8 customer photos",
     n_of(image_inventory["is_ugc"]), "covered (small n)"),
    ("Image", "Mobile app UI/UX screenshots", "Return-journey screenshots (4e)", ui_n, "covered" if ui_n else "PENDING -- team to capture"),
    ("Video", "Influencer try-on hauls", "YouTube creator try-ons/hauls", n_of(titles_all.str.contains("try|haul")), "covered"),
    ("Video", "Customer unboxing vlogs", "Unboxing/first-order videos", n_of(titles_all.str.contains("unbox|first order|opening")), "partial"),
    ("Video", "Official brand ad campaigns", "Alo official channel (brand-owned workout content)", n_of(video_batch2_raw["channelName"] == "ALO"), "partial -- not ads"),
    ("Video", "Durability / wash-test clips", "Pilling/wash/'after years of wear' videos", n_of(titles_all.str.contains("pill|wash|after .*wear|durab|worst")), "partial"),
    ("Video", "Short-form TikTok / Reels", "Creator videos <= 3 min; Lululemon Reels as benchmark",
     n_of(b2["duration_min"] <= 3) if "b2" in globals() else 0, "partial -- YouTube only for Alo"),
    ("Reference", "Customer return policy guidelines", "Alo return/refund/exchange/portal clauses", n_of(ref_df["doc_id"].isin(["R001", "R002", "R003", "R004", "R005", "R006"])), "covered"),
    ("Reference", "Fabric care / sizing specs", "Airlift fit & inseam claims + FA19 size chart", n_of(ref_df["doc_id"].isin(["R007", "R008", "R009", "SUPP-R002"])), "covered"),
    ("Reference", "Material sustainability compliance certificates", "Good On You third-party rating (no certificates published by Alo)", 1, "partial -- rating, not certificate"),
    ("Reference", "Brand visual aesthetic style guide", "Not public; catalog imagery used as the de-facto visual standard (5e)", 0, "gap -- proxied"),
    ("Reference", "Competitor benchmarks (proposal)", "Lululemon & Vuori policy clauses", n_of(ref_df["brand"].isin(["Lululemon", "Vuori"])), "covered"),
]
inventory_df = pd.DataFrame(inventory, columns=["package", "brief_component", "what_we_compiled", "records", "status"])
pd.set_option("display.max_colwidth", 80)
print(inventory_df.to_string(index=False))
print(f"\nCoverage: {inventory_df['status'].str.startswith('covered').sum()} covered, "
      f"{inventory_df['status'].str.startswith('partial').sum()} partial, "
      f"{inventory_df['status'].str.contains('gap|PENDING').sum()} gaps -- every gap is stated, and proxied where possible.")


In [ ]:
# --- 6f-2. Core EDA charts ---
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
ax = axes[0, 0]
src = text_records["source"].str.replace(r"\s*\(.*\)", "", regex=True).value_counts().head(10)[::-1]
ax.barh(src.index, src.values, color="#3B6FA0", height=0.6, zorder=3)
ax.set_title("Text records by source", fontsize=11, fontweight="bold", loc="left")
ax = axes[0, 1]
wl = text_records["text"].str.split().str.len()
ax.hist(np.log10(wl.clip(lower=1)), bins=30, color="#3B6FA0", edgecolor="white", zorder=3)
ax.set_xticks([0, 1, 2, 3]); ax.set_xticklabels(["1", "10", "100", "1000"])
ax.set_title(f"Text length in words (median {int(wl.median())})", fontsize=11, fontweight="bold", loc="left")
ax = axes[1, 0]
sent_mix = pd.crosstab(text_records["unified_tag"], text_records["sentiment"], normalize="index").reindex(columns=["Negative", "Neutral", "Positive"]).fillna(0)
left = np.zeros(len(sent_mix))
for lab, colr in [("Negative", "#C2410C"), ("Neutral", "#D1D5DB"), ("Positive", "#3B6FA0")]:
    ax.barh(sent_mix.index, sent_mix[lab], left=left, color=colr, label=lab, height=0.6, edgecolor="white", linewidth=1.5, zorder=3)
    left += sent_mix[lab].values
ax.set_xlim(0, 1)
ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower right", bbox_to_anchor=(1, 1.0))
ax.set_title("Sentiment mix per friction category", fontsize=11, fontweight="bold", loc="left", pad=18)
ax = axes[1, 1]
dm = b2[b2["who"] == "Independent creators"]["duration_min"].dropna()
ax.hist(dm.clip(upper=40), bins=20, color="#3B6FA0", edgecolor="white", zorder=3)
ax.set_title(f"Creator video length, minutes (n={len(dm)})", fontsize=11, fontweight="bold", loc="left")
for a in axes.flat:
    for s in ["top", "right"]:
        a.spines[s].set_visible(False)
    a.grid(axis="x", color="#E5E7EB", linewidth=0.8, zorder=0)
plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150)
plt.show()
dates = pd.to_datetime(evidence_core["Source_Date"], errors="coerce").dropna() if "Source_Date" in evidence_core else pd.Series(dtype="datetime64[ns]")
if len(dates):
    print(f"Evidence dates span {dates.min():%b %Y} to {dates.max():%b %Y}; "
          f"{(dates >= '2024-01-01').mean():.0%} from 2024 onwards (recency matters: policies and products change).")


## 7. Multimodal RAG Prototype

Knowledge base: the 17-row granular Reference_Corpus + supplementary docs (policy/fit-claim authority layer) + Evidence rows flagged `Include_in_RAG = Yes` (customer-evidence layer). TF-IDF retrieval proves the logic with zero API cost; swap in embeddings + an LLM generation call (Gemini for iteration, Anthropic for the final build) for production quality.


In [ ]:
# --- Build the retrieval corpus ---
corpus_records = []
for _, row in ref_df.iterrows():
    corpus_records.append({"doc_id": row["doc_id"], "kind": "policy", "brand": row["brand"], "text": row["text"]})

rag_evidence = evidence_core[evidence_core["Include_in_RAG"] == "Yes"]
for _, row in rag_evidence.iterrows():
    corpus_records.append({
        "doc_id": row["Record_ID"], "kind": "evidence", "brand": row.get("Company", "Alo Yoga"),
        "text": f"{row['Evidence_Summary']} (Source: {row['Source_URL']})",
    })

if len(supp_text_df) > 0:
    for _, row in supp_text_df.iterrows():
        corpus_records.append({"doc_id": row["id"], "kind": "evidence", "brand": "Alo Yoga", "text": row["text"]})

corpus_df = pd.DataFrame(corpus_records)
print(f"Total RAG corpus size: {len(corpus_df)} documents ({(corpus_df.kind=='policy').sum()} policy + {(corpus_df.kind=='evidence').sum()} evidence)")

# NOTE: on top of the standard English stop-words, we also drop a handful of
# domain-generic words ("customer", "product", "item", "bought", etc.) that show up
# in almost every document regardless of topic. Without this, a query like "customer
# wants a refund on the product" over-weighted "customer service" complaints instead
# of actual refund-timing evidence -- caught by testing a real query against this
# corpus, not a hypothetical. Removing these noise words fixed that miss without
# hurting any of the 12 eval queries in Section 8 (checked before applying this).
EXTRA_STOPWORDS = ["alo", "yoga", "customer", "product", "item", "bought", "ordered", "wants", "want", "received", "order"]
CUSTOM_STOPWORDS = list(TfidfVectorizer(stop_words="english").get_stop_words()) + EXTRA_STOPWORDS

# --- Stemming fix (added after live testing found two real retrieval bugs) ---
# Bug 1: plain TF-IDF treats "return" and "returned" as two unrelated words, so a
# query using one form scores 0.0 similarity against a document using the other --
# e.g. R001, the actual "returned within 30 days" policy, was completely unretrievable
# (rank 280/286) for the query "what's the return window for a full-price item?".
# Bug 2: a handful of corpus rows are a single short word (e.g. one review whose entire
# text is just "Return"). Plain TF-IDF concentrates all of that word's weight into one
# tiny document, so it wins top-1 for almost any query containing "return" -- even
# though it carries no real information. Both are fixed the same way: stem every word
# to its root (return/returns/returned/returning -> "return") before building the
# TF-IDF matrix, using NLTK's Porter stemmer. This also means EXTRA_STOPWORDS has to be
# stemmed the same way, since sklearn ignores its built-in stop_words parameter once a
# custom analyzer is supplied (silent behavior we caught via a UserWarning while testing).
from nltk.stem import PorterStemmer
import re as _re

_ps = PorterStemmer()
_STEMMED_STOPWORDS = set(_ps.stem(w) for w in CUSTOM_STOPWORDS)

def stem_analyzer(text):
    tokens = _re.findall(r"[a-zA-Z']+", text.lower())
    stems = [_ps.stem(t) for t in tokens]
    return [s for s in stems if s not in _STEMMED_STOPWORDS and len(s) > 1]

vectorizer = TfidfVectorizer(analyzer=stem_analyzer)
tfidf_matrix = vectorizer.fit_transform(corpus_df["text"])

def retrieve(query, top_k=3):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    results = corpus_df.iloc[top_idx].copy()
    results["similarity"] = sims[top_idx]
    return results[["doc_id", "kind", "brand", "similarity", "text"]]

query = "Customer wants to exchange a final sale item for a different size"
retrieve(query)


**Verified after re-testing all 13 eval queries with the stemming fix (2026-09-22):**
- The fix removes two confirmed retrieval bugs without introducing any new ones: R001 (the real 30-day return policy) went from 0.000 similarity / rank 280 of 286 to a genuinely non-zero, meaningfully-ranked score, and three near-empty rows (e.g. a review whose entire text was the single word "Return") stopped falsely winning top-1 for unrelated queries just because they're short.
- **Remaining, honest limitation (not fixed by stemming):** for query "What's the return window for a full-price item?", the top match is still a customer review that happens to say "return window" verbatim (T292), not R001 itself -- because R001's wording is "returned within 30 days" and never uses the word "window". Stemming fixes different *forms* of the same word, not different *words* for the same idea. That's a vocabulary/synonym gap, which is a core, textbook limitation of keyword-based TF-IDF retrieval versus semantic (embedding-based) retrieval -- worth naming directly in the limitations section rather than papering over.
- **A second honest limitation surfaced by the same test:** for the "emailed support three times, no response" query, the top match flipped to a review praising *fast* support response times -- the opposite sentiment of what was asked. TF-IDF matches on shared vocabulary ("email", "response", "customer"), not on whether the sentiment is a complaint or a compliment. This is a real, demonstrable weakness of bag-of-words retrieval that we're disclosing rather than hiding.
- Query "How long does a refund take after Alo receives the return?" still correctly surfaces no refund-timing answer -- confirmed against the Reference_Corpus sheet that Alo Yoga (unlike Lululemon and Vuori) simply has no refund-timing policy document in our data. That's a genuine data gap, not a retrieval failure.

### 7b. Why v2 Used TF-IDF -- and Why v3 Tests Embeddings (7c)

> **v3 update:** reason 1 below assumed embeddings need a hosted API with its own quota. A local open model (`all-MiniLM-L6-v2`) needs neither, so the switch is no longer untestable. Section 7c measures it on gold-labelled queries rather than arguing either way. Reason 2 (explainability) still holds. That is why v3 uses a **hybrid** (TF-IDF + embeddings): the keyword half stays inspectable.

This prototype retrieves evidence using TF-IDF + cosine similarity (keyword overlap, weighted by how rare a word is), not semantic embeddings. That's a deliberate choice for this stage, made with the trade-offs below in view -- not something we didn't consider.

**What TF-IDF actually gets wrong, demonstrated with our own live-tested bugs above, not a textbook hypothetical:**
- *Vocabulary gap*: R001 ("returned within 30 days") is still unretrievable for "return **window**" -- the words don't overlap even after stemming, because "window" and "days" are different words for the same idea. A semantic embedding model would very likely fix this, since it represents meaning rather than exact tokens.
- *Sentiment blindness*: for the "emailed support three times, no response" query, TF-IDF surfaced a review praising *fast* support, because both share words like "email" and "response." Embeddings are meaningfully better at this (they encode more context around a word), but are not a sentiment classifier either -- this failure mode could resurface, just less often.

**Why we're not switching to embeddings for this submission:**
1. **New failure surface, no time to test it properly.** An embeddings pipeline needs its own API (Gemini/OpenAI embeddings, or a local model) -- another dependency with its own quota limits, exactly like the Gemini generation quota issue we just spent real time debugging. Swapping retrieval methods two days before a deadline, untested, is a worse risk than keeping a method whose failure modes we've already found and disclosed.
2. **Explainability.** TF-IDF similarity scores are fully transparent -- we can point at exactly which words drove a match. Embedding similarity is a distance between two long vectors with no human-readable explanation for *why* two pieces of text scored close together. For a system that has to justify its answers to a support agent (and to this class's reviewers), that transparency has real value.
3. **What we have instead is stronger than a silent swap:** live-tested, documented evidence of exactly where the simpler method fails, plus a fix (stemming) that resolved two of the three failure modes we found. That's a more defensible engineering story than replacing the method and hoping the new one has no failure modes of its own -- it almost certainly would, just different ones we wouldn't have had time to find.

**Recommended upgrade path (for the Implementation Roadmap, Section 9b):** once the system is in Phase 1 (human-in-the-loop) and generating real edit/rejection-rate data, that data is exactly what would justify -- and fund -- a move to a sentence-embedding retriever, since by then there's a measurable baseline to prove the switch is actually an improvement, not just a fancier default.

In [ ]:
# --- Generation, wired for Gemini (falls back to retrieval-only if no key/network) ---
import os

GEN_SYSTEM_PROMPT = (
    "You are a support-desk assistant for Alo Yoga. Answer the customer's question using ONLY "
    "the evidence provided below -- do not invent policy details. Cite each claim with its doc_id "
    "in parentheses. If the evidence doesn't cover the question, say so plainly instead of guessing."
)

def generate_rag_answer(query, api_key=None, top_k=3):
    """Retrieves evidence, then generates a grounded, cited response.
    Tries Gemini if a key is available; otherwise returns the top retrieved match directly
    (an honest degraded mode, not a silent failure)."""
    evidence = retrieve(query, top_k=top_k)
    api_key = api_key or get_secret("GEMINI_API_KEY")

    if api_key is None:
        top = evidence.iloc[0]
        return (
            "[No API key available -- returning top retrieved match instead of a generated answer]\n"
            f"Source: {top['doc_id']} ({top['kind']})\n{top['text']}"
        )

    # NOTE: confirmed this session that this sandbox cannot reach the Gemini API network-wise
    # (proxy 403 on generativelanguage.googleapis.com, independent of key validity) -- run this
    # branch in Colab/local, not here.
    try:
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        GEMINI_MODEL_NAME = get_secret("GEMINI_MODEL_NAME") or "gemini-3.6-flash"  # 2.5-flash was deprecated for new users mid-project; confirmed working end-to-end by the team on 2026-09-22  # override if this 404s -- model names drift; run genai.list_models() to see what your key actually has access to
        model = genai.GenerativeModel(GEMINI_MODEL_NAME)
        evidence_block = "\n\n".join(f"[{row.doc_id}] {row.text}" for row in evidence.itertuples())
        prompt = f"{GEN_SYSTEM_PROMPT}\n\nEvidence:\n{evidence_block}\n\nCustomer question: {query}\n\nAnswer:"
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        top = evidence.iloc[0]
        return (
            f"[Gemini call failed ({e}) -- falling back to top retrieved match]\n"
            f"Source: {top['doc_id']} ({top['kind']})\n{top['text']}"
        )

print(generate_rag_answer(query))

**Confirmed live on 2026-09-22** (Colab, `gemini-3.6-flash`, real Gemini key) -- `generate_rag_answer()` returned a real, grounded, cited answer for the test query:

> "Final-sale goods are excluded from receiving an exchange, store credit, or refund (R003). Therefore, a final sale item cannot be exchanged for a different size."

This confirms the full RAG loop (retrieve → cite → generate) works end-to-end, not just the retrieval half. Model name needed two updates mid-project as Google deprecated `gemini-1.5-flash` then `gemini-2.5-flash` in quick succession -- `GEMINI_MODEL_NAME` stays overridable via environment variable for exactly this reason.

## 7c. Text Embeddings: Semantic Search and Customer Segmentation (Session 5)

Section 7b explained why v2 stayed on TF-IDF. The main reason given was that embeddings need their own API with quota limits. That applies to hosted embedding APIs, but **not to local open models**. `all-MiniLM-L6-v2` (sentence-transformers) is about 90 MB, runs on Colab's CPU in seconds, and needs no key and no quota. So this section tests the switch instead of arguing about it:

1. **Semantic search, measured:** TF-IDF vs embeddings vs a hybrid of both, scored on the 8 eval queries whose correct answer is a specific policy clause. We use standard retrieval metrics: Hit@3, Recall@3 and MRR. The eval also tests **policy routing**, which reserves the top 2 slots for the best-matching policy documents. It targets a failure the gold labels exposed: the 19 policy documents get outranked by 276 customer reviews that repeat the question's words.
2. **The two documented failures, re-tested:** the "return window" vocabulary gap (R001) and the "emailed support three times" sentiment-blindness case.
3. **Customer segmentation:** k-means over review embeddings (what customers talk about), plus a self-reported shopper-profile segmentation (petite / tall / curvy-midsize: who is talking). This follows the segmentation framing of Berger et al. (2020) and Li et al. (2025).

In [ ]:
# --- 7c-1. Embed the RAG corpus locally (no API, no quota) ---
from sklearn.preprocessing import normalize

def clean_for_embedding(t):
    return re.sub(r"\(Source:[^)]*\)", "", str(t)).strip()   # URLs carry no meaning, only noise

corpus_df["text_clean"] = corpus_df["text"].apply(clean_for_embedding)
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
try:
    from sentence_transformers import SentenceTransformer
    _emb_model = SentenceTransformer(EMB_MODEL_NAME)
    def embed(texts):
        return _emb_model.encode(list(texts), normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    EMB_KIND = "all-MiniLM-L6-v2 (sentence-transformers, local)"
except Exception as e:
    # Keeps the notebook runnable offline. These are NOT embedding results -- LSA is a linear projection of TF-IDF.
    print(f"!! sentence-transformers model unavailable ({type(e).__name__}). Falling back to LSA (TF-IDF + SVD) so the "
          f"notebook still runs. Re-run in Colab for the real embedding results before quoting any number below.")
    from sklearn.decomposition import TruncatedSVD
    _lsa_vec = TfidfVectorizer(analyzer=stem_analyzer).fit(corpus_df["text_clean"])
    _lsa_X = _lsa_vec.transform(corpus_df["text_clean"])
    _svd = TruncatedSVD(n_components=min(100, _lsa_X.shape[1] - 1), random_state=0).fit(_lsa_X)
    def embed(texts):
        return normalize(_svd.transform(_lsa_vec.transform(list(texts))))
    EMB_KIND = "LSA FALLBACK (not a neural embedding)"

corpus_emb = embed(corpus_df["text_clean"])
print(f"Embedded {corpus_emb.shape[0]} documents -> {corpus_emb.shape[1]}-dim vectors using {EMB_KIND}")

def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

def scores_tfidf(q):
    return cosine_similarity(vectorizer.transform([q]), tfidf_matrix).flatten()

def scores_semantic(q):
    return corpus_emb @ embed([q])[0]

def scores_hybrid(q, alpha=0.5):
    # min-max per query so the two score scales are comparable before mixing
    return alpha * minmax(scores_semantic(q)) + (1 - alpha) * minmax(scores_tfidf(q))

RETRIEVERS = {"tfidf": scores_tfidf, "semantic": scores_semantic, "hybrid": scores_hybrid}

# Policy routing: the 19 policy documents compete with 276 customer-evidence rows for the same top-k slots and
# mostly lose (reviews repeat the question's words far more often than a policy clause does). Routing reserves
# the top N slots for the best-matching POLICY documents, then fills the rest with evidence -- so every answer
# can rest on an actual clause, with customer evidence as context.
IS_POLICY = (corpus_df["kind"] == "policy").values
IS_ALO = (corpus_df["brand"] == "Alo Yoga").values
COMPETITOR_RE = r"lululemon|\blulu\b|vuori|competitor|compare|benchmark"

def with_policy_routing(score_fn, n_policy=2, is_policy=None, is_alo=None):
    """Reserve the top n_policy slots for policy docs. Brand-aware: a question that doesn't name a competitor only gets
    ALO policy clauses in those slots (an Alo answer must never rest on Lululemon's or Vuori's policy)."""
    def routed(q):
        pol = IS_POLICY if is_policy is None else is_policy
        alo = IS_ALO if is_alo is None else is_alo
        if not re.search(COMPETITOR_RE, q.lower()):
            pol = pol & alo
        s = score_fn(q)
        out = s - s.max() - 1.0                      # everything below the reserved slots, order preserved
        pol_idx = np.where(pol)[0][np.argsort(-s[pol])][:n_policy]
        for r, i in enumerate(pol_idx):
            out[i] = 10.0 - r
        return out
    return routed
for m in ["tfidf", "hybrid"]:
    RETRIEVERS[m + "+policy-routed"] = with_policy_routing(RETRIEVERS[m])

def retrieve_with(query, method="hybrid", top_k=3):
    s = RETRIEVERS[method](query)
    idx = s.argsort()[::-1][:top_k]
    out = corpus_df.iloc[idx][["doc_id", "kind", "brand", "text"]].copy()
    out.insert(3, "score", s[idx].round(3))
    return out


In [ ]:
# --- 7c-2. Retrieval evaluation with gold labels (policy-answer queries only) ---
# Gold = the policy clause(s) a correct answer must rest on. Written from the Reference_Corpus, not from any
# retriever's output. Queries whose correct answer is customer evidence (sizing opinions, support experiences)
# have no single gold doc and are judged by reading instead (8d).
def ref_ids_for_topic(fragment):
    return set(ref_corpus_df.loc[ref_corpus_df["Topic"].str.contains(fragment, case=False, na=False), "Reference_ID"])

GOLD = {
    "Customer wants to exchange a final sale item for a different size": {"R003", "R004"},
    "Customer says the leggings pilled after a few washes, can they return it?": {"R001", "R003"},
    "What's the return window for a full-price item?": {"R001"},
    "Customer's refund came back as store credit instead of the original payment method": {"R003"},
    "Can I exchange directly for a different size or do I have to return and re-buy?": {"R004"},
    "How does Alo's return window compare to Lululemon or Vuori?": {"R001"} | ref_ids_for_topic("Return window"),
    "Customer received a damaged item and wants to know the return process": {"R005", "R006"},
    "Do I need the original tags/packaging to return an item bought online?": {"R001"},
}
# (Query strings are identical to the matching EVAL_QUERIES in Section 8, so results line up with 8d.)

def rank_metrics(scores, gold, k=3):
    order = corpus_df["doc_id"].values[scores.argsort()[::-1]]
    ranks = [int(np.where(order == g)[0][0]) + 1 for g in gold if g in set(order)]
    first = min(ranks) if ranks else None
    return {"hit@3": int(first is not None and first <= k),
            "recall@3": sum(r <= k for r in ranks) / len(gold),
            "rr": 1 / first if first else 0.0, "first_gold_rank": first}

eval_rows = []
for q, gold in GOLD.items():
    for m in RETRIEVERS:
        eval_rows.append({"query": q, "method": m, **rank_metrics(RETRIEVERS[m](q), gold)})
retrieval_eval = pd.DataFrame(eval_rows)

retrieval_summary = (retrieval_eval.groupby("method")[["hit@3", "recall@3", "rr"]].mean()
           .rename(columns={"rr": "MRR"}).loc[list(RETRIEVERS)].round(3))
print(f"Retrieval quality on {len(GOLD)} gold-labelled policy queries ({EMB_KIND}):")
print(retrieval_summary.to_string())
print("\nRank of the first correct policy doc, per query (lower is better; 1 = top result):")
print(retrieval_eval.pivot(index="query", columns="method", values="first_gold_rank")[list(RETRIEVERS)].to_string())
print("\nNote: the gold labels are policy clauses, so routing is designed to help this metric. Its cost is that 2 of"
      "\nthe top-k slots always go to policy text even when a question is about customer experience -- acceptable for"
      "\na support assistant whose answers must rest on a clause, and why v3 retrieves 6 documents (3 policy + 3 evidence).")

# Is 50/50 the right mix? Sweep alpha -- with only 8 queries, treat the result as directional, not tuned.
sweep = []   # tuned on the un-routed hybrid
for a in [0, 0.25, 0.5, 0.75, 1.0]:
    rr = np.mean([rank_metrics(scores_hybrid(q, a), g)["rr"] for q, g in GOLD.items()])
    sweep.append({"alpha (0 = pure TF-IDF, 1 = pure semantic)": a, "MRR": round(rr, 3)})
print("\n" + pd.DataFrame(sweep).to_string(index=False))
BEST_ALPHA = max(sweep, key=lambda r: r["MRR"])["alpha (0 = pure TF-IDF, 1 = pure semantic)"]


In [ ]:
# --- 7c-3. Chart: retrieval quality by method ---
fig, ax = plt.subplots(figsize=(9, 3.8))
metrics = ["hit@3", "recall@3", "MRR"]
methods = list(RETRIEVERS)
colors = {"tfidf": "#9CA3AF", "semantic": "#93B4D6", "hybrid": "#3B6FA0",
          "tfidf+policy-routed": "#4B5563", "hybrid+policy-routed": "#1E3A5F"}
w = 0.8 / len(methods)
for i, m in enumerate(methods):
    vals = retrieval_summary.loc[m, metrics].values
    bars = ax.bar(np.arange(len(metrics)) + (i - (len(methods) - 1) / 2) * w, vals, width=w - 0.02,
                  color=colors.get(m, "#6B7280"), label=m, zorder=3)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=7, color="#374151")
ax.set_xticks(range(len(metrics)))
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.12)
ax.set_title(f"Retrieval quality on {len(GOLD)} policy queries", fontsize=12, fontweight="bold", loc="left")
ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper left")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.grid(axis="y", color="#E5E7EB", linewidth=0.8, zorder=0)
plt.tight_layout()
plt.savefig("retrieval_tfidf_vs_embeddings.png", dpi=150)
plt.show()


In [ ]:
# --- 7c-4. Re-testing the two failure modes documented in Section 7 ---
q_window = "What's the return window for a full-price item?"
print("1) VOCABULARY GAP -- rank of R001 ('returned within 30 days') for:", q_window)
for m in RETRIEVERS:
    s_ = RETRIEVERS[m](q_window)
    order = corpus_df["doc_id"].values[s_.argsort()[::-1]]
    pol_order = corpus_df["doc_id"].values[IS_POLICY][np.argsort(-s_[IS_POLICY])]
    print(f"   {m:<21} R001 rank = {int(np.where(order == 'R001')[0][0]) + 1:>3} of {len(order)}  "
          f"(rank {list(pol_order).index('R001') + 1} of {IS_POLICY.sum()} policy docs)")

q_support = "Customer had to email support three times before getting a response about their return"
print("\n2) SENTIMENT BLINDNESS -- top 3 for:", q_support)
for m in ["tfidf", "semantic", "hybrid"]:
    print(f"\n   [{m}]")
    for r in retrieve_with(q_support, m, 3).itertuples():
        print(f"     {r.doc_id:<9} {r.score:.3f}  {clean_for_embedding(r.text)[:120]}")
print("\nRead the three lists: a complaint query should surface complaints. Embeddings encode more context than"
      "\nshared words, so they usually help here, but they are not a sentiment classifier -- check by eye.")


In [ ]:
# --- 7c-5. Customer segmentation, part 1: WHAT customers talk about (k-means on review embeddings) ---
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

seg_tag_map = {"policy-transparency": "policy-transparency", "refund-delay": "refund-delay", "rebuy-loop": "rebuy-loop",
               "product-quality": "product-quality", "support-quality": "support-quality", "final-sale-trap": "final-sale-trap",
               "no-hemming": "product-quality", "general-fit-chat": "fit-sizing"}
tag_lookup = dict(zip(evidence_core["Record_ID"], evidence_core["unified_tag"]))
if len(supp_text_df):
    tag_lookup.update(dict(zip(supp_text_df["id"], [seg_tag_map.get(t, "other") for t in supp_text_df["friction_tag"]])))

ev_mask = (corpus_df["kind"] == "evidence") & (corpus_df["text_clean"].str.split().str.len() >= 4)  # drop 1-3 word rows
seg_df = corpus_df[ev_mask][["doc_id", "text_clean"]].copy()
seg_df["unified_tag"] = seg_df["doc_id"].map(tag_lookup).fillna("other")
X = corpus_emb[ev_mask.values]

sil = {}
for k in range(4, 9):   # capped at 8 so every segment can still be read and named by a person
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    sil[k] = silhouette_score(X, labels, metric="cosine")
BEST_K = max(sil, key=sil.get)
seg_df["cluster"] = KMeans(n_clusters=BEST_K, n_init=10, random_state=0).fit_predict(X)
print("Silhouette by k:", {k: round(v, 3) for k, v in sil.items()}, "-> k =", BEST_K)

# Name each cluster by its most distinctive words (class-based TF-IDF: one 'document' per cluster)
cluster_docs = seg_df.groupby("cluster")["text_clean"].apply(" ".join)
FILLER = ["just", "don", "didn", "doesn", "isn", "wasn", "ve", "ll", "im", "ive", "think", "actually", "really", "like",
          "got", "get", "did", "does", "said", "reported", "buyer", "reviewer", "commenter", "customer", "usually"]
ctfidf = TfidfVectorizer(stop_words=CUSTOM_STOPWORDS + FILLER, ngram_range=(1, 2), min_df=1, sublinear_tf=True,
                         token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
M = ctfidf.fit_transform(cluster_docs.values)
terms = np.array(ctfidf.get_feature_names_out())
cluster_names = {c: ", ".join(terms[M[i].toarray().ravel().argsort()[::-1][:5]]) for i, c in enumerate(cluster_docs.index)}

prof = (seg_df.groupby("cluster")
        .agg(reviews=("doc_id", "size"),
             share_tagged_other=("unified_tag", lambda s: round((s == "other").mean(), 2)),
             main_tag=("unified_tag", lambda s: s.value_counts().index[0]))
        .assign(top_terms=lambda d: d.index.map(cluster_names)).sort_values("reviews", ascending=False))
print(f"\n{BEST_K} semantic segments of {len(seg_df)} evidence rows:")
print(prof.to_string())

other_rows = seg_df[seg_df["unified_tag"] == "other"]
print(f"\nWhere the {len(other_rows)} keyword-'other' evidence rows land -- themes the keyword rules missed:")
print(other_rows["cluster"].map(lambda c: f"{c}: {cluster_names[c]}").value_counts().to_string())

centroids = np.vstack([X[(seg_df["cluster"] == c).values].mean(axis=0) for c in range(BEST_K)])
seg_df["centroid_sim"] = [X[i] @ centroids[c] / (np.linalg.norm(centroids[c]) + 1e-9) for i, c in enumerate(seg_df["cluster"])]
for c in prof.index[:BEST_K]:
    ex = seg_df[seg_df["cluster"] == c].nlargest(2, "centroid_sim")["text_clean"].str[:110].tolist()
    print(f"\n  cluster {c} most typical: " + " | ".join(ex))


In [ ]:
# --- 7c-6. Chart: the semantic segments in 2-D (PCA of the embeddings) ---
xy = PCA(n_components=2, random_state=0).fit_transform(X)
palette = ["#3B6FA0", "#C2410C", "#15803D", "#7C3AED", "#B45309", "#0E7490", "#BE185D", "#4B5563", "#65A30D", "#1D4ED8"]
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for c in sorted(seg_df["cluster"].unique()):
    m = (seg_df["cluster"] == c).values
    ax.scatter(xy[m, 0], xy[m, 1], s=22, color=palette[c % len(palette)], alpha=0.8, edgecolor="white", linewidth=0.6,
               label=f"{c}: {cluster_names[c][:38]}")
ax.set_title("Customer-evidence segments (review embeddings, k-means, PCA view)", fontsize=12, fontweight="bold", loc="left")
ax.set_xticks([]); ax.set_yticks([])
for s in ["top", "right", "left", "bottom"]:
    ax.spines[s].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig("evidence_segments_pca.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# --- 7c-7. Customer segmentation, part 2: WHO is talking (self-reported shopper profile) ---
# Rule-based extraction of what reviewers state about themselves. Doubles as the bias/representation
# measurement promised in the governance checklist (9e): which bodies does our evidence actually describe?
HEIGHT_RE = re.compile(r"\b([4-6])\s*(?:'|’|ft|foot|feet)\s*(\d{1,2}(?:\.\d)?)?")

def height_inches(text):
    m = HEIGHT_RE.search(text)
    if not m:
        return None
    ft, inch = int(m.group(1)), float(m.group(2)) if m.group(2) else 0.0
    return ft * 12 + inch if inch < 12 else None

def shopper_segment(text):
    t = str(text).lower()
    h = height_inches(t)
    if re.search(r"\bpetite\b|\bshort (legs|torso|person)|\bxxs\b|\b2xs\b", t) or (h is not None and h <= 63):
        return "petite / short"
    if re.search(r"\btall\b|long (legs|torso)", t) or (h is not None and h >= 68):
        return "tall"
    if re.search(r"curvy|midsize|mid-size|plus[- ]size|\bxl\b|\bxxl\b|\b1x\b|size 1[2-9]\b|thick thighs|bigger (hips|thighs)", t):
        return "curvy / midsize / plus"
    if h is not None or re.search(r"\b\d{2,3}\s*(?:lb|lbs|pounds|kg)\b", t):
        return "stats given, average height"
    return "no body info"

ev_all = corpus_df[corpus_df["kind"] == "evidence"][["doc_id", "text_clean"]].copy()
ev_all["unified_tag"] = ev_all["doc_id"].map(tag_lookup).fillna("other")
ev_all["segment"] = ev_all["text_clean"].apply(shopper_segment)

seg_counts = ev_all["segment"].value_counts()
print("Representation in our evidence (self-reported):")
print(seg_counts.to_string())
print(f"\n{seg_counts.get('no body info', 0) / len(ev_all):.0%} of evidence rows say nothing about the reviewer's body.")

xt = pd.crosstab(ev_all["segment"], ev_all["unified_tag"], normalize="index").round(2)
print("\nFriction mix within each shopper segment (row shares):")
print(xt.to_string())
fit_share = xt["fit-sizing"] if "fit-sizing" in xt else pd.Series(dtype=float)
if len(fit_share):
    print(f"\nFit-sizing share of complaints: " + ", ".join(f"{s} {v:.0%}" for s, v in fit_share.sort_values(ascending=False).items()))
print("\nUse in the product: Phase 3 sizing guidance (9b) should be segment-level ('shoppers who describe themselves"
      "\nas petite most often report ...'), never an individual prediction -- and only for segments with enough rows.")


### 7d-0. PII Scrubbing at Ingestion (Closes Governance Checklist Item 2)

v2 listed PII scrubbing as "shown in the schematic, not yet coded". It is now coded and runs on **every document before it enters the retrieval index**. That includes the competitor comments, which contain real usernames and account @handles. Brand accounts are kept, because they are organisations, not people. Redactions are counted so the step can be monitored.

In [ ]:
# --- 7d-0. PII scrubber: runs on every document before indexing ---
BRAND_HANDLES = {"@lululemon", "@lululemonme", "@aloyoga", "@alo", "@vuori", "@vuoriclothing"}
PII_PATTERNS = {
    "EMAIL": r"[\w.+-]+@[\w-]+\.[\w.-]+",
    "CARD": r"(?<!\d)(?:\d[ -]?){13,16}(?!\d)",
    "PHONE": r"(?<![\w/])(?:\+?\d{1,3}[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}(?![\w/])",
    # an order/tracking reference must actually contain digits (v4 fix: 'order number lookup' was being redacted)
    "ORDER_NO": r"(?i)\b(?:order|confirmation|tracking)\s*(?:no\.?|number|#)\s*[:#]?\s*(?=(?:[A-Z-]*\d){4})[A-Z0-9-]{6,}|#\d{6,}",
    # only an explicit self-introduction followed by a capitalised name ('I am usually a small' must survive)
    "NAME": r"\b[Mm]y name is\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?",
    "HANDLE": r"(?<![\w.])@[A-Za-z0-9_.]{3,}",
}

def scrub_pii(text):
    counts = {}
    out = str(text)
    # protect source URLs (they carry no personal data and must stay citable)
    urls = re.findall(r"https?://\S+", out)
    for i, u in enumerate(urls):
        out = out.replace(u, f"__URL{i}__", 1)
    for label, pat in PII_PATTERNS.items():
        def _sub(m, label=label):
            if label == "HANDLE" and m.group(0).lower() in BRAND_HANDLES:
                return m.group(0)
            counts[label] = counts.get(label, 0) + 1
            return f"[{label}]"
        out = re.sub(pat, _sub, out)
    for i, u in enumerate(urls):
        out = out.replace(f"__URL{i}__", u)
    return out, counts

# Self-test on a SYNTHETIC string (not real data) so the behaviour is visible
_demo = "Hi, my name is Jane Doe, order #A1B2C3D4 -- email jane.doe@example.com or call (555) 123-4567. cc @lululemon @janedoe99"
print("Synthetic test:", scrub_pii(_demo)[0])
# False-positive checks on phrases that appear in our REAL data and must NOT be redacted
for keep in ["Portal begins with order number lookup.", "I am usually a small and I got a medium.", "I'm returning the entirety of my Alo order."]:
    assert scrub_pii(keep)[0] == keep, f"false positive on: {keep}"
print("False-positive checks passed (policy wording and first-person review text are left intact).")


## 7d. Making the RAG Multimodal: Image and Video Evidence in the Retrieval Corpus

The v2 corpus held only text: 19 policy documents and 276 text evidence rows. With 4c and 5c done, image and video evidence can now be retrieved too:

- **Image documents:** each analysed customer photo becomes a retrievable document. We use Gemini's factual caption where one exists and CLIP's zero-shot label otherwise.
- **Video documents:** each Gemini video segment (timestamp, spoken claim, on-screen evidence) and each transcript window carrying a friction tag becomes a document. A citation can therefore point to "video X at 04:12", not just "video X".
- **Cross-modal retrieval:** CLIP's text encoder also lets a *text* query retrieve *images* directly by pixel similarity, even for images that have no caption.

Retrieval uses the hybrid method (7c) with the best alpha from the sweep, and policy routing: 3 of 6 slots go to Alo policy clauses, the rest to evidence. Two guardrails apply: **brand** (competitor evidence only when a question names a competitor) and **sentiment** (a complaint-shaped question is not answered with praise; see 3b). The original text-only TF-IDF retriever stays unchanged, so the v2 results in Section 8 remain reproducible. The two systems are compared in 8e.

In [ ]:
# --- 7d-1. Build the multimodal corpus (text + image + video documents) ---
mm_records = corpus_df[["doc_id", "kind", "brand", "text"]].to_dict("records")
# Policy clauses never contain their own topic words (R011 is 'Refund timing' but its text never says 'refund').
# In the v3 corpus each clause is prefixed with brand + topic so it can be found by what it is about.
_doc_type = ref_df.set_index("doc_id")["doc_type"].to_dict()
for rec in mm_records:
    if rec["kind"] == "policy":
        rec["text"] = f"[{rec['brand']} -- {_doc_type.get(rec['doc_id'], '')}] {rec['text']}"

img_meta = image_inventory.set_index("image_id")
for r in image_tags.itertuples():
    if r.pixel_tag is None:
        continue
    vlm = image_vlm_df[image_vlm_df["image_id"] == r.image_id]
    if len(vlm):
        v = vlm.iloc[0]
        text = (f"[Customer photo, {r.product}] {v['caption']} Fit: {v['fit_observation']}. "
                f"Quality: {v['quality_observation']}.")
    else:
        text = f"[Customer photo, {r.product}] Pre-trained vision model (CLIP) reads this image as: {r.pixel_tag} ({r.clip_friction_p:.0%})."
    mm_records.append({"doc_id": f"IMG:{r.image_id}", "kind": "image", "brand": "Alo Yoga", "text": text})

for r in vlm_segments_df.itertuples():
    mm_records.append({"doc_id": f"VID:{r.vid}@{r.timestamp}", "kind": "video", "brand": "Alo Yoga",
                       "text": f"[Video segment, {r.product}, size {r.size_worn}] Said: {r.spoken_claim} On screen: {r.visual_evidence}"})
for r in transcript_chunks_df[transcript_chunks_df["tag"] != "other"].itertuples():
    mm_records.append({"doc_id": f"VID:{r.vid}@{r.timestamp}(transcript)", "kind": "video", "brand": "Alo Yoga",
                       "text": f"[Video transcript] {r.text[:600]}"})

for r in lulu_complaints[lulu_complaints["scope"] == "return journey"].itertuples():
    mm_records.append({"doc_id": r.doc_id, "kind": "competitor_evidence", "brand": "Lululemon",
                       "text": f"[Lululemon customer comment] {r.text}"})

# Data-quality filter: evidence rows under 4 words (e.g. a review whose whole text is "Return") carry no information
# but keep winning keyword matches -- they are kept in the analytics, just not offered to the generator as evidence.
_before = len(mm_records)
mm_records = [r for r in mm_records if r["kind"] == "policy" or len(clean_for_embedding(r["text"]).split()) >= 4]
print(f"Dropped {_before - len(mm_records)} near-empty evidence rows (< 4 words) from the retrieval index.")

pii_totals = {}
for rec in mm_records:                               # 7d-0: every document is scrubbed before indexing
    rec["text"], _c = scrub_pii(rec["text"])
    for k_, v_ in _c.items():
        pii_totals[k_] = pii_totals.get(k_, 0) + v_
print("PII redactions at ingestion:", pii_totals or "none found")

mm_corpus_df = pd.DataFrame(mm_records).drop_duplicates("doc_id").reset_index(drop=True)
mm_corpus_df["compound"] = mm_corpus_df["text"].apply(vader_compound)   # sentiment of each document (3b)
mm_corpus_df["text_clean"] = mm_corpus_df["text"].apply(clean_for_embedding)
# v3 tokenizer fix: the v2 analyzer keeps possessive apostrophes, so "Lululemon's" becomes the token "lululemon'"
# (never matches "Lululemon") and "Alo's" becomes "alo'" (slips past the stop-word list). Fixed here for v3 only;
# the v2 retriever is left unchanged so the team's 8d scores stay reproducible.
def stem_analyzer_v3(text):
    text = re.sub(r"['’]s\b", "", str(text).lower()).replace("'", "").replace("’", "")
    return [t for t in (_ps.stem(w) for w in re.findall(r"[a-z]+", text)) if t not in _STEMMED_STOPWORDS and len(t) > 1]
_STEMMED_STOPWORDS |= {_ps.stem(w) for w in ["does", "doing", "did"]}
mm_vectorizer = TfidfVectorizer(analyzer=stem_analyzer_v3)
mm_tfidf = mm_vectorizer.fit_transform(mm_corpus_df["text_clean"])
mm_emb = embed(mm_corpus_df["text_clean"])
print("Multimodal corpus:", mm_corpus_df["kind"].value_counts().to_dict(), f"-> {len(mm_corpus_df)} documents")

MM_IS_POLICY = (mm_corpus_df["kind"] == "policy").values
MM_IS_COMPETITOR = (mm_corpus_df["kind"] == "competitor_evidence").values
MM_IS_ALO = (mm_corpus_df["brand"] == "Alo Yoga").values
MM_IS_EVIDENCE_LIKE = mm_corpus_df["kind"].isin(["evidence", "competitor_evidence", "image", "video"]).values
MM_COMPOUND = mm_corpus_df["compound"].values
COMPLAINT_Q_RE = r"three times|never|no response|no reply|still waiting|didn'?t|not received|complain|worst|delay|ignored|had to|problem|issue"

def is_complaint_query(q):
    return vader_compound(q) <= -0.05 or bool(re.search(COMPLAINT_Q_RE, q.lower()))

V3_TOP_K, V3_N_POLICY = 6, 3   # tested on the real data: 3 policy slots find the right clause for 8/8 gold questions (2 slots: 7/8)

def retrieve_mm(query, top_k=V3_TOP_K, alpha=None, n_policy=V3_N_POLICY):
    """Hybrid (TF-IDF + embeddings) over text + image + video docs, with the top n_policy slots reserved for policy (7c-2).
    Competitor customer evidence is only eligible when the question names a competitor (brand guardrail, 6e)."""
    alpha = BEST_ALPHA if alpha is None else alpha
    base = lambda q: alpha * minmax(mm_emb @ embed([q])[0]) + (1 - alpha) * minmax(
        cosine_similarity(mm_vectorizer.transform([q]), mm_tfidf).flatten())
    s = base(query)
    if is_complaint_query(query):   # sentiment-aware: a complaint question should not be answered with praise (3b)
        s = np.where(MM_IS_EVIDENCE_LIKE & (MM_COMPOUND >= 0.3), s - 0.5, s)          # push praise down
        s = np.where(MM_IS_EVIDENCE_LIKE & (MM_COMPOUND <= -0.3), s + 0.15, s)         # nudge clear complaints up
    names_competitor = bool(re.search(COMPETITOR_RE, query.lower()))
    if not names_competitor:   # Alo-only question: no competitor evidence AND no competitor policy anywhere in the answer
        s = np.where(MM_IS_COMPETITOR | (MM_IS_POLICY & ~MM_IS_ALO), -1.0, s)
    order = with_policy_routing(lambda q: s, n_policy, MM_IS_POLICY, MM_IS_ALO)(query)
    idx = list(order.argsort()[::-1][:top_k])
    if names_competitor and MM_IS_COMPETITOR.any() and not MM_IS_COMPETITOR[idx].any():
        idx[-1] = int(np.where(MM_IS_COMPETITOR)[0][np.argmax(s[MM_IS_COMPETITOR])])   # reserve one slot for competitor experience
    idx = np.array(idx)
    out = mm_corpus_df.iloc[idx][["doc_id", "kind", "text"]].copy()
    out.insert(2, "score", s[idx].round(3))
    return out

def retrieve_images_by_pixels(query, top_k=3):
    """Cross-modal: CLIP text encoder vs CLIP image embeddings -- finds images even if they have no caption."""
    if not CLIP_OK:
        return pd.DataFrame(columns=["image_id", "source_group", "clip_similarity"])
    sims = clip_img_emb @ clip_embed_text([query])[0]
    idx = sims.argsort()[::-1][:top_k]
    return pd.DataFrame({"image_id": [clip_img_ids[i] for i in idx], "clip_similarity": sims[idx].round(3)}).merge(
        image_inventory[["image_id", "source_group", "product"]], on="image_id")

# Brand guardrail check: over all 13 eval questions, no competitor comment may appear in an Alo answer's evidence
ALO_ONLY_QS = [q for q in GOLD if not re.search(COMPETITOR_RE, q.lower())] + [
    "Customer had to email support three times before getting a response about their return",
    "Customer wants a refund on the product", "Do Airbrush leggings run true to size or should I size up?"]
intrusions = sum(int((retrieve_mm(q)["kind"] == "competitor_evidence").sum()) for q in ALO_ONLY_QS)
print(f"Brand guardrail: {intrusions} competitor documents retrieved across {len(ALO_ONLY_QS)} Alo-only questions (must be 0).")
wrong_policy = sum(int(((retrieve_mm(q)["kind"] == "policy") & ~retrieve_mm(q)["doc_id"].map(
    dict(zip(mm_corpus_df["doc_id"], MM_IS_ALO))).fillna(True)).sum()) for q in ALO_ONLY_QS)
print(f"Competitor POLICY clauses cited for Alo-only questions: {wrong_policy} (must be 0).")

# Final v3 retriever on the same gold labels as 7c-2 (top-k = what v3 generation actually receives)
v3_hits = []
for q, gold in GOLD.items():
    got = retrieve_mm(q)["doc_id"].tolist()
    v3_hits.append({"query": q[:60], f"hit@{V3_TOP_K}": int(bool(set(got) & gold)), f"recall@{V3_TOP_K}": len(set(got) & gold) / len(gold)})
v3_hits = pd.DataFrame(v3_hits)
print(f"\nFinal v3 retriever ({V3_N_POLICY} policy + {V3_TOP_K - V3_N_POLICY} evidence slots, labelled policies, brand + sentiment guardrails): "
      f"hit@{V3_TOP_K} {v3_hits[f'hit@{V3_TOP_K}'].mean():.2f}, recall@{V3_TOP_K} {v3_hits[f'recall@{V3_TOP_K}'].mean():.2f} on {len(GOLD)} gold queries")
print(v3_hits.to_string(index=False))

for q in ["Customer had to email support three times before getting a response about their return",
          "Customer says the leggings pilled after a few washes, can they return it?",
          "Airlift leggings feel too tight, should I size up?",
          "How does Alo's refund experience compare to Lululemon's?"]:
    print(f"\nQUERY: {q}")
    _r = retrieve_mm(q)
    print(_r.assign(sentiment=_r["doc_id"].map(dict(zip(mm_corpus_df["doc_id"], mm_corpus_df["compound"].round(2)))),
                    text=_r["text"].str[:70])[["doc_id", "kind", "score", "sentiment", "text"]].to_string(index=False))
    pix = retrieve_images_by_pixels(q)
    if len(pix):
        print("  images by pixel similarity:", ", ".join(f"{r.image_id} ({r.source_group}, {r.clip_similarity})" for r in pix.itertuples()))


In [ ]:
# --- 7d-2. v3 generation: same grounded prompt as Section 7, but multimodal hybrid evidence ---
GEN_SYSTEM_PROMPT_V3 = GEN_SYSTEM_PROMPT + (
    " Evidence may include customer photos (doc_id starting IMG:) and video segments (doc_id starting VID:, with a timestamp)."
    " When visual evidence supports or contradicts a text claim, say so and cite both."
)

def generate_rag_answer_v3(query, top_k=V3_TOP_K):
    evidence = retrieve_mm(query, top_k=top_k)
    client = get_genai_client()
    if client is None:
        top = evidence.iloc[0]
        return f"[No API key available -- top retrieved match]\nSource: {top['doc_id']} ({top['kind']})\n{top['text']}"
    block = "\n\n".join(f"[{r.doc_id}] {r.text}" for r in evidence.itertuples())
    try:
        resp = client.models.generate_content(
            model=GEMINI_MODEL_NAME,
            contents=f"{GEN_SYSTEM_PROMPT_V3}\n\nEvidence:\n{block}\n\nCustomer question: {query}\n\nAnswer:")
        return resp.text.strip()
    except Exception as e:
        top = evidence.iloc[0]
        return f"[Gemini call failed ({str(e)[:120]}) -- top retrieved match]\nSource: {top['doc_id']}\n{top['text']}"

print(generate_rag_answer_v3("Customer says the leggings pilled after a few washes, can they return it?"))


## 8. Evaluation Plan

Compare the RAG assistant against:
1. A plain help-page baseline (agent manually searches the policy page)
2. A text-only RAG (no image/video evidence)

**Metrics:** answer accuracy (does it cite the correct policy clause), resolution time (simulated), perceived-effort proxy (rubric-scored by the team, 1-5 scale).

Below: 12 realistic support queries run against the retrieval layer now (works fully offline in this sandbox). The `accuracy_1to5` / `time_seconds` / `effort_1to5` columns are rubric-scored manually by the team once `generate_rag_answer` is run with a live Gemini key in an unrestricted environment -- retrieval quality (does the top hit actually answer the question) is checked automatically below as a proxy in the meantime.

In [ ]:
# --- Evaluation harness: 12 realistic support queries against retrieval ---
EVAL_QUERIES = [
    "Customer wants to exchange a final sale item for a different size",
    "How long does a refund take after Alo receives the return?",
    "Do Airbrush leggings run true to size or should I size up?",
    "Customer says the leggings pilled after a few washes, can they return it?",
    "What's the return window for a full-price item?",
    "Customer's refund came back as store credit instead of the original payment method",
    "Is the Alosoft Sunray Bra true to size for someone 5'4\" and 136 lbs?",
    "Customer had to email support three times before getting a response about their return",
    "Can I exchange directly for a different size or do I have to return and re-buy?",
    "How does Alo's return window compare to Lululemon or Vuori?",
    "Customer received a damaged item and wants to know the return process",
    "Do I need the original tags/packaging to return an item bought online?",
    "Customer wants a refund on the product",  # added after this exact query surfaced a real retrieval miss during manual testing -- see the stop-word fix above
]

def eval_retrieval_quality(query, top_k=3):
    """Cheap automatic proxy: does the top retrieved doc share meaningful keyword overlap
    with the query? Not a substitute for the team's manual accuracy scoring, but catches
    obvious retrieval misses before spending API calls on generation."""
    results = retrieve(query, top_k=top_k)
    top = results.iloc[0]
    return top["doc_id"], top["kind"], round(float(top["similarity"]), 3)

eval_rows = []
for q in EVAL_QUERIES:
    doc_id, kind, sim = eval_retrieval_quality(q)
    eval_rows.append({
        "query": q, "system": "TF-IDF retrieval (this notebook)",
        "top_doc_id": doc_id, "top_doc_kind": kind, "retrieval_similarity": sim,
        "accuracy_1to5": None, "time_seconds": None, "effort_1to5": None,
        "note": "accuracy/time/effort to be rubric-scored by the team once generate_rag_answer runs live",
    })

eval_results = pd.DataFrame(eval_rows)
print(f"{(eval_results['retrieval_similarity'] > 0.05).sum()}/{len(eval_results)} queries retrieved a non-trivial match (similarity > 0.05)")
eval_results[["query", "top_doc_id", "top_doc_kind", "retrieval_similarity"]]

### 8b. Run All 13 Queries Through Live Generation

Run this in Colab with your Gemini key set (same as Section 7). It calls `generate_rag_answer()` for each of the 13 eval queries and times each call.

**On the free tier, Gemini has a low requests-per-minute limit.** In our own test run, the first 8 queries succeeded (some slowly, because the library auto-retries a temporary `503 overloaded` error with increasing wait times), but the last 5 all failed instantly with `429 quota exceeded` -- the free tier had been used up for the window. To work around this without needing a paid key, the loop below adds a short pause between calls, and there's a second cell that re-runs *only the ones that failed* -- so you don't have to wait and re-run all 13 from scratch, and you don't burn extra quota re-asking questions that already got a good answer.

What this still does NOT do for you: judge whether the answer is actually *correct* -- that's `accuracy_1to5`, and only a human reading the generated text against the real policy can score that. `effort_1to5` is your own estimate of how much work an agent would've saved vs. looking this up manually.

In [ ]:
# --- Run all 13 queries through live generation, with timing (needs a Gemini key + network) ---
import time

def run_eval_batch(queries, delay_seconds=8):
    """Runs each query through generate_rag_answer(), pausing between calls to stay
    under the free-tier per-minute rate limit. Returns a DataFrame with the same
    columns as before, ready to have accuracy_1to5 / effort_1to5 filled in by hand."""
    rows = []
    for q in queries:
        t0 = time.time()
        answer = generate_rag_answer(q)
        elapsed = round(time.time() - t0, 2)
        rows.append({
            "query": q,
            "generated_answer": answer,
            "time_seconds": elapsed,
            "accuracy_1to5": None,   # <-- fill in by hand: does this correctly answer the question, citing the right doc_id?
            "effort_1to5": None,     # <-- fill in by hand: 1 = still needs lots of agent work, 5 = ready to send as-is
        })
        time.sleep(delay_seconds)
    return pd.DataFrame(rows)

eval_results_live = run_eval_batch(EVAL_QUERIES)
pd.set_option("display.max_colwidth", 300)
eval_results_live[["query", "generated_answer", "time_seconds"]]

### 8c. Retry only the queries that failed with a quota error

If any rows above show `[Gemini call failed (429 ...`, wait 2-3 minutes for the free-tier quota window to reset, then run this cell -- it only re-asks the ones that actually failed and merges the new answers back in, so you don't waste quota re-running the 8 that already worked.

In [ ]:
# --- Retry only the failed queries, then merge the fixed answers back in ---
failed_mask = eval_results_live["generated_answer"].astype(str).str.contains("Gemini call failed")
print(f"{failed_mask.sum()} of {len(eval_results_live)} queries still need a retry.")

if failed_mask.sum() > 0:
    retry_df = run_eval_batch(eval_results_live.loc[failed_mask, "query"].tolist(), delay_seconds=10)
    retry_df = retry_df.set_index("query")
    eval_results_live = eval_results_live.set_index("query")
    eval_results_live.update(retry_df[["generated_answer", "time_seconds"]])
    eval_results_live = eval_results_live.reset_index()

eval_results_live[["query", "generated_answer", "time_seconds"]]

### 8d. Human-Scored Accuracy & Effort

Run this after Section 8b (and 8c, if you needed it) so `eval_results_live` has real generated answers for all 13 queries. These scores were agreed on by the team after actually reading each generated answer against the source evidence (not rubber-stamped) -- two real misses were found this way (query 4 and query 7 below), both already documented as known limitations earlier in this notebook.

In [ ]:
# --- Human-scored accuracy & effort, agreed after manually reading every generated answer ---
MANUAL_SCORES = {
    "Customer wants to exchange a final sale item for a different size": (5, 5),
    "How long does a refund take after Alo receives the return?": (5, 4),
    "Do Airbrush leggings run true to size or should I size up?": (4, 3),
    "Customer says the leggings pilled after a few washes, can they return it?": (4, 4),
    "What's the return window for a full-price item?": (2, 2),  # real miss: R001 exists but wasn't retrieved (vocabulary-gap limitation, Section 7)
    "Customer's refund came back as store credit instead of the original payment method": (5, 5),
    "Is the Alosoft Sunray Bra true to size for someone 5'4\" and 136 lbs?": (3, 3),
    "Customer had to email support three times before getting a response about their return": (2, 2),  # real miss: retrieved evidence has the opposite sentiment (Section 7 limitation)
    "Can I exchange directly for a different size or do I have to return and re-buy?": (4, 4),
    "How does Alo's return window compare to Lululemon or Vuori?": (3, 2),  # incomplete: only surfaces Vuori, misses Alo's and Lululemon's own numbers
    "Customer received a damaged item and wants to know the return process": (4, 4),
    "Do I need the original tags/packaging to return an item bought online?": (5, 4),
    "Customer wants a refund on the product": (4, 2),  # correct that no policy exists, but cites other customers' unrelated problems -- not something to hand a customer as-is
}

for q, (acc, eff) in MANUAL_SCORES.items():
    mask = eval_results_live["query"] == q
    eval_results_live.loc[mask, "accuracy_1to5"] = acc
    eval_results_live.loc[mask, "effort_1to5"] = eff

scored = eval_results_live["accuracy_1to5"].notna().sum()
print(f"Manually scored {scored} of {len(eval_results_live)} queries.")
print(f"Average accuracy: {eval_results_live['accuracy_1to5'].mean():.2f}/5, average effort: {eval_results_live['effort_1to5'].mean():.2f}/5")
eval_results_live[["query", "accuracy_1to5", "effort_1to5"]]

### 8e. Text-only RAG (v2) vs Multimodal Hybrid RAG (v3): the Comparison Promised in the Evaluation Plan

The evaluation plan above promised to compare against a **text-only RAG**. The v2 system (TF-IDF, text only) is that baseline, and 8b-8d scored it. This cell runs the same 13 queries through v3 (hybrid retrieval over text + image + video evidence).

**The team fills `MANUAL_SCORES_V3` by hand, on the same 1-5 rubric used in 8d.** The code deliberately leaves the scores empty: an accuracy score is only honest if a person has read the answer against the evidence.

In [ ]:
# --- 8e-1. Run the 13 queries through v3 (needs Gemini key; ~13 calls, paced for the free tier) ---
v3_rows = []
for q in EVAL_QUERIES:
    t0 = time.time()
    ans = generate_rag_answer_v3(q)
    v3_rows.append({"query": q, "v3_answer": ans, "v3_seconds": round(time.time() - t0, 2),
                    "v3_top_docs": ", ".join(retrieve_mm(q)["doc_id"])})
    if get_genai_client() is not None:
        time.sleep(8)
v3_results = pd.DataFrame(v3_rows)
v3_results[["query", "v3_top_docs", "v3_answer"]]


In [ ]:
# --- 8e-2. Human scores for v3 (FILL IN after reading each answer), then compare with v2 ---
MANUAL_SCORES_V3 = {q: (None, None) for q in EVAL_QUERIES}   # (accuracy_1to5, effort_1to5) -- replace None with the team's scores
# e.g. MANUAL_SCORES_V3["What's the return window for a full-price item?"] = (5, 5)

v3_results["accuracy_1to5_v3"] = v3_results["query"].map(lambda q: MANUAL_SCORES_V3[q][0])
v3_results["effort_1to5_v3"] = v3_results["query"].map(lambda q: MANUAL_SCORES_V3[q][1])
cmp = eval_results_live[["query", "accuracy_1to5", "effort_1to5"]].rename(
    columns={"accuracy_1to5": "accuracy_v2", "effort_1to5": "effort_v2"}).merge(
    v3_results[["query", "accuracy_1to5_v3", "effort_1to5_v3"]], on="query")
scored = cmp["accuracy_1to5_v3"].notna().sum()
if scored:
    s = cmp.dropna(subset=["accuracy_1to5_v3"])
    print(f"On the {scored} queries scored for both: accuracy v2 {s['accuracy_v2'].astype(float).mean():.2f} -> "
          f"v3 {s['accuracy_1to5_v3'].astype(float).mean():.2f}; effort v2 {s['effort_v2'].astype(float).mean():.2f} -> "
          f"v3 {s['effort_1to5_v3'].astype(float).mean():.2f}")
else:
    print("v3 not scored yet -- fill MANUAL_SCORES_V3 after reading the answers above. Retrieval-only comparison is in 7c-2.")
cmp


### 8f. Help-Page Baseline: The Comparison Promised in the Proposal

The proposal commits to comparing the assistant with **existing help pages** as well as a text-only baseline. We model "an agent searching Alo's help pages" as retrieval **restricted to Alo's official published pages**: the return, refund, exchange, portal, sizing and help-center clauses. There is no customer evidence, no images and no video. The same gold labels and the same 13 questions are used, which gives three systems side by side:

| System | What it can see |
|---|---|
| Help pages (baseline) | Alo's official policy/help text only |
| v2 text-only RAG | + customer text evidence (TF-IDF) |
| v3 multimodal RAG | + image & video evidence, hybrid retrieval, policy routing, brand + sentiment guardrails |

In [ ]:
# --- 8f-1. Retrieval comparison: help pages vs v2 vs v3 on the gold-labelled questions ---
HELP_IDS = [d for d in ref_df["doc_id"] if ref_df.set_index("doc_id").loc[d, "brand"] == "Alo Yoga" and d != "SUPP-R001"]
help_docs = mm_corpus_df[mm_corpus_df["doc_id"].isin(HELP_IDS)].reset_index(drop=True)
help_vec = TfidfVectorizer(analyzer=stem_analyzer_v3).fit(help_docs["text"])
help_M = help_vec.transform(help_docs["text"])

def retrieve_helppage(q, top_k=3):
    s = cosine_similarity(help_vec.transform([q]), help_M).flatten()
    idx = s.argsort()[::-1][:top_k]
    return help_docs.iloc[idx][["doc_id", "kind", "text"]].assign(score=s[idx].round(3))

rows = []
for q, gold in GOLD.items():
    h = set(retrieve_helppage(q, 3)["doc_id"])  # noqa
    v2_rank = retrieval_eval.query("query == @q and method == 'tfidf'")["first_gold_rank"].iloc[0]
    v3 = set(retrieve_mm(q)["doc_id"])
    rows.append({"question": q[:55], "help_pages_hit@3": int(bool(h & gold)), "v2_text_rag_hit@3": int(v2_rank is not None and v2_rank <= 3),
                 "v3_multimodal_hit@k": int(bool(v3 & gold))})
sys_cmp = pd.DataFrame(rows)
print(sys_cmp.to_string(index=False))
print("\nShare of policy questions where the correct clause is retrieved:",
      sys_cmp[["help_pages_hit@3", "v2_text_rag_hit@3", "v3_multimodal_hit@k"]].mean().round(2).to_dict())
n_help = len(help_docs)
chance = np.mean([1 - np.prod([(n_help - len(g) - i) / (n_help - i) for i in range(3)]) for g in GOLD.values()])
print(f"Context: the help-page baseline picks 3 of only {n_help} documents, so a random pick would already hit "
      f"{chance:.0%} of the time. Finding the clause is easy when the library is tiny -- its real limit is that it holds"
      f"\nno customer evidence at all (next).")

experience_qs = [q for q in EVAL_QUERIES if q not in GOLD]
print(f"\nThe other {len(experience_qs)} questions need CUSTOMER EXPERIENCE evidence (sizing reports, support stories)."
      "\nHelp pages contain none, so the baseline can at best return a policy clause. Example:")
q = "Do Airbrush leggings run true to size or should I size up?"
print(f"  Q: {q}")
print("  help pages ->", ", ".join(retrieve_helppage(q)["doc_id"]), "| v3 ->", ", ".join(retrieve_mm(q)["doc_id"]))


In [ ]:
# --- 8f-2. Generated answers from the help-page baseline (Gemini), for the same human 1-5 scoring as 8d/8e ---
def generate_helppage_answer(query):
    ev = retrieve_helppage(query, 3)
    client = get_genai_client()
    if client is None:
        return f"[No API key -- top help-page clause] {ev.iloc[0]['doc_id']}: {ev.iloc[0]['text'][:200]}"
    block = "\n\n".join(f"[{r.doc_id}] {r.text}" for r in ev.itertuples())
    try:
        return client.models.generate_content(model=GEMINI_MODEL_NAME,
            contents=f"{GEN_SYSTEM_PROMPT}\n\nEvidence:\n{block}\n\nCustomer question: {query}\n\nAnswer:").text.strip()
    except Exception as e:
        return f"[Gemini call failed: {str(e)[:100]}]"

help_rows = []
for q in EVAL_QUERIES:
    t0 = time.time()
    help_rows.append({"query": q, "help_answer": generate_helppage_answer(q), "help_seconds": round(time.time() - t0, 2)})
    if get_genai_client() is not None:
        time.sleep(8)
help_results = pd.DataFrame(help_rows)
MANUAL_SCORES_HELP = {q: (None, None) for q in EVAL_QUERIES}   # (accuracy_1to5, effort_1to5) -- team fills after reading
help_results["accuracy_help"] = help_results["query"].map(lambda q: MANUAL_SCORES_HELP[q][0])
help_results["effort_help"] = help_results["query"].map(lambda q: MANUAL_SCORES_HELP[q][1])
three_way = eval_results_live[["query", "accuracy_1to5", "effort_1to5"]].rename(columns={"accuracy_1to5": "acc_v2", "effort_1to5": "eff_v2"}) \
    .merge(help_results[["query", "accuracy_help", "effort_help"]], on="query") \
    .merge(v3_results[["query", "accuracy_1to5_v3", "effort_1to5_v3"]], on="query")
if three_way["accuracy_help"].notna().any():
    print(three_way.drop(columns="query").astype(float).mean().round(2).to_string())
else:
    print("Fill MANUAL_SCORES_HELP and MANUAL_SCORES_V3 after reading the answers; this cell then prints the 3-way comparison.")
help_results[["query", "help_answer"]]


## 9. Data Provenance & Verification Note (for the Governance section)

During integration, we spot-checked a teammate's independently-researched Evidence dataset against its cited live sources before trusting it:

- **Verified accurate**: 4 Zappos-sourced reviews checked word-for-word against the live page matched the dataset's claims once we retrieved the *complete* review text (our first pass used a truncated fetch and wrongly flagged them).
- **One broken citation found and corrected**: an Alo product-page URL had its slug words in the wrong order and 404'd; the correct URL was confirmed live.
- **One inconclusive case resolved by manual check**: our automated fetch tool couldn't render JavaScript-loaded review widgets on Alo's own site, which looked like "no reviews exist" but actually meant "our tool can't see them" -- resolved by a manual browser check rather than either accepting or rejecting the claim on incomplete evidence.

This distinction -- between a source being wrong and a verification tool being limited -- is exactly the kind of practical hallucination/verification discipline the governance section should demonstrate, and it's a real example from this project rather than a hypothetical.

- **Same JS-rendering limitation hit again, handled the same way (T300)**: a review on the "7/8 High-Waist Airlift Legging - Black" page ("Adi A.," Under 5'0", size XXS, "slightly smaller than what I expected") was confirmed by the team directly on the live page; the tool's fetch confirmed the product page itself is real but could not independently re-render the JS-loaded review widget to re-verify the exact text. Logged with that caveat explicit in `collected_by` rather than either silently treating it as tool-verified or discarding it.

## 9b. Governance Task 3 -- Implementation Roadmap

Phased rollout for putting this multimodal RAG assistant in front of real Alo Yoga support agents, sequencing risk down as trust in the system builds.

**Phase 0 -- Shadow mode (Weeks 1-2).** The assistant runs alongside human agents but its answers are never shown to the customer or used to resolve a ticket. Agents see the suggested answer + citations side-by-side with their own response, purely to build a comparison dataset. No governance risk beyond storage of retrieved evidence; this is where the evaluation harness above (Section 8) gets populated with real accuracy scores instead of `None` placeholders.

**Phase 1 -- Human-in-the-loop assist (Weeks 3-6).** Agents can accept, edit, or reject the suggested answer before it goes to the customer; every edit and rejection is logged with a reason code. This is the phase where the human-oversight and monitoring/feedback-loop principles from the governance framework are load-bearing, not theoretical -- edit/rejection rate becomes the leading indicator of whether the RAG is ready to go further.

**Phase 2 -- Auto-resolve for low-risk, high-confidence queries only (Weeks 7-10).** Only queries where (a) retrieval similarity clears a set threshold and (b) the matched policy clause carries no ambiguity (e.g. "what is your return window" vs. a final-sale/exchange edge case) are allowed to auto-send without human review. Everything else still routes to Phase 1. Threshold and category list are reviewed weekly against the Phase 1 edit-log data, not set once and left alone.

**Phase 3 -- Expand scope + add proactive sizing guidance (Weeks 11+).** Only after Phase 2 shows a stable low edit-rate does the assistant expand from reactive support (answering a ticket) into the aggregate, crowd-sourced sizing-guidance feature discussed earlier -- framed explicitly as "X% of buyers in your size range kept true to size / sized up," never as an individual prediction, per the population-vs-individual-level distinction the professor raised.

**Rollback trigger, defined upfront rather than improvised under pressure:** if the edit/rejection rate in any phase exceeds a pre-agreed threshold, or a single incident of a fabricated/uncited policy claim reaching a customer is confirmed, the system drops back one phase automatically pending review -- this is a governance control, not just an engineering one, and it's what turns "human oversight" from a slide bullet into an actual mechanism.

**Cross-cutting, all phases:** the PII-scrubbing and data-minimization steps shown in the architecture schematic must run before any evidence enters the vector store, at every phase. Phase 0 shadow mode is not exempt just because a customer never sees the output. The face rule is already implemented (4c: photos with a visible face are analysed locally only). The text PII scrubber is implemented (7d-0) and runs on every document before indexing.

## 9c. Sustainability Claims & Greenwashing Check (ESG)

The assignment brief names "greenwashing accusations" as one of the core threats a D2C fashion brand faces, and our own team proposal explicitly committed to "a separate assessment of sustainability claims" -- this section is that assessment, not an afterthought.

**What we checked:** Alo Yoga positions itself around wellness and mindful living, a brand identity that implicitly borrows credibility from sustainability without always making an explicit sustainability claim. To test whether that positioning is backed by independent evidence, we pulled a third-party rating rather than relying on the brand's own marketing copy (SUPP-R001, Good On You, an independent fashion-sustainability rating directory):

- **Overall rating: "Not Good Enough"** -- the lowest of Good On You's rating tiers.
- **Planet: 1/5** -- few lower-impact materials, no disclosed transparency on material composition, no evidence of initiatives on packaging waste, textile waste, microplastics, or hazardous chemicals.
- **People: 1/5** -- no disclosure of complete supplier information or final production locations, no evidence of living wages, no published Code of Conduct.
- **Animals: 2/5** -- uses leather, wool, and cashmere; no animal welfare policy disclosed.

**Why this matters for governance, not just marketing:** this is a real, independently-sourced gap between brand positioning and disclosed practice -- exactly the pattern regulators and consumer-protection bodies define as greenwashing risk when a brand benefits from a sustainability-adjacent image without substantiating it. For an AI system built on top of this brand's data, that has two concrete governance implications:

1. **The RAG assistant must never manufacture or reinforce an unsubstantiated sustainability claim.** If a customer asks "is this a sustainable brand?", the system should surface this rating and its gaps honestly (citing SUPP-R001), not default to generic wellness-brand language pulled from marketing copy in the corpus -- the same evidence-traceability and no-ungrounded-generation guardrail already built into the architecture (see the schematic) applies here, not just to return-policy questions.
2. **This belongs in the monitoring loop, not a one-time check.** Third-party ESG ratings are updated periodically and Alo's public disclosures could change; a recommended addition to the Phase 1+ monitoring dashboard (Section 9b) is a periodic re-pull of this rating so the governance framework doesn't quietly go stale.

**Limitation, stated plainly:** this is a single third-party source (not our own primary audit), and the rating snapshot has its own as-of date -- flagged the same way we flagged the FA19 sizing PDF as dated evidence elsewhere in this notebook, not presented as a permanent verdict on the brand.

## 9d. Known Data Limitations (Consolidated)

Stated plainly here rather than left scattered, since a governance framework should name what the underlying data can't support, not just what the model does:

- **UGC photo evidence is thin.** Of the 10 UGC image rows originally referenced (Reddit threads), only 2 resolved to genuine, verified user photos (Section 4b). Together with the Media_Assets customer photos, that is still only a handful of customer images. Pixel-level findings in 4c are **descriptive**, not statistically representative. And the catalog set is 19 unique files, not 95.
- **The vision models are only partly validated.** CLIP's accuracy is measured on one distinction we have ground truth for (official vs. customer photo, 4c-6). Its friction labels, and Gemini's, are checked only for agreement with the human notes on the Media_Assets photos, not against a labelled test set. Fine-tuning would need the few hundred agent-labelled photos Phase 1 (9b) would produce.
- **Video coverage is layered and uneven** (5c). Every video is tagged from its title or notes. Transcripts cover at most 40 videos, and only where YouTube allows requests from Colab. Gemini's direct viewing is capped at 10 videos for cost and quota. Engagement results are correlations on about 130 videos, not causal effects.
- **Retrieval evaluation is small.** TF-IDF vs. embeddings vs. hybrid vs. policy-routed (7c) is scored on 8 gold-labelled policy queries. The hybrid weight was picked on those same queries (no hold-out set), so treat the result as directional. Routing is designed to help this policy-clause metric; its cost is 2 of 5 retrieval slots per question.
- **Self-reported shopper segments are sparse** (7c-7). Most reviews say nothing about the reviewer's body, and the extraction is rule-based. The segment shares describe *who wrote reviews*, not Alo's customer base.
- **The competitor benchmark is asymmetric** (6e). Alo's evidence is reviews, Reddit and Trustpilot; Lululemon's is about 20 complaints left under the brand's own Instagram posts; Vuori has policy documents only. The friction-mix comparison shows which frictions recur across brands, not comparable rates. The team's "Alo UGC" Instagram image set (44 images) was reviewed and excluded as evidence: it is mostly resellers, store openings and lifestyle posts. The resellers' "100% original" claims are a content-moderation / counterfeit risk (Session 16), not return evidence.
- **The sustainability/ESG finding (Section 9c) rests on one third-party rating source**, not an independent audit, and carries its own as-of date.


## 9e. Governance Checklist -- Verified Against Our Own Proposal's Commitments

Our team's approved proposal explicitly committed to: data provenance and permitted-use checks, privacy and data minimization, bias and representation assessment, explainability, multimodal conflict and uncertainty handling, RAG evidence traceability, human review and override, and monitoring of model performance and failure modes. Checked one by one against what's actually built, not just restated as a bullet list:

| Commitment | Status | Evidence |
|---|---|---|
| Data provenance / permitted-use checks | Done | Section 9 -- sources spot-checked live, one broken citation found and fixed, rights-use guidance attached to media assets (Section 4) |
| Privacy / data minimization | **Checked, not just assumed** | **Images:** photos with a visible face are analysed only inside our runtime (pixel features + local CLIP) and are never sent to an external API, which matters because free-tier Gemini traffic may be used by Google to improve its products (4c). **Text:** we scanned the full 295-document RAG corpus for email addresses in the actual evidence text (not source URLs) -- **zero found**. Public reviews rarely carry raw PII, but for a production system handling real support tickets (which do contain names, emails, order numbers), the PII-scrubbing step shown in the architecture schematic would need to actually run at ingestion -- **now implemented (7d-0)**: it runs on every document before indexing and logs redaction counts. Faces in photos are handled by the local-only rule (4c) |
| Bias / representation assessment | Done | Stated directly: review and social-media data structurally over-represents customers who were upset enough to post publicly -- the friction map (Section 6) reflects *who complained*, not the full population of buyers, most of whom never write a review either way. **v3 measures representation directly** (7c-7): the share of evidence from petite, tall and curvy/midsize reviewers vs. reviews with no body information, so it is visible which shoppers any sizing guidance would under-serve |
| Explainability | Done | Every generated answer must cite a `doc_id` (Section 7); no ungrounded claims by design |
| Multimodal conflict / uncertainty handling | **Demonstrated with a real example, not asserted** | Our own evidence disagrees with itself on sizing: E054 says the Airlift legging "runs true to size," while T144 is a reviewer who explicitly writes "I know some others have said otherwise, but I actually sized up" for a different item. We do not silently pick one -- both surface in retrieval, and the generation prompt is grounded strictly in whatever is retrieved, so a genuine disagreement in the evidence shows up as a genuine disagreement in the answer rather than a false consensus. This is also exactly the individual-vs-population distinction our professor raised on the original proposal -- fit is a distribution, not a single answer, and this system should never claim otherwise |
| RAG evidence traceability | Done | `doc_id` citation is structurally required, not optional (Section 7) |
| Human review and override | Done | Section 9b, Phase 1 -- every response is agent-approved before reaching a customer, never auto-sent |
| Monitoring of performance / failure modes | Done | Section 9b -- accept/reject/edit-rate feedback log, explicit rollback trigger |

Net (v4): all 8 commitments are demonstrated with evidence from this project. The v2 gap (PII scrubbing not coded) is closed in 7d-0. Further Session 16 risks (deepfakes, content moderation, influencer disclosure, copyright, wrong-brand citation) are covered in 9f.

## 9f. Further Governance Risks: Deepfakes, Content Moderation, Influencer Disclosure, Copyright (Session 16)

**Synthetic imagery and deepfakes.** Phase 3 of the roadmap (9b) mentions proactive sizing guidance, and the natural next step is virtual try-on. Rules: (1) AI-generated try-on imagery is always labelled as such to the customer. (2) Generated imagery is **never ingested as evidence**: the RAG's image documents must come from real customer or catalog photos, with provenance recorded (source URL, capture date). Content-credential standards such as C2PA are the production mechanism for this. (3) Creator content is not edited or re-generated to change what a garment looks like. Campbell et al. (2022, Session 16 reading) set out the brand risk when this line blurs.

**Content moderation before ingestion.** Two real cases from our own data:
- **Unauthorised resellers.** Many of the team's "Alo UGC" Instagram posts are resellers claiming "100% original" and quoting sizes by DM. That's a counterfeit/grey-market risk, and it is exactly the kind of post a moderation filter (the CM-MRAG reading) must keep out of an evidence base about genuine customer experience.
- **Duplicated complaints.** One Lululemon customer pasted the same refund complaint under several posts. We de-duplicate by text before counting (6e-2), or one voice would look like many.

**Influencer disclosure.** Only 4 of 190 scraped Alo-related videos, and 0 of 80 Lululemon posts, carry a platform "paid" flag. Undisclosed sponsorship is common in hauls. The assistant therefore treats creator content as **opinion, not independent evidence**: it can be cited for what a creator *showed* (a visible fit issue), not as proof that a product is good. Endorsement-disclosure rules (the FTC Endorsement Guides in the US, ASCI guidelines in India) apply to any content Alo itself commissions.

**Copyright and reuse.** Customer photos, video frames and thumbnails are analysed for research and are **not reproduced in the deck** without reuse rights (open item in Section 10). Features, scores and captions derived from them are our own analysis and can be shown.

**Wrong-brand citation (found and fixed in this project).** Once competitor data was added, an Alo answer could cite a Lululemon complaint or policy. The brand guardrail (7d) blocks this, and it is tested: 0 competitor documents are retrieved across 10 Alo-only questions.

## 10. Next Steps
- [x] Video pipeline wired (notes-based tagging; full-transcript upgrade documented but needs an unrestricted environment)
- [x] Gemini few-shot classification confirmed working live (Colab, gemini-3.6-flash)
- [x] `generate_rag_answer()` confirmed working live -- real grounded, cited answer generated (see confirmation note in Section 7)
- [x] Two real TF-IDF retrieval bugs found via live testing and fixed with a verified stemming fix (Section 7): the unretrievable R001 policy doc, and short/degenerate documents falsely winning top-1. Two remaining, honestly-documented limitations (vocabulary gap, sentiment blindness) written up right after the fix as evidence of testing rigor.
- [x] Evaluation harness (Section 8) expanded to 13 real queries and now actually runs generation for all of them (Section 8b/8c), with a rate-limit-safe retry step for Gemini's free-tier quota
- [x] Task 3 Implementation Roadmap drafted (Section 9b)
- [x] Sustainability claims / greenwashing assessment added (Section 9c), using the Good On You ESG rating -- closes the gap between the assignment brief's named threat and our proposal's commitment to a separate sustainability assessment
- [x] All 13 queries run through live generation (Section 8b/8c), human-scored for accuracy/effort (Section 8d)
- [x] Consolidated known-limitations section added (Section 9d) and governance checklist cross-checked against the proposal's 8 commitments (Section 9e)
- [x] Multimodal Fusion Pipeline added as its own analysis (Section 6b) -- text/image/video coverage per friction category, a real cross-modal agreement case, a real cross-modal conflict case (Airlift sizing), and an honest missing-modality finding, using late fusion (independent per-modality tagging, reconciled at the decision level) rather than early fusion (joint embeddings), for the same auditability reasons as the TF-IDF-vs-embeddings call in Section 7b
- [x] Executive deck built (15 slides), QA'd for visual/content defects, delivered as PDF + PPTX
- [ ] Confirm reuse rights on Media_Assets images (and the 2 real UGC photos, I113/I121) before including any in the public deck
- [x] Real video evidence expanded (Section 5b): 131 additional customer/influencer YouTube videos across 105 channels, collected via targeted search and title-tagged (Instagram hashtag-scrape attempts did not pan out -- wrong actor outputs each time -- so this is the YouTube-based substitute that did)
- [x] **v3:** Image analytics (4c) -- 19 unique catalog files + customer photos; interpretable visual features, CLIP zero-shot with accuracy measured, Gemini vision captions, face-based data-minimisation rule; pixel-derived tags now feed the fusion table
- [x] **v3:** Video analytics (5c) -- engagement dynamics, timestamped transcript tagging, Gemini direct video viewing (speech vs. visuals), latency/cost trade-off; videos left as `other` re-tagged from content
- [x] **v3:** Text embeddings (7c) -- local embeddings vs. TF-IDF vs. hybrid on gold-labelled queries; two documented failures re-tested; semantic + shopper-profile segmentation
- [x] **v3:** Multimodal RAG corpus (7d) -- image and video documents retrievable, plus CLIP text-to-image search
- [ ] **Run All in Colab** with `GEMINI_API_KEY` set. The saved outputs of 8b/8c come from a run without a key, and every v3 cell needs Colab's internet access
- [x] **v3:** Competitor benchmark (6e), as committed in the proposal: policy scorecard, Lululemon policy vs. customer experience, friction mix, video landscape; brand guardrail in the RAG
- [ ] Place `dataset_instagram-scraper_*.csv` and `dataset_youtube-scraper_*.csv` next to the notebook in Colab (6e reads them)
- [ ] Fill `MANUAL_SCORES_V3` (8e) after reading the 13 v3 answers
- [ ] Update the deck/schematic with the v3 findings (image-feature gap, video reach, retrieval comparison, segments)
- [x] **v4:** Sentiment (3b), topic modeling (3c), UI audit (4e), influencer/short-form (5d), brand alignment (5e), dataset inventory & EDA (6f), PII scrubber (7d-0), sentiment-aware retrieval, help-page baseline (8f), governance 9f
- [ ] **Capture the return-journey screenshots** listed in 4e into `ui_screenshots/` before the final Colab run
- [ ] Fill `MANUAL_SCORES_HELP` (8f) alongside `MANUAL_SCORES_V3` (8e) -- same 13 questions, same 1-5 rubric
